In [ ]:
# === Importação de Biliotecas ===
import os
import gc
import math
import time
import math
import time
#import tqdm
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from tqdm.notebook import tqdm
from sklearn.metrics import auc
import zipfile
import subprocess
import numpy as np
import pandas as pd
from fpdf import FPDF
import seaborn as sns
import miceforest as mf
import matplotlib.pyplot as plt
from scipy.stats import bootstrap
from sklearn.metrics import log_loss
import matplotlib.patches as mpatches
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from matplotlib.backends.backend_pdf import PdfPages
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, balanced_accuracy_score, roc_auc_score, roc_curve, confusion_matrix


In [ ]:
import warnings

warnings.filterwarnings(
    "ignore",
    message=r"X does not have valid feature names, but .* was fitted with feature names",
    category=UserWarning,
)

import os
os.environ["PYTHONWARNINGS"] = (
     "ignore:X does not have valid feature names, but .* was fitted with feature names:UserWarning"
 )

In [ ]:
from joblib import load

# === Parametros Genéricos ===
# === Definição de Parâmetros - Obrigatório ser informados pelo usuario
BASE_PATH = "SEPSIS_LIME_XGBOOST" 
FILE_IN = Path('../SEPSIS/data/processed/exams.csv')

os.makedirs(BASE_PATH, exist_ok=True)

METHOD = "" #Informado na célula de cada método

direcao_esperada = {
    # -------------------------------------------------
    # 1) Variáveis clínicas / laboratoriais (fisiológicas)
    # -------------------------------------------------

    # Peso de admissão: relação com sepse é bem indireta / não monotônica (obesidade, caquexia, etc.)
    "Admission Weight (Kg)": 0,

    # Marcadores metabólicos / ácido-base:
    "Anion gap": +1,          # gap aniônico ↑ geralmente indica acidose metabólica → pior quadro / maior gravidade
    "HCO3 (serum)": -1,       # bicarbonato ↑ costuma indicar menor acidose → melhor quadro (↓ prob. sepse grave)

    # Função renal / catabolismo:
    "BUN": +1,                # ureia ↑ → disfunção renal / catabolismo → pior prognóstico
    "Creatinine (serum)": +1, # creatinina ↑ → lesão renal aguda / crônica → maior gravidade

    # Eletrólitos e minerais:
    "Calcium non-ionized": -1,    # em sepse costuma cair; cálcio mais alto tende a ser menos grave (bem aproximado)
    "Chloride (serum)": 0,        # associação com sepse é muito dependente de reposição volêmica/fluídos
    "Magnesium": 0,               # hipo/hipermagnesemia → relação complexa, não estritamente monotônica
    "Phosphorous": 0,             # idem: muito dependente de função renal / nutrição
    "Potassium (serum)": 0,       # relação nitidamente em U (hipo e hiper são ruins) → melhor marcar 0
    "Sodium (serum)": 0,          # hipo/hipernatremia → ambos prejudiciais → direção não monotônica

    # Glicose:
    "Glucose (serum)": +1,        # hiperglicemia de estresse / diabetes → em geral associada a piores desfechos

    # Hemograma / coagulação:
    "Hemoglobin": 0,              # anemia vs. hemoconcentração → relação bem complexa, evitamos direção fixa
    "INR": +1,                    # INR ↑ → coagulopatia / disfunção hepática → pior prognóstico
    "Platelet Count": -1,         # plaquetas ↓ são típicas na sepse; mais plaquetas tende a ser melhor
    "WBC": +1,                    # pensando “de normal para leucocitose” → inflamação/infeção ↑ (U-shape com leucopenia, mas usamos +1 como aproximação)

    # Sinais vitais:
    "Heart Rate": +1,             # taquicardia → critério SIRS / sepse
    "Respiratory Rate": +1,       # taquipneia → critério de sepse / gravidade respiratória
    "Temperature Fahrenheit": +1, # febre é típico de infecção (sabendo que hipotermia também é grave, mas mantemos +1 como tendência)

    # Pressão arterial:
    "Non Invasive Blood Pressure systolic": -1,   # quanto maior (até normal), menor prob. choque séptico
    "Non Invasive Blood Pressure diastolic": -1,  # idem (↓ PA diastólica → pior perfusão)

    # Oxigenação:
    "O2 saturation pulseoxymetry": -1,    # saturação ↑ → melhor oxigenação → menor gravidade
    "SpO2 Desat Limit": 0,               # limite de desaturação é típico de configuração / contexto, não fisiologia direta

    # Coagulação (tempo parcial de tromboplastina):
    "PTT": +1,                # PTT ↑ → coagulopatia / CID → pior prognóstico

    # Idade:
    "anchor_age": +1,         # idade ↑ → maior risco de infecção/sepse e piores desfechos

    # -------------------------------------------------
    # 2) Variáveis de raça / sexo — NÃO impor direção
    # -------------------------------------------------
    # Por questões éticas e de justiça algorítmica, é mais seguro
    # não impor direção causal nesses atributos (deixar 0) e usar
    # directional soundness só em variáveis clínicas.

    #"race_black": 0,
    #"race_hispanic/latino": 0,
    #"race_other": 0,
    #"race_unknown": 0,
    #"race_white": 0,

    "gender_M": 0,            # diferença de risco entre sexos existe em estudos, mas é melhor não tratar
                              # como “mais é pior” ou “mais é melhor” de forma rígida nesse tipo de métrica.
}


PATH_CSV = Path('../SEPSIS/model_reports/xgboost/basico/csv/')
#X_test_basic_full.csv
#X_train_basic_full.csv
ENTRADAS_SELECIONADAS = pd.read_csv(Path('../SEPSIS/clusters_reports/random_forest/csv/entradas_selecionadas_final_id_only.csv'))

X_train = pd.read_csv(PATH_CSV / 'X_train_basic_full.csv')  
X_test = pd.read_csv(PATH_CSV / 'X_test_basic_full.csv')  

X_train = X_train.drop(X_train.columns[0], axis=1)
X_test = X_test.drop(X_test.columns[0], axis=1)
X_train.rename(columns={"stay_id": "id"}, inplace=True)
X_test.rename(columns={"stay_id": "id"}, inplace=True)  

X = pd.concat([X_train, X_test], ignore_index=True).sort_values(by="id").reset_index(drop=True)
X_ = X.copy()

EXCLUDE_COLS = ['stay_id', 'y_pred', 'y_proba', 'y','septic']
EXCLUDE_COLS = ['id','ID', 'patient_id', 'pid','septic','stay_id', 'race_white', 'race_unknown', 'race_black', 'race_hispanic/latino', 'race_other', 'y','y_proba', 'y_pred']

# ======================================================================
# 1. PREPARAÇÃO INICIAL E CONFIGURAÇÃO
# ======================================================================

# Identifica as colunas que DEVEM ser processadas
FEATURES = [c for c in X.columns if c not in EXCLUDE_COLS]

# Seleciona apenas as features relevantes para o processamento
X_processed_raw = X[FEATURES].copy()
X_train_processed_raw = X_train[FEATURES].copy()
X_ENTRADAS_SELECIONADAS_processed_raw = X[FEATURES].copy()

idisin = ENTRADAS_SELECIONADAS.index.tolist()
X_raw_entradas_selecionadas = X_ENTRADAS_SELECIONADAS_processed_raw.iloc[idisin]

# Detecta tipos de dados nas features selecionadas
num_cols = [c for c in FEATURES if pd.api.types.is_numeric_dtype(X_processed_raw[c])]
cat_cols = [c for c in FEATURES if c not in num_cols]

# ======================================================================
# 2. PIPELINE DE PRÉ-PROCESSAMENTO
# ======================================================================

# Pipelines para dados numéricos e categóricos (exatamente os mesmos do seu código)
num_pipe = Pipeline(steps=[
    ("imp", SimpleImputer(strategy="median")),
])
cat_pipe = Pipeline(steps=[
    ("imp", SimpleImputer(strategy="most_frequent")),
    ("ord", OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=-1,
        encoded_missing_value=-1
    ))
])

# ColumnTransformer para aplicar os pipelines corretos a cada tipo de coluna
preprocess = ColumnTransformer(
    transformers=[
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols),
    ],
    remainder="drop" # Garante que apenas as colunas em FEATURES sejam mantidas
)

# ======================================================================
# 3. APLICAÇÃO DO PRÉ-PROCESSAMENTO
# ======================================================================

# Aplicar o fit_transform se este for o conjunto de treino
# Se você já treinou seu preprocessador em outro lugar, use apenas .transform()

# Exemplo usando fit_transform no seu DataFrame X de entrada:
X_ = preprocess.fit_transform(X_processed_raw)
X_train_ = preprocess.transform(X_train_processed_raw)
X_entradas_selecionadas = preprocess.transform(X_raw_entradas_selecionadas)

# O resultado X_ é um array numpy com seus dados prontos,
# com todas as nulos tratados e categóricas transformadas em números.

# ======================================================================
# 4. VOLTANDO PARA DATAFRAME (AJUSTE MÍNIMO)
# ======================================================================

try:
    # sklearn >= 1.0 geralmente tem isso
    col_names = preprocess.get_feature_names_out()
    # opcional: remover prefixos "num__" / "cat__"
    col_names = [name.split("__", 1)[1] if "__" in name else name
                 for name in col_names]
except Exception:
    # fallback simples: como não houve expansão (ordinal), nº de colunas é o mesmo
    col_names = num_cols + cat_cols

X_df = pd.DataFrame(X_, columns=col_names, index=X_processed_raw.index)
X_train_df = pd.DataFrame(X_train_, columns=col_names, index=X_train_processed_raw.index)


ENTRADAS_SELECIONADAS = ENTRADAS_SELECIONADAS.T.squeeze().to_list() 

model = load("../SEPSIS/common/models/xgboost/v1/xgb_model_cf10_8_1_1_final.pkl")



In [ ]:
# ===  Lime Interpretabilidade ===
# (Se já instalou o LIME no ambiente, pode comentar a linha abaixo)
# !pip install lime

from lime.lime_tabular import LimeTabularExplainer
import numpy as np
import pandas as pd
from pathlib import Path

lime_results = []
explicacoes_raw_add = []

# Defina a lista de features usadas no treino do explainer (sem colunas excluídas):
feature_cols = [c for c in X.columns if c not in EXCLUDE_COLS]

# Explainer com largura de kernel (você pode manter 1.5 se preferir; aqui deixei como no seu código)
lime_explainer = LimeTabularExplainer(
    training_data=X_entradas_selecionadas,
    feature_names=feature_cols,
    class_names=list(model.classes_),
    mode='classification',
    kernel_width=1.5,
    discretize_continuous=False
)

# Wrapper para predict_proba usando as MESMAS colunas do explainer
def _predict_proba(x):
    df = pd.DataFrame(x, columns=feature_cols)
    return model.predict_proba(df)

# Índice da classe positiva (assumindo que o modelo foi treinado com classes 0/1)
if hasattr(model, "classes_") and 1 in model.classes_:
    pos_idx = list(model.classes_).index(1)
else:
    # fallback: usa a maior prob como "positiva"
    pos_idx = 1

for i in range(len(X_raw_entradas_selecionadas)):
    # Use APENAS as features no mesmo ordenamento do explainer
    instancia_feat = X_raw_entradas_selecionadas.iloc[i]

    exp_lime = lime_explainer.explain_instance(
        data_row=instancia_feat,
        predict_fn=_predict_proba,
        num_features=len(feature_cols)  # antes: X.shape[1] (X não existe)
    )

    explicacoes_raw = exp_lime.as_list()

    # Salva explicações "brutas" para debug em algumas linhas específicas (opcional)
    if X_raw_entradas_selecionadas.index[i] in (2, 31):
        explicacoes_raw_add.append(explicacoes_raw)

    # Intercepto do LIME para a classe positiva
    # (exp_lime.intercept tem 1 valor por classe no modo 'classification')
    intercepto = exp_lime.intercept[pos_idx]

    # Probabilidade do modelo para a classe positiva
    prob_modelo = model.predict_proba(pd.DataFrame([instancia_feat], columns=feature_cols))[0][pos_idx]

    # DataFrame das contribuições (feature/intervalo, peso)
    df_explicacoes = pd.DataFrame(explicacoes_raw, columns=["Condicao", "Contribuicao_LIME"])

    # Consolidar contribuição por feature (mesmo se o LIME quebrar em intervalos)
    contrib_por_feature = {}
    for col in feature_cols:  # antes: X_.columns (entravam colunas excluídas)
        match = df_explicacoes[df_explicacoes["Condicao"].str.contains(col, regex=False)]
        contrib_por_feature[col] = match.iloc[0]["Contribuicao_LIME"] if not match.empty else 0.0

    # Checagem: intercepto + soma das contribuições ≈ probabilidade do modelo
    soma_real = sum(contrib_por_feature.values())
    aproximacao_lime = intercepto + soma_real
    erro_aproximacao = abs(prob_modelo - aproximacao_lime)

    # Rótulo verdadeiro (opcional) e mapeamento textual (ajuste se quiser manter 'M'/'B')
    diagnostico = X.iloc[i].septic # (mantive seu texto; mude se preferir 'M'/'B')

    linha_dict = {
        'linha': X.index[i],
        'intercepto': intercepto,
        'soma_LIME_check': soma_real,
        'aproximacao_LIME': aproximacao_lime,
        'prob_modelo': prob_modelo,
        'erro_aproximacao': erro_aproximacao,
        'diagnostico': diagnostico
    }
    linha_dict.update(contrib_por_feature)
    lime_results.append(linha_dict)

df_lime_completo = pd.DataFrame(lime_results)

# Exporta com precisão (to_excel pode ignorar float_format em algumas versões do pandas; fazemos round antes)
df_lime_export = df_lime_completo.copy()
for c in df_lime_export.columns:
    if pd.api.types.is_float_dtype(df_lime_export[c]):
        df_lime_export[c] = df_lime_export[c].round(10)
base_path_obj = Path(BASE_PATH)
output_file = base_path_obj / f"{base_path_obj.name}_lime_completo.xlsx"
df_lime_export.to_excel(output_file, index=False)
print(f"✅ Arquivo salvo com sucesso em: {output_file}")
output_file = base_path_obj / f"{base_path_obj.name}_lime_completo.csv"
df_lime_export.to_csv(output_file, index=False)
print(f"✅ Arquivo salvo com sucesso em: {output_file}")

In [ ]:
# ===  Lime Interpretabilidade [Funções de Calculo das Métricas]===
# @title
#!pip install torch
#import torch
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics.pairwise import cosine_similarity
from lime.lime_tabular import LimeTabularExplainer
import re
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import Ridge

from sklearn.metrics import pairwise_distances
from typing import Sequence, Optional, Dict, List
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")
import numpy as np

from sklearn.model_selection import train_test_split

from scipy.stats import kendalltau, spearmanr, pearsonr
from typing import Optional, List, Dict, Tuple, Sequence

def calcula_fidelity_lime_all(
    x_row,
    explainer,
    classifier_fn,          # ex.: lambda X: model.predict_proba(pd.DataFrame(X, columns=feat_cols_eff))
    X_ref,                  # DataFrame para contar d_bar e padronizar (use X ou X_train)
    drop_cols=('classe','y'),
    num_samples=5000,
    pos_class=1,
    alpha=1.0,              # regularização L2 do surrogate linear
    k_features=None,        # ex.: 10 para top-K (aproxima K-LASSO). None = sem sparsidade
    eval_mode='holdout',    # 'holdout' (recomendado) ou 'train'
    test_size=0.30,
    random_state=42,
    sigma=None,             # None -> 0.75*sqrt(d_bar); float; ou 'from_explainer'
    *,
    decimals=8              # casas decimais para as perdas agregadas
):
    """
    Retorna um dicionário com:
      - 'fidelity': métricas de fidelidade do surrogate ao modelo
          * r2_abs / r2_percent: R² ponderado (maior=melhor; pode ser < 0)
          * mse_mean, mae_mean, logloss_mean: perdas médias (menor=melhor)
          * opcionalmente "mse_fid", "mae_fid" como 1 - loss (maior=melhor)
      - 'loss'    : perdas agregadas (média ponderada e soma ponderada) para MSE, MAE e LogLoss
      - 'meta'    : informações auxiliares (n_test, soma de pesos etc.)
    """
    import numpy as np, pandas as pd
    from sklearn.preprocessing import StandardScaler
    from sklearn.metrics import pairwise_distances
    from sklearn.linear_model import Ridge
    from sklearn.model_selection import train_test_split

    EPS = 1e-12

    # 1) Espaço efetivo e σ
    feat_cols_eff = [c for c in X_ref.columns if c not in set(drop_cols)]
    if not feat_cols_eff:
        raise ValueError("Nenhuma feature efetiva após remover {'classe','y'}.")
    d_bar = len(feat_cols_eff)

    if sigma == 'from_explainer' and hasattr(explainer, 'kernel_width'):
        sigma_val = float(explainer.kernel_width)
    else:
        sigma_val = (0.75 * np.sqrt(d_bar)) if (sigma is None) else float(sigma)

    # 2) Formatos e scaler
    x_df = pd.DataFrame(
        np.asarray(x_row, dtype=float).reshape(1, -1),
        columns=feat_cols_eff
    )
    scaler = StandardScaler().fit(X_ref[feat_cols_eff].astype(float))

    # 3) Vizinhança do LIME (ou fallback)
    try:
        exp, Z, distances = explainer.explain_instance(
            x_df.values.ravel(),
            classifier_fn,
            num_features=d_bar,
            num_samples=num_samples,
            labels=[pos_class],
            return_data=True,
            distance_metric='euclidean'
        )
        Z = np.asarray(Z, dtype=float)
        distances = None if distances is None else np.asarray(distances, dtype=float).ravel()
    except TypeError:
        std = X_ref[feat_cols_eff].astype(float).std(ddof=0).replace(0, 1e-6).values
        n_rest = max(1, num_samples - 1)
        noise = np.random.normal(0.0, 0.01*(std + EPS), size=(n_rest, d_bar))
        Z = np.vstack([x_df.values, x_df.values + noise])
        distances = None

    # 4) Pesos do kernel no espaço padronizado
    Z_df  = pd.DataFrame(Z, columns=feat_cols_eff)
    Z_std = scaler.transform(Z_df)
    x_std = scaler.transform(x_df)
    if distances is None:
        distances = pairwise_distances(Z_std, x_std, metric='euclidean').ravel()
    weights = np.exp(-(distances**2) / (sigma_val**2))
    # sw = float(np.sum(weights)) + EPS   # (não usado diretamente aqui)

    # 5) f(z): saída do modelo (probabilidade)
    fz = np.asarray(classifier_fn(Z_df.values), dtype=float)[:, pos_class]
    fz = np.clip(fz, EPS, 1 - EPS)

    # 6) Split treino/teste
    if eval_mode == 'holdout':
        idx = np.arange(Z_std.shape[0])
        idx_tr, idx_te = train_test_split(
            idx,
            test_size=test_size,
            random_state=random_state,
            shuffle=True
        )
    else:
        idx_tr = np.arange(Z_std.shape[0])
        idx_te = idx_tr

    Ztr, Zte = Z_std[idx_tr], Z_std[idx_te]
    ftr, fte = fz[idx_tr], fz[idx_te]
    wtr, wte = weights[idx_tr], weights[idx_te]
    sw_te = float(np.sum(wte)) + EPS

    # 7) Surrogate linear local (Ridge) + opcional top-K
    reg = Ridge(alpha=alpha, fit_intercept=True)
    reg.fit(Ztr, ftr, sample_weight=wtr)
    if (k_features is not None) and (0 < k_features < Ztr.shape[1]):
        top_idx = np.argsort(np.abs(reg.coef_))[::-1][:int(k_features)]
        reg_k = Ridge(alpha=alpha, fit_intercept=True)
        reg_k.fit(Ztr[:, top_idx], ftr, sample_weight=wtr)
        gte = np.clip(reg_k.predict(Zte[:, top_idx]), EPS, 1 - EPS)
    else:
        gte = np.clip(reg.predict(Zte), EPS, 1 - EPS)

    # 8) Perdas ponto-a-ponto (modelo vs surrogate)
    diff   = fte - gte
    ell_mse = diff**2
    ell_mae = np.abs(diff)
    ell_ll  = -(fte*np.log(gte) + (1 - fte)*np.log(1 - gte))

    # 9) Agregações ponderadas
    mse_sum  = float(np.sum(wte * ell_mse))
    mse_mean = float(mse_sum / sw_te)
    mae_sum  = float(np.sum(wte * ell_mae))
    mae_mean = float(mae_sum / sw_te)
    ll_sum   = float(np.sum(wte * ell_ll))
    ll_mean  = float(ll_sum / sw_te)

    # 10) R² ponderado (maior=melhor; pode ser < 0)
    fw  = float(np.sum(wte * fte) / sw_te)         # média ponderada de fte
    sse = mse_sum                                  # soma dos erros quadráticos
    sst = float(np.sum(wte * (fte - fw)**2)) + EPS # variância ponderada
    r2w = 1.0 - sse/sst

    # 11) Scores de fidelidade
    #    - definição mais próxima do espírito do LIME:
    #      * usar MSE/MAE/LogLoss como perdas (menor=melhor)
    #      * usar R² como medida principal de fidelidade (maior=melhor)
    #    Opcionalmente, 1 - loss como "score" em [~0,1].

    # Como f está em [0,1], MSE e MAE tendem a estar em [0,1], então 1-loss faz sentido.
    fid_mse_abs = 1.0 - mse_mean        # pode ser <0 se MSE > 1 (ruim mesmo)
    fid_mae_abs = 1.0 - mae_mean        # idem
    fid_ll_abs  = -ll_mean              # maior=melhor (menor perda logarítmica)
    fid_r2_abs  = r2w                   # pode ser negativo

    out = {
        "fidelity": {
            # PRINCIPAL (paper LIME costuma olhar para erro e R²):
            "r2_abs":       fid_r2_abs,
            "r2_percent":   round(fid_r2_abs * 100.0, 3),

            # Esses três são *losses médios* (menor=melhor),
            # mas também devolvo a forma "1 - loss" para você poder usar como score.
            "mse_mean":     mse_mean,
            "mae_mean":     mae_mean,
            "logloss_mean": ll_mean,

            "mse_fid_abs":  fid_mse_abs,
            "mse_fid_percent": round(fid_mse_abs * 100.0, 3),
            "mae_fid_abs":  fid_mae_abs,
            "mae_fid_percent": round(fid_mae_abs * 100.0, 3),
            "logloss_fid_abs": fid_ll_abs,
            "logloss_fid_percent": round(fid_ll_abs * 100.0, 3),

            # R² bruto (se quiser ver negativo explicitamente)
            "r2_raw":       r2w,
        },
        "loss": {
            "mse_mean":     round(mse_mean, decimals),
            "mse_sum":      round(mse_sum,  decimals),
            "mae_mean":     round(mae_mean, decimals),
            "mae_sum":      round(mae_sum,  decimals),
            "logloss_mean": round(ll_mean,  decimals),
            "logloss_sum":  round(ll_sum,   decimals),
        },
        "meta": {
            "n_train": int(len(idx_tr)),
            "n_test":  int(len(idx_te)),
            "sum_weights_test": float(sw_te),
            "sigma": float(sigma_val),
            "d_bar": int(d_bar)
        }
    }
    return out

def debug_infidelity_displays(details, top_k=10, title_prefix=""):
    import matplotlib.pyplot as plt
    import numpy as np

    pred = details["pred_delta"]
    true = details["true_delta"]
    w    = details["weights_test"]
    err  = details["err_sq"]**0.5

    # A) Scatter: delta previsto vs. delta real
    plt.figure(figsize=(6,6))
    plt.scatter(true, pred, s=20, alpha=0.7)
    lims = [min(true.min(), pred.min()), max(true.max(), pred.max())]
    plt.plot(lims, lims, 'g--', label='Ideal y=x')
    plt.xlabel("Δ real = f(x) − f(z)")
    plt.ylabel("Δ previsto = Iᵀ Φ(x)")
    plt.title(f"{title_prefix}Infidelity — Δ previsto vs Δ real\nR²Δ(peso) = {details['r2_delta']:.3f}")
    plt.legend(); plt.grid(True, ls='--', alpha=0.4); plt.tight_layout(); plt.show()

    # B) Histograma dos resíduos (ponderado pelo peso via replicação aproximada)
    # (para visual só, não é exato; se quiser algo exato, use barras com alturas = soma de pesos no bin)
    reps = np.clip((w / w.max()) * 100, 1, None).astype(int)
    resid = np.repeat((pred - true), reps)
    plt.figure(figsize=(8,4))
    plt.hist(resid, bins=30, edgecolor='black', alpha=0.85)
    plt.axvline(np.mean(resid), color='r', ls='--', label=f"média ≈ {np.mean(resid):.4f}")
    plt.xlabel("Resíduo (Δ previsto − Δ real)")
    plt.ylabel("Frequência (aprox. ponderada)")
    plt.title(f"{title_prefix}Distribuição de resíduos (aprox. ponderada)")
    plt.legend(); plt.grid(True, ls='--', alpha=0.4); plt.tight_layout(); plt.show()

    # C) Barras: features com maior |I_j * Φ_j| médio ponderado
    s = details["abs_contrib_sorted"]
    if s is not None and len(s) > 0:
        s_head = s.head(top_k)
        plt.figure(figsize=(10,4))
        plt.bar(s_head.index, s_head.values, edgecolor='black')
        plt.xticks(rotation=45, ha='right')
        plt.ylabel("média ponderada de |I_j · Φ_j|")
        plt.title(f"{title_prefix}Top-{min(top_k, len(s_head))} features (|I·Φ| médio)")
        plt.grid(True, ls='--', alpha=0.4); plt.tight_layout(); plt.show()
        
def calcula_infidelity_lime_strict(
    x_row,
    explainer,
    classifier_fn,      # ex.: lambda X: model.predict_proba(pd.DataFrame(X, columns=feat_cols_eff))
    X_ref,              # DataFrame para padronizar e contar d_bar (use X ou X_train)
    drop_cols=('classe','y'),
    num_samples=5000,
    pos_class=1,
    alpha=1.0,          # regularização do linear local
    k_features=None,    # None = sem sparsidade; int = top-K |coef|
    eval_mode='holdout',# 'holdout' (recomendado) ou 'train'
    test_size=0.30,
    random_state=42,
    sigma=None,         # None -> 0.75*sqrt(d_bar); float; ou 'from_explainer'
    return_details=True, # retorna DataFrames e info auxiliar
    use_kernel_weights: bool = True,
    # --- NOVOS CONTROLES ---
    expectation_mode: str = "gaussian",  # "gaussian" (canônico) | "from_Z" (LIME-local)
    target: str = "logit",               # "logit" (recomendado) | "prob"
    noise_scale: float = 1.0             # desvio padrão de I no espaço padronizado
):
    """
    Implementa:  Infid(Φ,f,x) = E_I[( I^T Φ(x) − (f_t(x) − f_t(x−I)) )^2]
      - I ~ N(0, noise_scale^2 I) no espaço padronizado quando expectation_mode="gaussian".
      - f_t(·): alvo na escala definida por 'target' ("prob" ou "logit").

    Retornos:
      - infidelity_abs        : MSE ponderado (menor=melhor)
      - infidelity_pct_small  : porcentagem em 0–100 (MENOR=melhor)
      - details               : dict com métricas auxiliares (inclui o score original “maior=melhor”)
    """
    import numpy as np, pandas as pd
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import Ridge
    from sklearn.metrics import pairwise_distances
    from sklearn.model_selection import train_test_split

    EPS = 1e-12
    target = str(target).lower()
    expectation_mode = str(expectation_mode).lower()

    def _to_target(u: np.ndarray) -> np.ndarray:
        u = np.clip(u, EPS, 1.0 - EPS)
        if target == "logit":
            return np.log(u / (1.0 - u))
        return u  # "prob"

    # 1) Espaço efetivo e σ (para treino do surrogate)
    feat_cols_eff = [c for c in X_ref.columns if c not in set(drop_cols)]
    if not feat_cols_eff:
        raise ValueError("Nenhuma feature efetiva após remover {'classe','y'}.")
    d_bar = len(feat_cols_eff)

    if sigma == 'from_explainer' and hasattr(explainer, 'kernel_width'):
        sigma_val = float(explainer.kernel_width)
    else:
        sigma_val = (0.75 * np.sqrt(d_bar)) if (sigma is None) else float(sigma)

    # 2) Formatos + scaler
    x_df  = pd.DataFrame(np.asarray(x_row, dtype=float).reshape(1, -1), columns=feat_cols_eff)
    scaler = StandardScaler().fit(X_ref[feat_cols_eff].astype(float))

    # 3) Vizinhança do LIME (ou fallback suave) — usada para TREINO do surrogate
    try:
        exp, Z, distances = explainer.explain_instance(
            x_df.values.ravel(),
            classifier_fn,
            num_features=d_bar,
            num_samples=num_samples,
            labels=[pos_class],
            return_data=True,
            distance_metric='euclidean'
        )
        Z = np.asarray(Z, dtype=float)
        distances = None if distances is None else np.asarray(distances, dtype=float).ravel()
    except TypeError:
        std = X_ref[feat_cols_eff].astype(float).std(ddof=0).replace(0, 1e-6).values
        n_rest = max(1, num_samples - 1)
        noise  = np.random.normal(0.0, 0.01*(std + EPS), size=(n_rest, d_bar))
        Z = np.vstack([x_df.values, x_df.values + noise])
        distances = None

    # 4) Pesos do kernel no espaço padronizado (se usar no treino)
    Z_df  = pd.DataFrame(Z, columns=feat_cols_eff)
    Z_std = scaler.transform(Z_df)
    x_std = scaler.transform(x_df)
    if distances is None:
        distances = pairwise_distances(Z_std, x_std, metric='euclidean').ravel()
    weights = np.exp(-(distances**2) / (sigma_val**2)) if use_kernel_weights else np.ones_like(distances, dtype=float)

    # 5) f(x) e f(Z) — probabilidades e na escala alvo
    fz_prob = np.asarray(classifier_fn(Z_df.values), dtype=float)[:, pos_class]
    fz_prob = np.clip(fz_prob, EPS, 1 - EPS)
    fx_prob = float(np.clip(classifier_fn(x_df.values)[0, pos_class], EPS, 1 - EPS))
    fz_t = _to_target(fz_prob)
    fx_t = float(_to_target(np.array([fx_prob])))

    # 6) Split treino/teste (para obter Φ(x) sem superestimar)
    idx_all = np.arange(Z_std.shape[0])
    if eval_mode == 'holdout':
        idx_tr, idx_te = train_test_split(idx_all, test_size=test_size, random_state=random_state, shuffle=True)
    else:
        idx_tr = idx_all
        idx_te = idx_all

    Ztr, Zte = Z_std[idx_tr], Z_std[idx_te]
    wtr, wte = weights[idx_tr], weights[idx_te]

    # 7) Regressão linear local ponderada → Φ(x) (treino na escala 'target')
    ytr = fz_t[idx_tr]
    reg = Ridge(alpha=alpha, fit_intercept=True)
    reg.fit(Ztr, ytr, sample_weight=wtr if use_kernel_weights else None)
    coef = reg.coef_.copy()  # Φ(x)

    # Sparsidade opcional (top-K) para o dot I^T Φ(x)
    use_mask = np.ones_like(coef, dtype=bool)
    if (k_features is not None) and (0 < k_features < Ztr.shape[1]):
        top_idx = np.argsort(np.abs(coef))[::-1][:int(k_features)]
        use_mask = np.zeros_like(coef, dtype=bool); use_mask[top_idx] = True

    # 8) Constrói I e define o conjunto de expectativa
    if expectation_mode == "from_Z":
        I_std = Zte - x_std
        fte_prob = fz_prob[idx_te]
        fte_t    = fz_t[idx_te]
        w_exp    = wte.astype(float)
    else:
        n_exp = max(1024, int(num_samples * 0.5))
        rng = np.random.default_rng(random_state)
        I_std = rng.normal(0.0, noise_scale, size=(n_exp, Ztr.shape[1]))
        Zte_canon_std = x_std + I_std
        Zte_canon = scaler.inverse_transform(Zte_canon_std)
        Zte_canon_df = pd.DataFrame(Zte_canon, columns=feat_cols_eff)
        fte_prob = np.asarray(classifier_fn(Zte_canon_df.values), dtype=float)[:, pos_class]
        fte_prob = np.clip(fte_prob, EPS, 1 - EPS)
        fte_t    = _to_target(fte_prob)
        w_exp    = np.ones(n_exp, dtype=float)

    # 9) Δ previsto e Δ real (na escala 'target')
    delta_pred = (I_std[:, use_mask] @ coef[use_mask])   # I^T Φ(x)
    delta_real = fx_t - fte_t                            # f_t(x) − f_t(x−I)

    # 10) Infidelity = MSE ponderado de (Δpred − Δreal)
    resid = delta_pred - delta_real
    sw_te = float(np.sum(w_exp)) + EPS
    EI_resid2 = float(np.sum(w_exp * (resid**2)) / sw_te)   # menor=melhor
    infid_abs = EI_resid2

    # 11) Scores auxiliares (ponderados) — original (maior=melhor) e invertido (menor=melhor)
    mu_dr  = float(np.sum(w_exp * delta_real) / sw_te)
    sse    = float(np.sum(w_exp * (delta_real - delta_pred)**2))
    sst    = float(np.sum(w_exp * (delta_real - mu_dr )**2)) + EPS
    r2_delta = 1.0 - sse/sst
    infid_pct_orig = round(max(0.0, r2_delta) * 100.0, 3)               # MAIOR = melhor (histórico)
    EI_resid2_score_pct = 100.0 * (sst / (sst + EI_resid2 + EPS))        # MAIOR = melhor (SNR-like)

    # --------- NOVO: percent em que MENOR = melhor ---------
    # Opção A (simples e estável): inverter o score original
    infid_pct_small = float(np.clip(100.0 - infid_pct_orig, 0.0, 100.0))
    # (Se preferir usar o SNR-like: 100 - EI_resid2_score_pct)

    if not return_details:
        return infid_pct_small, infid_abs

    # 12) DataFrame por amostra (para auditoria)
    df_samples = pd.DataFrame({
        "weight":      w_exp,
        "fx":          fx_prob,
        "fz":          fte_prob,
        "delta_real":  delta_real,
        "delta_pred":  delta_pred,
        "resid":       resid,
        "resid2":      resid**2
    })

    # 13) Contribuições por feature (|I_j * Φ_j| médio ponderado) p/ ranking
    contrib  = I_std * coef
    abs_mean = (np.abs(contrib) * w_exp.reshape(-1,1)).sum(axis=0) / sw_te
    coef_df = pd.DataFrame({
        "feature": feat_cols_eff,
        "phi":     coef,
        "abs_mean_Iphi": abs_mean,
        "used":    use_mask
    }).sort_values("abs_mean_Iphi", ascending=False).reset_index(drop=True)

    details = {
        "EI_resid2": EI_resid2,
        "EI_resid2_score_pct": EI_resid2_score_pct,  # MAIOR=melhor (SNR-like)
        "infidelity_pct_orig": infid_pct_orig,       # MAIOR=melhor (histórico)
        "infidelity_pct_small": infid_pct_small,     # MENOR=melhor (novo retorno)
        "var_delta_real_w": sst,
        "r2_delta":  r2_delta,
        "coef_df":   coef_df,
        "samples":   df_samples,
        "sigma":     sigma_val,
        "d_bar":     d_bar,
        "mode":      expectation_mode,
        "target":    target,
        "noise_scale": float(noise_scale)
    }
    return infid_pct_small, infid_abs, details





def calcula_faithfulness_lime(
    x_row,
    explainer,
    classifier_fn,          # ex.: lambda X: model.predict_proba(pd.DataFrame(X, columns=feat_cols_eff))
    X_ref,                  # DataFrame p/ padronizar e obter valores de referência
    drop_cols=('classe','y'),
    num_samples=5000,
    pos_class=1,
    alpha=1.0,              # regularização do surrogate local
    k_features=None,        # None=usa todas; int=Top-K (as outras viram 0)
    eval_mode='holdout',    # 'holdout' (recomendado) ou 'train'
    test_size=0.30,
    random_state=42,
    sigma='from_explainer', # None=>0.75*sqrt(d̄); 'from_explainer'=>kernel_width do LIME; ou float
    ref_strategy='mean',    # como “remover” a feature: 'mean'|'median'|'zero'|'mode' ou dict col->valor
    corr='kendall',         # 'kendall' (tau-b), 'spearman' ou 'pearson'
    return_details=False
):
    """
    Retorna (faith_percent, faith_abs, corr_raw [, details]) para a instância x_row.
      - corr_raw ∈ [-1,1] é a correlação pedida (Kendall/Spearman/Pearson).
      - faith_abs ∈ [0,1] = (corr_raw+1)/2 (MAIOR=melhor), faith_percent com 3 casas.
    """
    EPS = 1e-12

    # 1) features efetivas e σ
    feat_cols_eff = [c for c in X_ref.columns if c not in set(drop_cols)]
    if not feat_cols_eff:
        raise ValueError("Nenhuma feature efetiva após remover {'classe','y'}.")
    d_bar = len(feat_cols_eff)

    if sigma is None:
        sigma_val = 0.75 * np.sqrt(d_bar)
    elif isinstance(sigma, str) and sigma.lower() == 'from_explainer' and hasattr(explainer, 'kernel_width'):
        sigma_val = float(explainer.kernel_width)
    elif isinstance(sigma, (int, float)):
        sigma_val = float(sigma)
    else:
        sigma_val = 0.75 * np.sqrt(d_bar)

    # 2) formatos/escala
    x_df = pd.DataFrame(np.asarray(x_row, dtype=float).reshape(1, -1), columns=feat_cols_eff)
    scaler = StandardScaler().fit(X_ref[feat_cols_eff].astype(float))

    # 3) vizinhança do LIME (ou fallback)
    try:
        exp, Z, distances = explainer.explain_instance(
            x_df.values.ravel(),
            classifier_fn,
            num_features=d_bar,
            num_samples=num_samples,
            labels=[pos_class],
            return_data=True,
            distance_metric='euclidean'
        )
        Z = np.asarray(Z, dtype=float)
        distances = None if distances is None else np.asarray(distances, dtype=float).ravel()
    except TypeError:
        std = X_ref[feat_cols_eff].astype(float).std(ddof=0).replace(0, 1e-6).values
        n_rest = max(1, num_samples - 1)
        Z = np.vstack([x_df.values,
                       x_df.values + np.random.normal(0.0, 0.05 * (std + 1e-12), size=(n_rest, d_bar))])
        distances = None
        exp = None

    # 4) pesos do kernel (no espaço padronizado) — usados só para treinar o surrogate
    Z_df  = pd.DataFrame(Z, columns=feat_cols_eff)
    Z_std = scaler.transform(Z_df)
    x_std = scaler.transform(x_df)
    if distances is None:
        distances = pairwise_distances(Z_std, x_std, metric='euclidean').ravel()
    weights = np.exp(-(distances**2) / (sigma_val**2))

    # 5) f(z) e split treino/teste
    fz = np.asarray(classifier_fn(Z_df.values), dtype=float)[:, pos_class]
    fz = np.clip(fz, EPS, 1 - EPS)

    if eval_mode == 'holdout':
        idx_all = np.arange(Z_std.shape[0])
        idx_tr, idx_te = train_test_split(idx_all, test_size=test_size,
                                          random_state=random_state, shuffle=True)
    else:
        idx_tr = np.arange(Z_std.shape[0]); idx_te = idx_tr

    Ztr, ftr, wtr = Z_std[idx_tr], fz[idx_tr], weights[idx_tr]

    # 6) surrogate linear local (Ridge)
    reg = Ridge(alpha=alpha, fit_intercept=True)
    reg.fit(Ztr, ftr, sample_weight=wtr)
    phi = reg.coef_.copy()              # w_i(x) no espaço padronizado

    # Top-K opcional (zera coeficientes fora do conjunto)
    mask = np.ones_like(phi, dtype=bool)
    if (k_features is not None) and (0 < k_features < len(phi)):
        top_idx = np.argsort(np.abs(phi))[::-1][:int(k_features)]
        mask = np.zeros_like(phi, dtype=bool); mask[top_idx] = True
        phi[~mask] = 0.0

    # 7) vetor de impactos Δ_i = f(x) - f(x_{-i})
    # referência por coluna
    if isinstance(ref_strategy, dict):
        ref_vals = {c: float(ref_strategy.get(c, X_ref[c].astype(float).mean())) for c in feat_cols_eff}
    else:
        if ref_strategy == 'median':
            ref_vals = X_ref[feat_cols_eff].astype(float).median().to_dict()
        elif ref_strategy == 'zero':
            ref_vals = {c: 0.0 for c in feat_cols_eff}
        elif ref_strategy == 'mode':
            ref_vals = {c: float(X_ref[c].mode(dropna=True)[0]) if not X_ref[c].mode(dropna=True).empty else float(X_ref[c].astype(float).mean())
                        for c in feat_cols_eff}
        else:  # 'mean' (default)
            ref_vals = X_ref[feat_cols_eff].astype(float).mean().to_dict()

    fx = float(np.asarray(classifier_fn(x_df.values), dtype=float)[0, pos_class])
    fx = float(np.clip(fx, EPS, 1 - EPS))

    delta = []
    for col in feat_cols_eff:
        xm = x_df.copy()
        xm[col] = ref_vals[col]
        f_xminus_i = float(np.asarray(classifier_fn(xm.values), dtype=float)[0, pos_class])
        f_xminus_i = float(np.clip(f_xminus_i, EPS, 1 - EPS))
        delta.append(fx - f_xminus_i)
    delta = np.asarray(delta, dtype=float)

    # 8) correlação entre w_i(x) e Δ_i (apenas features consideradas)
    w_use = phi.copy()
    d_use = delta.copy()
    if mask is not None:
        w_use = w_use[mask]
        d_use = d_use[mask]

    # se tudo constante, define correlação 0
    if (np.allclose(w_use, w_use[0])) or (np.allclose(d_use, d_use[0])):
        corr_raw = 0.0
        pval = np.nan
    else:
        if corr == 'spearman':
            corr_raw, pval = spearmanr(w_use, d_use)
        elif corr == 'pearson':
            corr_raw, pval = pearsonr(w_use, d_use)
        else:  # 'kendall' (tau-b)
            corr_raw, pval = kendalltau(w_use, d_use, variant='b')

        # NaNs podem surgir com muitos empates; trate como 0
        if not np.isfinite(corr_raw):
            corr_raw, pval = 0.0, np.nan

    faith_abs = float(np.clip((corr_raw + 1.0) / 2.0, 0.0, 1.0))  # [0,1], MAIOR=melhor
    faith_percent = float(f"{faith_abs * 100:.3f}")

    if not return_details:
        return faith_percent, faith_abs, corr_raw

    details = {
        "feat_cols": feat_cols_eff,
        "phi": phi,
        "mask_used": mask,
        "ref_vals": ref_vals,
        "fx": fx,
        "delta": delta,
        "corr_raw": corr_raw,
        "p_value": pval,
        "corr_type": corr,
    }
    return faith_percent, faith_abs, corr_raw, details



# def calcula_simplicity_lime(
#     x_row,
#     explainer,
#     classifier_fn,          # ex.: lambda X: model.predict_proba(pd.DataFrame(X, columns=feat_cols_eff))
#     X_ref,                  # DataFrame para padronizar e contar d̄ (use X ou X_train)
#     drop_cols=('classe','y'),
#     num_samples=5000,
#     pos_class=1,
#     alpha=1.0,              # regularização do surrogate (Ridge)
#     k_features=None,        # se informado, refita no Top-K (como K-LASSO do LIME)
#     tau=0.05,               # pode ser 0.05 (5%) ou 5 (interpreta como 5%)
#     sigma='from_explainer', # None => 0.75*sqrt(d̄); float; ou 'from_explainer'
#     return_details=False
# ):
#     """
#     Complexity_LIME(x) baseada em τ como PERCENTUAL da massa L1 das importâncias:
#         p_i = |w_i| / sum_j |w_j|
#         k_tau = #{ i : p_i > τ }

#     Onde:
#       - τ pode ser dado como 0.05 (5%) ou 5 (5%). Ambos funcionam.
#       - d̄ = nº de features efetivas (após remover drop_cols).
#       - Simplicity_orig(x)  = 1 - k_tau / d̄   (0..1; MAIOR = mais simples)
#       - Simplicity_small(%) = 100 - 100*Simplicity_orig(x)  (0..100; MENOR = melhor)

#     Retorna (k_tau, simplicity_percent_small[, details]).
#     """
#     import numpy as np, pandas as pd
#     from sklearn.preprocessing import StandardScaler
#     from sklearn.linear_model import Ridge
#     from sklearn.metrics import pairwise_distances
#     from sklearn.model_selection import train_test_split

#     EPS = 1e-12

#     # 1) espaço efetivo e sigma
#     feat_cols_eff = [c for c in X_ref.columns if c not in set(drop_cols)]
#     if not feat_cols_eff:
#         raise ValueError("Nenhuma feature efetiva após remover {'classe','y'}.")
#     d_bar = len(feat_cols_eff)

#     # --- PATCH ROBUSTO PARA sigma ---
#     if sigma is None:
#         sigma_val = 0.75 * np.sqrt(d_bar)
#     elif isinstance(sigma, (int, float, np.floating)):
#         sigma_val = float(sigma)
#     elif isinstance(sigma, str):
#         if sigma.lower() == 'from_explainer':
#             sigma_val = float(getattr(explainer, 'kernel_width', 0.75 * np.sqrt(d_bar)))
#         else:
#             raise ValueError(f"sigma string inválido: {sigma!r}. Use 'from_explainer', None ou numérico.")
#     else:
#         sigma_val = 0.75 * np.sqrt(d_bar)

#     # 2) formatos e scaler
#     x_df = pd.DataFrame(np.asarray(x_row, dtype=float).reshape(1, -1), columns=feat_cols_eff)
#     scaler = StandardScaler().fit(X_ref[feat_cols_eff].astype(float))

#     # 3) vizinhança do LIME (ou fallback)
#     try:
#         exp, Z, distances = explainer.explain_instance(
#             x_df.values.ravel(),
#             classifier_fn,
#             num_features=d_bar,
#             num_samples=num_samples,
#             labels=[pos_class],
#             return_data=True,
#             distance_metric='euclidean'
#         )
#         Z = np.asarray(Z, dtype=float)
#         distances = None if distances is None else np.asarray(distances, dtype=float).ravel()
#     except TypeError:
#         std = X_ref[feat_cols_eff].astype(float).std(ddof=0).replace(0, 1e-6).values
#         n_rest = max(1, num_samples - 1)
#         Z = np.vstack([x_df.values,
#                        x_df.values + np.random.normal(0.0, 0.05*(std + EPS), size=(n_rest, d_bar))])
#         distances = None

#     # 4) pesos do kernel (espaço padronizado) – só para treinar o surrogate
#     Z_df  = pd.DataFrame(Z, columns=feat_cols_eff)
#     Z_std = scaler.transform(Z_df)
#     x_std = scaler.transform(x_df)
#     if distances is None:
#         distances = pairwise_distances(Z_std, x_std, metric='euclidean').ravel()
#     weights = np.exp(-(distances**2) / (sigma_val**2))

#     # 5) alvo local f(z)
#     fz = np.asarray(classifier_fn(Z_df.values), dtype=float)[:, pos_class]

#     # 6) ajusta o surrogate linear local
#     reg = Ridge(alpha=alpha, fit_intercept=True)
#     reg.fit(Z_std, fz, sample_weight=weights)
#     w_full = reg.coef_.copy()  # coeficientes no espaço padronizado

#     # 7) opcional: Top-K (aproxima K-LASSO)
#     if (k_features is not None) and (0 < k_features < len(w_full)):
#         top_idx = np.argsort(np.abs(w_full))[::-1][:int(k_features)]
#         mask = np.zeros_like(w_full, dtype=bool); mask[top_idx] = True
#         reg_k = Ridge(alpha=alpha, fit_intercept=True)
#         reg_k.fit(Z_std[:, mask], fz, sample_weight=weights)
#         w = np.zeros_like(w_full); w[mask] = reg_k.coef_
#         topk_count = int(mask.sum())
#     else:
#         w = w_full
#         mask = np.ones_like(w, dtype=bool)
#         topk_count = None

#     # 8) complexidade e simplicidade — τ como % da massa L1
#     abs_w = np.abs(w)
#     l1_sum = float(abs_w.sum() + EPS)
#     p_l1 = abs_w / l1_sum

#     tau_eff = float(tau)
#     if tau_eff > 1.0:      # aceita 5 => 0.05
#         tau_eff = tau_eff / 100.0

#     above_tau = p_l1 > tau_eff
#     complexity_k    = int(np.sum(above_tau))
#     complexity_pct  = 100.0 * complexity_k / d_bar

#     simplicity_orig_abs = 1.0 - (complexity_k / d_bar)         # 0..1 (MAIOR = mais simples)
#     simplicity_orig_pct = 100.0 * simplicity_orig_abs           # 0..100 (MAIOR = melhor)
#     # -------- NOVO RETORNO: MENOR = melhor --------
#     simplicity_pct_small = float(np.clip(100.0 - simplicity_orig_pct, 0.0, 100.0))

#     if not return_details:
#         return complexity_k, round(simplicity_pct_small, 3)

#     # 9) DataFrame de detalhes por feature (ordenado por p_l1 desc)
#     rank = p_l1.argsort()[::-1].argsort() + 1
#     feat_df = pd.DataFrame({
#         "feature":      feat_cols_eff,
#         "w_full":       w_full,
#         "w_used":       w,
#         "abs_w":        abs_w,
#         "p_l1":         p_l1,              # fração (0..1) da massa L1 por feature
#         "above_tau_pct": above_tau,
#         "rank_p_l1":    rank
#     }).sort_values("p_l1", ascending=False).reset_index(drop=True)

#     details = {
#         "feature_table":        feat_df,
#         "tau_input":            float(tau),
#         "tau_eff_fraction":     float(tau_eff),
#         "norm_kind":            "l1_pct",
#         "d_bar":                int(d_bar),
#         "sigma":                float(sigma_val),
#         "complexity_k":         complexity_k,
#         "complexity_percent":   float(round(complexity_pct, 3)),
#         # valores para auditoria/compatibilidade:
#         "simplicity_orig_abs":  float(round(simplicity_orig_abs, 8)),    # MAIOR = melhor (0..1)
#         "simplicity_orig_pct":  float(round(simplicity_orig_pct, 3)),    # MAIOR = melhor (0..100)
#         "simplicity_small_pct": float(round(simplicity_pct_small, 3)),   # MENOR = melhor (0..100)
#         "mask_topk":            mask,
#         "topk_count":           topk_count,
#         "l1_sum":               l1_sum
#     }
#     return complexity_k, round(simplicity_pct_small, 3), details

def calcula_simplicity_lime(
    x_row,
    explainer,
    classifier_fn,          # ex.: lambda X: model.predict_proba(pd.DataFrame(X, columns=feat_cols_eff))
    X_ref,                  # DataFrame para padronizar e contar d̄ (use X ou X_train)
    drop_cols=('classe','y'),
    num_samples=5000,
    pos_class=1,
    alpha=1.0,              # regularização do surrogate (Ridge)
    k_features=None,        # se informado, refita no Top-K (como K-LASSO do LIME)
    tau=0.05,               # pode ser 0.05 (5%) ou 5 (interpreta como 5%)
    sigma='from_explainer', # None => 0.75*sqrt(d̄); float; ou 'from_explainer'
    return_details=False
):
    """
    Complexity_LIME(x) baseada em τ como PERCENTUAL da massa L1 das importâncias:
        p_i = |w_i| / sum_j |w_j|
        k_tau = #{ i : p_i > τ }

    Onde:
      - τ pode ser dado como 0.05 (5%) ou 5 (5%). Ambos funcionam.
      - d̄ = nº de features efetivas (após remover drop_cols).
      - Simplicity_orig(x)  = 1 - k_tau / d̄   (0..1; MAIOR = mais simples)
      - Simplicity_orig(%)  = 100 * Simplicity_orig(x)  (0..100; MAIOR = melhor)

    Retorno principal (MAIOR = melhor): (complexity_k, simplicity_orig_pct[, details])
    """
    import numpy as np, pandas as pd
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import Ridge
    from sklearn.metrics import pairwise_distances
    from sklearn.model_selection import train_test_split

    EPS = 1e-12

    # 1) espaço efetivo e sigma
    feat_cols_eff = [c for c in X_ref.columns if c not in set(drop_cols)]
    if not feat_cols_eff:
        raise ValueError("Nenhuma feature efetiva após remover {'classe','y'}.")
    d_bar = len(feat_cols_eff)

    # --- PATCH ROBUSTO PARA sigma ---
    if sigma is None:
        sigma_val = 0.75 * np.sqrt(d_bar)
    elif isinstance(sigma, (int, float, np.floating)):
        sigma_val = float(sigma)
    elif isinstance(sigma, str):
        if sigma.lower() == 'from_explainer':
            sigma_val = float(getattr(explainer, 'kernel_width', 0.75 * np.sqrt(d_bar)))
        else:
            raise ValueError(f"sigma string inválido: {sigma!r}. Use 'from_explainer', None ou numérico.")
    else:
        sigma_val = 0.75 * np.sqrt(d_bar)

    # 2) formatos e scaler
    x_df = pd.DataFrame(np.asarray(x_row, dtype=float).reshape(1, -1), columns=feat_cols_eff)
    scaler = StandardScaler().fit(X_ref[feat_cols_eff].astype(float))

    # 3) vizinhança do LIME (ou fallback)
    try:
        exp, Z, distances = explainer.explain_instance(
            x_df.values.ravel(),
            classifier_fn,
            num_features=d_bar,
            num_samples=num_samples,
            labels=[pos_class],
            return_data=True,
            distance_metric='euclidean'
        )
        Z = np.asarray(Z, dtype=float)
        distances = None if distances is None else np.asarray(distances, dtype=float).ravel()
    except TypeError:
        std = X_ref[feat_cols_eff].astype(float).std(ddof=0).replace(0, 1e-6).values
        n_rest = max(1, num_samples - 1)
        Z = np.vstack([x_df.values,
                       x_df.values + np.random.normal(0.0, 0.05*(std + EPS), size=(n_rest, d_bar))])
        distances = None

    # 4) pesos do kernel (espaço padronizado) – só para treinar o surrogate
    Z_df  = pd.DataFrame(Z, columns=feat_cols_eff)
    Z_std = scaler.transform(Z_df)
    x_std = scaler.transform(x_df)
    if distances is None:
        distances = pairwise_distances(Z_std, x_std, metric='euclidean').ravel()
    weights = np.exp(-(distances**2) / (sigma_val**2))

    # 5) alvo local f(z)
    fz = np.asarray(classifier_fn(Z_df.values), dtype=float)[:, pos_class]

    # 6) ajusta o surrogate linear local
    reg = Ridge(alpha=alpha, fit_intercept=True)
    reg.fit(Z_std, fz, sample_weight=weights)
    w_full = reg.coef_.copy()  # coeficientes no espaço padronizado

    # 7) opcional: Top-K (aproxima K-LASSO)
    if (k_features is not None) and (0 < k_features < len(w_full)):
        top_idx = np.argsort(np.abs(w_full))[::-1][:int(k_features)]
        mask = np.zeros_like(w_full, dtype=bool); mask[top_idx] = True
        reg_k = Ridge(alpha=alpha, fit_intercept=True)
        reg_k.fit(Z_std[:, mask], fz, sample_weight=weights)
        w = np.zeros_like(w_full); w[mask] = reg_k.coef_
        topk_count = int(mask.sum())
    else:
        w = w_full
        mask = np.ones_like(w, dtype=bool)
        topk_count = None

    # 8) complexidade e simplicidade — τ como % da massa L1
    abs_w = np.abs(w)
    l1_sum = float(abs_w.sum() + EPS)
    p_l1 = abs_w / l1_sum

    tau_eff = float(tau)
    if tau_eff > 1.0:      # aceita 5 => 0.05
        tau_eff = tau_eff / 100.0

    above_tau = p_l1 > tau_eff
    complexity_k    = int(np.sum(above_tau))
    complexity_pct  = 100.0 * complexity_k / d_bar

    # MAIOR = melhor
    simplicity_orig_abs = 1.0 - (complexity_k / d_bar)       # 0..1
    simplicity_orig_pct = 100.0 * simplicity_orig_abs         # 0..100

    if not return_details:
        return complexity_k, float(round(simplicity_orig_pct, 3))

    # 9) DataFrame de detalhes por feature (ordenado por p_l1 desc)
    rank = p_l1.argsort()[::-1].argsort() + 1
    feat_df = pd.DataFrame({
        "feature":      feat_cols_eff,
        "w_full":       w_full,
        "w_used":       w,
        "abs_w":        abs_w,
        "p_l1":         p_l1,              # fração (0..1) da massa L1 por feature
        "above_tau_pct": above_tau,
        "rank_p_l1":    rank
    }).sort_values("p_l1", ascending=False).reset_index(drop=True)

    # Mantém ambos para auditoria; **simplicity_orig_pct** é o métrico principal (maior=melhor)
    simplicity_pct_small = float(np.clip(100.0 - simplicity_orig_pct, 0.0, 100.0))  # legado (menor=melhor)

    details = {
        "feature_table":        feat_df,
        "tau_input":            float(tau),
        "tau_eff_fraction":     float(tau_eff),
        "norm_kind":            "l1_pct",
        "d_bar":                int(d_bar),
        "sigma":                float(sigma_val),
        "complexity_k":         complexity_k,
        "complexity_percent":   float(round(complexity_pct, 3)),
        # principais:
        "simplicity_orig_abs":  float(round(simplicity_orig_abs, 8)),     # 0..1 (MAIOR=melhor)
        "simplicity_orig_pct":  float(round(simplicity_orig_pct, 3)),     # 0..100 (MAIOR=melhor)
        # legado p/ compatibilidade:
        "simplicity_small_pct": float(round(simplicity_pct_small, 3)),
        "mask_topk":            mask,
        "topk_count":           topk_count,
        "l1_sum":               l1_sum
    }
    return complexity_k, float(round(simplicity_orig_pct, 3)), details

def _normalize_corr(corr):
    """Normaliza apelidos de correlação."""
    if corr is None: return "spearman"
    c = str(corr).strip().lower()
    aliases = {
        # Spearman
        "spearman":"spearman", "spear":"spearman", "spearmn":"spearman", "s":"spearman",
        # Kendall (tau-b)
        "kendall":"kendall", "kendal":"kendall", "tau":"kendall", "k":"kendall",
        # Pearson
        "pearson":"pearson", "perarson":"pearson", "pearsonr":"pearson", "p":"pearson", "linear":"pearson",
    }
    return aliases.get(c, "spearman")

def calcula_consistency_lime(
    x_row,
    explainer,
    classifier_fn,          # ex.: lambda X: model.predict_proba(pd.DataFrame(X, columns=feat_cols_eff))
    X_ref,                  # DataFrame (use X ou X_train)
    drop_cols=('classe','y'),
    num_samples=5000,
    pos_class=1,
    alpha=1.0,              # regularização do surrogate (Ridge)
    k_features=None,        # None = usa todas; int => Top-K por |coef|
    sigma='from_explainer', # None => 0.75*sqrt(d̄), float, ou 'from_explainer'
    R=10,                   # nº de runs/repetições
    corr='spearman',        # 'spearman' | 'kendall' | 'pearson' + apelidos
    rank_by='abs',          # 'abs' (recomendado) ou 'raw' (usa sinal)
    random_state=42,
    return_details=False
):
    """
    Consistency(x) = média, sobre todos os pares de runs r1<r2, da correlação entre
    os vetores de explicações Φ^(r1)(x) e Φ^(r2)(x).
      - No LIME: Φ^(r)(x) = vetor de pesos do surrogate linear local no run r.
      - rank_by='abs' usa |w_i| (prática comum); 'raw' usa w_i com sinal.
      - k_features aplica refit no Top-K por |coef| (aproxima K-LASSO).

    Retorna: (consistency_percent, consistency_abs[, details])
      * consistency_abs ∈ [0,1] = (rho_mean + 1)/2  (MAIOR = melhor)
      * consistency_percent = 100 * consistency_abs
    """
    import numpy as np, pandas as pd
    from sklearn.preprocessing import StandardScaler
    from sklearn.metrics import pairwise_distances
    from sklearn.linear_model import Ridge

    # --- setup básico
    EPS = 1e-12
    corr = _normalize_corr(corr)

    feat_cols_eff = [c for c in X_ref.columns if c not in set(drop_cols)]
    if not feat_cols_eff:
        raise ValueError("Nenhuma feature efetiva após remover {'classe','y'}.")
    d_bar = len(feat_cols_eff)

    # sigma
    if sigma is None:
        sigma_val = 0.75 * np.sqrt(d_bar)
    elif isinstance(sigma, str) and sigma.lower() == 'from_explainer' and hasattr(explainer, 'kernel_width'):
        sigma_val = float(explainer.kernel_width)
    elif isinstance(sigma, (int, float)):
        sigma_val = float(sigma)
    else:
        sigma_val = 0.75 * np.sqrt(d_bar)

    # formatos
    x_df  = pd.DataFrame(np.asarray(x_row, dtype=float).reshape(1, -1), columns=feat_cols_eff)
    scaler = StandardScaler().fit(X_ref[feat_cols_eff].astype(float))

    # para reprodutibilidade por run
    base_rs = np.random.RandomState(random_state)

    # --- coleta dos vetores de explicação por run
    V = []        # matriz R x d com vetores (abs(w) ou w)
    coefs_all = []  # guarda os coefs "finais" por run (para details)
    for r in range(int(R)):
        # controlar semente do NumPy (o LIME usa NumPy/Random)
        np.random.seed(int(base_rs.randint(0, 2**31-1)))

        # tenta usar a vizinhança do LIME
        try:
            exp, Z, distances = explainer.explain_instance(
                x_df.values.ravel(),
                classifier_fn,
                num_features=d_bar,
                num_samples=num_samples,
                labels=[pos_class],
                return_data=True,
                distance_metric='euclidean'
            )
            Z = np.asarray(Z, dtype=float)
            distances = None if distances is None else np.asarray(distances, dtype=float).ravel()
        except TypeError:
            # fallback com ruído leve se a API não suportar return_data
            std = X_ref[feat_cols_eff].astype(float).std(ddof=0).replace(0, 1e-6).values
            n_rest = max(1, num_samples - 1)
            Z = np.vstack([x_df.values,
                           x_df.values + np.random.normal(0.0, 0.05*(std + EPS), size=(n_rest, d_bar))])
            distances = None
            exp = None

        # pesos do kernel (no espaço padronizado) – usados para treinar o surrogate
        Z_df  = pd.DataFrame(Z, columns=feat_cols_eff)
        Z_std = scaler.transform(Z_df)
        x_std = scaler.transform(x_df)
        if distances is None:
            distances = pairwise_distances(Z_std, x_std, metric='euclidean').ravel()
        weights = np.exp(-(distances**2) / (sigma_val**2))

        # alvo local f(z)
        fz = np.asarray(classifier_fn(Z_df.values), dtype=float)[:, pos_class]

        # surrogate local
        reg = Ridge(alpha=alpha, fit_intercept=True)
        reg.fit(Z_std, fz, sample_weight=weights)
        w = reg.coef_.copy()

        # Top-K opcional
        if (k_features is not None) and (0 < k_features < len(w)):
            top_idx = np.argsort(np.abs(w))[::-1][:int(k_features)]
            mask = np.zeros_like(w, dtype=bool); mask[top_idx] = True
            reg_k = Ridge(alpha=alpha, fit_intercept=True)
            reg_k.fit(Z_std[:, mask], fz, sample_weight=weights)
            w_final = np.zeros_like(w); w_final[mask] = reg_k.coef_
        else:
            w_final = w

        coefs_all.append(w_final)
        V.append(np.abs(w_final) if str(rank_by).lower() == "abs" else w_final)

    V = np.vstack(V)                  # (R, d)
    coefs_all = np.vstack(coefs_all)  # (R, d)

    # --- correlação entre todos os pares
    from scipy.stats import spearmanr, kendalltau, pearsonr
    rhos = []
    for i in range(R):
        for j in range(i+1, R):
            u, v = V[i], V[j]
            # se vetores constantes, define 1 quando idênticos, senão 0
            if np.allclose(u, u[0]) or np.allclose(v, v[0]):
                rho = 1.0 if np.allclose(u, v) else 0.0
            else:
                if corr == 'kendall':
                    rho, _ = kendalltau(u, v, variant='b')
                elif corr == 'pearson':
                    rho, _ = pearsonr(u, v)
                else:
                    rho, _ = spearmanr(u, v)
                if not np.isfinite(rho):
                    rho = 1.0 if np.allclose(u, v) else 0.0
            rhos.append(float(rho))

    # estatísticas dos pares
    rhos = np.asarray(rhos, dtype=float)
    rho_mean = float(np.mean(rhos)) if rhos.size else 1.0
    rho_std  = float(np.std(rhos))  if rhos.size else 0.0
    rho_min  = float(np.min(rhos))  if rhos.size else 1.0
    rho_max  = float(np.max(rhos))  if rhos.size else 1.0

    # score [0,1] e em %
    consistency_abs = float(np.clip((rho_mean + 1.0)/2.0, 0.0, 1.0))
    consistency_percent = float(f"{consistency_abs*100:.3f}")

    if not return_details:
        return consistency_percent, consistency_abs

    details = {
        "corr_type": corr,
        "rank_by": str(rank_by).lower(),
        "R": int(R),
        "pairs": int(R*(R-1)//2),
        "rho_mean": rho_mean,
        "rho_std":  rho_std,
        "rho_min":  rho_min,
        "rho_max":  rho_max,
        "rhos":     rhos,         # vetor com todas as correlações de pares
        "phis":     coefs_all,    # matriz (R,d) com os coeficientes finais por run
        "d_bar":    int(d_bar),
        "sigma":    float(sigma_val),
    }
    return consistency_percent, consistency_abs, details

def make_consistency_row(inst, cons_pct, cons_abs, det, decimals=8):
    """Monta um dicionário (1 linha) com os campos principais da consistency."""
    r = lambda v: round(float(v), decimals)  # atalho p/ arredondar números
    return {
        "instancia":       int(inst),
        "consistency_%":   r(cons_pct),
        "consistency_abs": r(cons_abs),
        "rho_mean":        r(det["rho_mean"]),
        "rho_std":         r(det["rho_std"]),
        "rho_min":         r(det["rho_min"]),
        "rho_max":         r(det["rho_max"]),
        "pairs":           int(det["pairs"]),
        "R":               int(det["R"]),
        "corr_type":       det["corr_type"],
        "rank_by":         det["rank_by"],
        "d_bar":           int(det["d_bar"]),
        "sigma":           r(det["sigma"]),
    }

def avalia_consistency_lime_lote(
    indices,              # ex.: lista de índices/linhas do X_test
    X_test,               # DataFrame com as features (mesmas de X_ref)
    explainer,
    classifier_fn,
    X_ref,
    *,
    drop_cols=('classe','y'),
    num_samples=5000,
    pos_class=1,
    alpha=1.0,
    k_features=None,
    sigma='from_explainer',
    R=10,
    corr='spearman',
    rank_by='abs',
    random_state=42
):
    """
    Roda Consistency para todas as instâncias em 'indices' e devolve UM DataFrame
    agregado com um resumo por instância + um dicionário de detalhes.
    """
    import numpy as np, pandas as pd

    feat_cols_eff = [c for c in X_ref.columns if c not in set(drop_cols)]
    rows = []
    details_by_inst = {}

    for inst in indices:
        x_row = X_test.loc[inst, feat_cols_eff].values
        cons_pct, cons_abs, det = calcula_consistency_lime(
            x_row=x_row,
            explainer=explainer,
            classifier_fn=classifier_fn,
            X_ref=X_ref,
            drop_cols=drop_cols,
            num_samples=num_samples,
            pos_class=pos_class,
            alpha=alpha,
            k_features=k_features,
            sigma=sigma,
            R=R,
            corr=corr,
            rank_by=rank_by,
            random_state=random_state,
            return_details=True
        )

        rows.append({
            "instancia":        inst,
            "consistency_%":    cons_pct,
            "consistency_abs":  round(cons_abs, 8),
            "rho_mean":         round(det["rho_mean"], 8),
            "rho_std":          round(det["rho_std"], 8),
            "rho_min":          round(det["rho_min"], 8),
            "rho_max":          round(det["rho_max"], 8),
            "pairs":            det["pairs"],
            "R":                det["R"],
            "corr_type":        det["corr_type"],
            "rank_by":          det["rank_by"],
            "d_bar":            det["d_bar"],
            "sigma":            det["sigma"],
        })
        details_by_inst[inst] = det

    df_consistency = pd.DataFrame(rows).sort_values("instancia").reset_index(drop=True)
    return df_consistency, details_by_inst

def _lime_weights_dense(exp, feature_names: Sequence[str], class_index: int) -> np.ndarray:
    """
    Converte o mapa do LIME para vetor denso alinhado a feature_names.
    """
    amap = exp.as_map()
    # fallback robusto: pega a primeira chave se a classe pedida não existir
    if class_index not in amap:
        class_index = sorted(list(amap.keys()))[0]
    w = np.zeros(len(feature_names), dtype=float)
    for idx, weight in amap[class_index]:
        if 0 <= idx < len(feature_names):
            w[idx] = weight
    return w

def _split_num_cat(X_ref: pd.DataFrame, feat_cols_eff: Sequence[str]) -> Tuple[List[int], List[int]]:
    num_idx, cat_idx = [], []
    for j, c in enumerate(feat_cols_eff):
        if pd.api.types.is_numeric_dtype(X_ref[c]):
            num_idx.append(j)
        else:
            cat_idx.append(j)
    return num_idx, cat_idx

def _bounds_numeric(X_ref: pd.DataFrame, feat_cols_eff: Sequence[str]) -> Dict[int, Tuple[float, float]]:
    b = {}
    for j, c in enumerate(feat_cols_eff):
        if pd.api.types.is_numeric_dtype(X_ref[c]):
            v = X_ref[c].to_numpy()
            b[j] = (float(np.nanmin(v)), float(np.nanmax(v)))
    return b

def _perturb_instance(x: np.ndarray,
                      X_ref: pd.DataFrame,
                      feat_cols_eff: Sequence[str],
                      num_idx: List[int],
                      cat_idx: List[int],
                      bounds: Dict[int, Tuple[float, float]],
                      epsilon: float = 0.05,
                      epsilon_mode: str = "relative_range",  # 'relative_range' | 'absolute'
                      strategy: str = "all",                  # 'all' (canônico) | 'one'
                      features_to_test: Optional[List[str]] = None,
                      cat_flip_prob: float = 0.1,
                      rng: Optional[np.random.Generator] = None) -> np.ndarray:
    """
    Gera x' dentro de uma vizinhança de x. Numéricas: ruído uniforme; Categóricas: flip com prob.
    """
    rng = rng or np.random.default_rng()
    xprime = x.copy()

    # define o conjunto de colunas-alvo
    if features_to_test is None:
        target_cols = list(feat_cols_eff)
    else:
        target_cols = list(features_to_test)

    if strategy == "one":
        # escolhe UMA feature desse conjunto
        target_cols = [rng.choice(target_cols)]

    target_idx = [feat_cols_eff.index(c) for c in target_cols]

    # numéricas
    for j in [j for j in target_idx if j in num_idx]:
        lo, hi = bounds.get(j, (x[j], x[j]))
        if epsilon_mode == "relative_range":
            radius = epsilon * max(1e-12, (hi - lo))
        elif epsilon_mode == "absolute":
            radius = float(epsilon)
        else:
            raise ValueError("epsilon_mode deve ser 'relative_range' ou 'absolute'.")
        val = x[j] + rng.uniform(-radius, radius)
        # clamp no intervalo observado
        if np.isfinite(lo) and np.isfinite(hi) and hi > lo:
            val = min(max(val, lo), hi)
        xprime[j] = val

    # categóricas
    for j in [j for j in target_idx if j in cat_idx]:
        if rng.random() < cat_flip_prob:
            col = feat_cols_eff[j]
            cats = pd.Series(X_ref[col].dropna().unique()).tolist()
            if len(cats) > 1:
                cur = x[j]
                # tenta trocar para outra categoria (se x veio codificado como string/obj na matriz, preserva tipo)
                choices = [c for c in cats if c != cur]
                if choices:
                    xprime[j] = rng.choice(choices)

    return xprime

def make_robustness_row(inst_id, robust_max, robust_mean, robust_std, det):
    return {
        "instancia": inst_id,
        "robustness_max": round(float(robust_max), 8),
        "robustness_mean": round(float(robust_mean), 8),
        "robustness_std": round(float(robust_std), 8),
        "robust_score": round(float(det["robust_score"]), 4),
        "score_mode": det.get("score_mode"),
        "norm": det["norm"],
        "epsilon": det["epsilon"],
        "epsilon_mode": det["epsilon_mode"],
        "strategy": det["strategy"],
        "n_trials": det["n_trials"],
        "n_lime_repeats": det["n_lime_repeats"],
        "tested_features": det["tested_features"],
        # campos ricos:
        "dists": det["dists"],
        "top_weights_delta": det["top_weights_delta"],
    }

def calcula_robustez_lime(
    x: np.ndarray,
    explainer_lime,                         # <- 2º argumento POSICIONAL, como você chamou
    *,
    classifier_fn,                          # <- agora é keyword obrigatório
    X_ref: pd.DataFrame,
    feat_cols_eff,
    class_index: int,
    n_trials: int = 300,
    epsilon: float = 0.05,
    epsilon_mode: str = "relative_range",   # 'relative_range' | 'absolute'
    norm: str = "l2",                       # 'l2' ou 'l1'
    strategy: str = "all",                  # 'all' (canônico) ou 'one'
    features_to_test=None,
    cat_flip_prob: float = 0.10,
    n_lime_repeats: int = 1,
    random_state: int = 42,
    topk_delta: int = 8,
    score_mode: str = "relative",
    score_alpha: float = None
):
    import numpy as np
    import pandas as pd

    def _lime_weights_dense(exp, feature_names, class_index):
        amap = exp.as_map()
        if class_index not in amap:
            class_index = sorted(list(amap.keys()))[0]
        w = np.zeros(len(feature_names), dtype=float)
        for idx, wgt in amap[class_index]:
            if 0 <= idx < len(feature_names):
                w[idx] = wgt
        return w

    def _split_num_cat(X_ref, feat_cols_eff):
        num_idx, cat_idx = [], []
        for j, c in enumerate(feat_cols_eff):
            if pd.api.types.is_numeric_dtype(X_ref[c]):
                num_idx.append(j)
            else:
                cat_idx.append(j)
        return num_idx, cat_idx

    def _bounds_numeric(X_ref, feat_cols_eff):
        b = {}
        for j, c in enumerate(feat_cols_eff):
            if pd.api.types.is_numeric_dtype(X_ref[c]):
                v = X_ref[c].to_numpy()
                b[j] = (float(np.nanmin(v)), float(np.nanmax(v)))
        return b

    def _perturb_instance(x, X_ref, feat_cols_eff, num_idx, cat_idx, bounds,
                          epsilon, epsilon_mode, strategy, features_to_test,
                          cat_flip_prob, rng):
        xprime = x.copy()
        if features_to_test is None:
            target_cols = list(feat_cols_eff)
        else:
            target_cols = list(features_to_test)
        if strategy == "one":
            target_cols = [rng.choice(target_cols)]
        target_idx = [feat_cols_eff.index(c) for c in target_cols]

        # numéricas
        for j in [j for j in target_idx if j in num_idx]:
            lo, hi = bounds.get(j, (x[j], x[j]))
            radius = (epsilon * max(1e-12, hi - lo)) if (epsilon_mode == "relative_range") else float(epsilon)
            val = float(x[j]) + rng.uniform(-radius, radius)
            if np.isfinite(lo) and np.isfinite(hi) and hi > lo:
                val = min(max(val, lo), hi)
            xprime[j] = val

        # categóricas
        for j in [j for j in target_idx if j in cat_idx]:
            if rng.random() < cat_flip_prob:
                col = feat_cols_eff[j]
                cats = pd.Series(X_ref[col].dropna().unique()).tolist()
                if len(cats) > 1:
                    cur = x[j]
                    choices = [c for c in cats if c != cur]
                    if choices:
                        xprime[j] = rng.choice(choices)
        return xprime

    rng = np.random.default_rng(random_state)

    # LIME para x (média de repetições para reduzir ruído do LIME)
    def _lime_dense(row_1d: np.ndarray) -> np.ndarray:
        ws = []
        for _ in range(max(1, n_lime_repeats)):
            exp = explainer_lime.explain_instance(
                row_1d, classifier_fn,
                num_features=len(feat_cols_eff),
                labels=[class_index]
            )
            ws.append(_lime_weights_dense(exp, feat_cols_eff, class_index))
        return np.mean(ws, axis=0)

    w0 = _lime_dense(x)

    num_idx, cat_idx = _split_num_cat(X_ref, feat_cols_eff)
    bounds = _bounds_numeric(X_ref, feat_cols_eff)

    dists = []
    best = {"dist": -np.inf, "xprime": None, "wprime": None}

    for _ in range(n_trials):
        xprime = _perturb_instance(
            x=x, X_ref=X_ref, feat_cols_eff=feat_cols_eff,
            num_idx=num_idx, cat_idx=cat_idx, bounds=bounds,
            epsilon=epsilon, epsilon_mode=epsilon_mode,
            strategy=strategy, features_to_test=features_to_test,
            cat_flip_prob=cat_flip_prob, rng=rng
        )
        wprime = _lime_dense(xprime)

        dist = float(np.linalg.norm(w0 - wprime)) if norm == "l2" else float(np.linalg.norm(w0 - wprime, ord=1))
        dists.append(dist)
        if dist > best["dist"]:
            best["dist"] = dist
            best["xprime"] = xprime.copy()
            best["wprime"] = wprime.copy()

    robust_max  = float(np.max(dists))
    robust_mean = float(np.mean(dists))
    robust_std  = float(np.std(dists, ddof=1)) if len(dists) > 1 else 0.0

    delta = np.abs(w0 - best["wprime"])
    topk_idx = np.argsort(-delta)[:min(topk_delta, len(delta))]
    top_weights_delta = [
        {"feature": feat_cols_eff[j], "w_x": float(w0[j]), "w_xprime": float(best["wprime"][j]), "abs_delta": float(delta[j])}
        for j in topk_idx
    ]

    detRob = {
        "norm": norm,
        "epsilon": float(epsilon),
        "epsilon_mode": epsilon_mode,
        "strategy": strategy,
        "n_trials": int(n_trials),
        "n_lime_repeats": int(n_lime_repeats),
        "tested_features": ",".join(features_to_test) if features_to_test else "ALL",
        "dists": dists,
        "top_weights_delta": top_weights_delta,
    }
    
    if norm == "l2":
        base_norm = float(np.linalg.norm(w0))
        wprime_norm = float(np.linalg.norm(best["wprime"]))
    else:
        base_norm = float(np.linalg.norm(w0, ord=1))
        wprime_norm = float(np.linalg.norm(best["wprime"], ord=1))

    eps = 1e-12

    if score_mode == "relative":
        ratio = robust_max / (base_norm + eps)
        robust_score = 100.0 * max(0.0, 1.0 - min(1.0, float(ratio)))

    elif score_mode == "cosine":
        dot = float(np.dot(w0, best["wprime"]))
        denom = (np.linalg.norm(w0) * np.linalg.norm(best["wprime"]) + eps)
        cos_sim = float(dot / denom)
        robust_score = 100.0 * (cos_sim + 1.0) / 2.0

    elif score_mode == "exp":
        if score_alpha is None:
            med = float(np.median(dists)) if len(dists) else 0.0
            score_alpha = np.log(2.0) / (med + eps) if med > 0 else 1.0
        robust_score = 100.0 * float(np.exp(-score_alpha * robust_max))

    else:
        raise ValueError("score_mode deve ser 'relative', 'cosine' ou 'exp'.")

    detRob.update({
        "robust_score": robust_score,
        "score_mode": score_mode,
        "score_alpha": score_alpha if score_mode == "exp" else None,
        "base_norm_w0": base_norm,
        "wprime_norm": wprime_norm,
    })



    return robust_max, robust_mean, robust_std, detRob

def calcula_selectivity_lime(
    x_vec: np.ndarray,                 # instância 1D na ordem de feat_cols_eff
    lime_explainer,
    *,                                  # força keyword-only a partir daqui
    classifier_fn,                      # sua função Xmat -> predict_proba
    X_ref: pd.DataFrame,                # p/ tipos e baselines
    feat_cols_eff: Sequence[str],
    class_index: int,
    K: Optional[int] = None,            # nº de passos (k=1..K). None => min(10, nº de pares do LIME)
    num_features: Optional[int] = None, # nº de features pedidas ao LIME (None => len(feat_cols_eff))
    ranking: str = "abs",               # "abs" (|peso|, padrão) ou "signed" (peso desc)
    masking: str = "mean_mode",         # como mascarar removendo S_k
    baseline_values: Optional[Dict[str, object]] = None,  # map {col: valor} opcional
    f_is_proba: bool = True,            # se f(x)∈[0,1], normaliza score por fx
):
    eps = 1e-12
    num_features = num_features or len(feat_cols_eff)

    # 1) f(x)
    fx = float(classifier_fn(x_vec.reshape(1, -1))[0, class_index])

    # 2) explicação LIME (classe alvo)
    exp = lime_explainer.explain_instance(
        x_vec,
        classifier_fn,
        num_features=num_features,
        labels=[class_index]
    )
    pairs = exp.as_map()[class_index]   # [(feature_idx, weight), ...]

    # 3) ordenação por importância
    if ranking == "abs":
        pairs = sorted(pairs, key=lambda t: abs(t[1]), reverse=True)
    elif ranking == "signed":
        pairs = sorted(pairs, key=lambda t: t[1], reverse=True)
    else:
        raise ValueError("ranking deve ser 'abs' ou 'signed'.")

    K_used = min(K if K is not None else 10, len(pairs))
    ranked = [{"idx": i, "feature": feat_cols_eff[i], "weight": float(w), "abs_weight": float(abs(w))}
              for i, w in pairs[:K_used]]

    # 4) construir baselines de masking
    if baseline_values is None:
        baseline_values = {}
    # completa baselines ausentes conforme estratégia
    if masking in ("mean_mode", "median_mode", "zero"):
        for c in feat_cols_eff:
            if c in baseline_values: 
                continue
            s = X_ref[c]
            if pd.api.types.is_numeric_dtype(s):
                if masking == "mean_mode":
                    baseline_values[c] = float(s.mean())
                elif masking == "median_mode":
                    baseline_values[c] = float(s.median())
                else:  # "zero"
                    baseline_values[c] = 0.0
            else:
                # categórica: usa moda
                baseline_values[c] = s.mode().iloc[0] if not s.mode().empty else s.dropna().iloc[0]
    elif masking == "sample":
        # será amostrado a cada passo (sem preencher agora)
        pass
    else:
        raise ValueError("masking deve ser 'mean_mode', 'median_mode', 'zero' ou 'sample'.")

    # 5) laço k = 1..K: mascara S_k e mede a queda
    deltas = []
    steps  = []
    for k in range(1, K_used + 1):
        S_k = [r["idx"] for r in ranked[:k]]
        x_mask = x_vec.copy()

        # aplicar máscara/remover S_k
        for j in S_k:
            col = feat_cols_eff[j]
            if masking == "sample":
                col_vals = X_ref[col].dropna().values
                # amostra 1 valor do dataset (respeita distribuição)
                val = np.random.choice(col_vals) if len(col_vals) else x_mask[j]
            else:
                val = baseline_values[col]
            x_mask[j] = val

        fx_mask = float(classifier_fn(x_mask.reshape(1, -1))[0, class_index])
        delta = fx - fx_mask
        deltas.append(delta)

        steps.append({
            "k": k,
            "features_masked": [feat_cols_eff[j] for j in S_k],
            "f_x_minus_Sk": round(fx_mask, 8),
            "delta": round(delta, 8)
        })

    # 6) Selectivity absoluta (equação do anexo): média das quedas
    selectivity_abs = float(np.mean(deltas))

    # 7) Score 0–100 (normalizado)
    if f_is_proba:
        denom = max(fx, eps)  # média da queda relativa a f(x)
    else:
        denom = 1.0
    selectivity_score = 100.0 * max(0.0, 1.0 - min(1.0, 1.0 - (selectivity_abs / (denom + eps))))
    # equivalente a: 100 * clip(selectivity_abs/denom, 0, 1)

    # 8) detalhamento
    detSel = {
        "fx": fx,
        "K_used": K_used,
        "ranking": ranking,
        "masking": masking,
        "selectivity_steps": steps,       # por k
        "features_ranked": ranked,        # ranking base
        "baseline_values_used": baseline_values if masking != "sample" else "sample",
    }
    return selectivity_score, selectivity_abs, detSel

def make_selectivity_row(inst_id, sel_score, sel_abs, det: dict) -> dict:
    return {
        "instancia": inst_id,
        "selectivity_score": round(float(sel_score), 4),   # 0–100
        "selectivity_abs": round(float(sel_abs), 8),       # média das quedas absolutas
        "fx": round(float(det["fx"]), 8),
        "K_used": det["K_used"],
        "ranking": det["ranking"],
        "masking": det["masking"],
        "features_ranked": det["features_ranked"],
        "selectivity_steps": det["selectivity_steps"],
        "baseline_values_used": det["baseline_values_used"],
    }

def _lime_weights_dense(exp, feature_names: Sequence[str], class_index: int) -> np.ndarray:
    amap = exp.as_map()
    if class_index not in amap:
        class_index = sorted(list(amap.keys()))[0]
    w = np.zeros(len(feature_names), dtype=float)
    for idx, weight in amap[class_index]:
        if 0 <= idx < len(feature_names):
            w[idx] = weight
    return w

def calcula_soundness_lime(
    x_vec: np.ndarray,                 # instância 1D (ordem = feat_cols_eff)
    lime_explainer,
    *,                                  # keyword-only abaixo
    classifier_fn,                      # Xmat -> predict_proba
    X_ref: pd.DataFrame,                # para baselines (auto_local)
    feat_cols_eff: Sequence[str],
    class_index: int,
    num_features: Optional[int] = None, # quantas features pedir ao LIME (None => len(feat_cols_eff))
    U_cols: Optional[List[str]] = None, # se fornecido, usa como conjunto U
    U_mode: str = "given",              # "given" | "auto_local"
    tau: float = 0.01,                  # limiar p/ auto_local (queda máxima em f(x) p/ considerar irrelevante)
    masking: str = "mean_mode",         # como mascarar no auto_local
    baseline_values: Optional[Dict[str, object]] = None,
):
    eps = 1e-12
    num_features = num_features or len(feat_cols_eff)

    # f(x)
    fx = float(classifier_fn(x_vec.reshape(1, -1))[0, class_index])

    # explicação LIME
    exp = lime_explainer.explain_instance(
        x_vec, classifier_fn,
        num_features=num_features,
        labels=[class_index]
    )
    w_dense = _lime_weights_dense(exp, feat_cols_eff, class_index)
    abs_w = np.abs(w_dense)
    denom = float(np.sum(abs_w)) + eps  # soma do "mass" atribuicional

    # ----------------- definir U -----------------
    U_used = []   # nomes
    U_idx  = []   # índices
    auto_local_details = []

    if U_mode == "given":
        if U_cols is None:
            U_cols = []
        for c in U_cols:
            if c in feat_cols_eff:
                U_used.append(c)
                U_idx.append(feat_cols_eff.index(c))

    elif U_mode == "auto_local":
        # construir baselines
        if baseline_values is None:
            baseline_values = {}
        # completa baselines
        for c in feat_cols_eff:
            if c in baseline_values: 
                continue
            s = X_ref[c]
            if pd.api.types.is_numeric_dtype(s):
                if masking == "mean_mode":
                    baseline_values[c] = float(s.mean())
                elif masking == "median_mode":
                    baseline_values[c] = float(s.median())
                elif masking == "zero":
                    baseline_values[c] = 0.0
                else:
                    # sample
                    vals = s.dropna().values
                    baseline_values[c] = float(vals[np.random.randint(len(vals))]) if len(vals) else 0.0
            else:
                # categóricas -> moda (ou amostra)
                if masking == "sample":
                    vals = s.dropna().values
                    baseline_values[c] = vals[np.random.randint(len(vals))] if len(vals) else (s.mode().iloc[0] if not s.mode().empty else None)
                else:
                    baseline_values[c] = s.mode().iloc[0] if not s.mode().empty else s.dropna().iloc[0]

        # testa feature a feature
        for j, col in enumerate(feat_cols_eff):
            x_mask = x_vec.copy()
            x_mask[j] = baseline_values[col]
            fx_mask = float(classifier_fn(x_mask.reshape(1, -1))[0, class_index])
            delta = abs(fx - fx_mask)
            auto_local_details.append({"feature": col, "delta_fx": float(delta)})
            if delta <= tau:
                U_used.append(col)
                U_idx.append(j)
    else:
        raise ValueError("U_mode deve ser 'given' ou 'auto_local'.")

    # ----------------- soundness -----------------
    numer = float(np.sum(abs_w[U_idx])) if len(U_idx) else 0.0
    soundness = 1.0 - (numer / denom)
    soundness_score = 100.0 * max(0.0, min(1.0, soundness))  # 0–100

    # detalhamento
    weights_by_feature = [
        {"feature": feat_cols_eff[i], "weight": float(w_dense[i]), "abs_weight": float(abs_w[i])}
        for i in range(len(feat_cols_eff))
    ]
    weights_by_feature.sort(key=lambda d: d["abs_weight"], reverse=True)

    detSnd = {
        "fx": fx,
        "denom_sum_abs_w": denom,
        "numer_sum_abs_w_U": numer,
        "U_mode": U_mode,
        "U_cols_used": U_used,
        "auto_local_details": auto_local_details if U_mode == "auto_local" else None,
        "weights_by_feature": weights_by_feature,
    }
    return soundness_score, soundness, detSnd

def make_soundness_row(inst_id, snd_score, snd_raw, det: dict) -> dict:
    return {
        "instancia": inst_id,
        "soundness_score": round(float(snd_score), 4),     # 0–100 (maior = melhor)
        "soundness_raw": round(float(snd_raw), 8),         # em [0,1]
        "fx": round(float(det["fx"]), 8),
        "denom_sum_abs_w": round(float(det["denom_sum_abs_w"]), 8),
        "numer_sum_abs_w_U": round(float(det["numer_sum_abs_w_U"]), 8),
        "U_mode": det["U_mode"],
        "U_cols_used": det["U_cols_used"],
        "auto_local_details": det["auto_local_details"],
        "weights_by_feature": det["weights_by_feature"],
    }

def _lime_weights_dense(exp, feature_names: Sequence[str], class_index: int) -> np.ndarray:
    amap = exp.as_map()
    if class_index not in amap:
        class_index = sorted(list(amap.keys()))[0]
    w = np.zeros(len(feature_names), dtype=float)
    for idx, weight in amap[class_index]:
        if 0 <= idx < len(feature_names):
            w[idx] = weight
    return w

def _sign_from_weight(w: float, abs_max: float, eps_rel: float = 0.01) -> int:
    """
    Converte peso -> {-1,0,+1}, tratando valores muito pequenos como neutros.
    eps_rel=0.01 => abaixo de 1% do |peso|max local vira 0.
    """
    thr = eps_rel * (abs_max if abs_max > 0 else 1.0)
    if abs(w) <= thr:
        return 0
    return 1 if w > 0 else -1

def calcula_directional_soundness_lime(
    x_vec: np.ndarray,                  # instância 1D (ordem = feat_cols_eff)
    lime_explainer,
    *,                                   # keyword-only abaixo
    classifier_fn,                       # Xmat -> predict_proba
    X_ref: pd.DataFrame,                 # usado se for derivar R/GT automaticamente
    feat_cols_eff: Sequence[str],
    class_index: int,
    num_features: Optional[int] = None,  # quantas features pedir ao LIME (None => len(feat_cols_eff))

    # --- Como obter R (features relevantes) e a direção esperada (GT) ---
    direcao_esperada: Optional[Dict[str, int]] = None,  # {"feat": -1|0|+1}
    R_mode: str = "given",            # "given" | "auto_local" | "topk_lime"
    topk: int = 10,                   # se R_mode="topk_lime": usa top-|w| do LIME
    tau: float = 0.01,                # se "auto_local": variação mínima em f(x) p/ considerar relevante
    tau_mode: str = "abs",            # "abs" (valor absoluto) | "rel_fx" (fração de f(x))
    masking: str = "mean_mode",       # como mascarar ao testar relevância/direção
    baseline_values: Optional[Dict[str, object]] = None,

    # --- Como interpretar “sinal” do LIME ---
    sign_eps_rel: float = 0.01,       # até 1% do |w|max conta como 0 (neutro)
    count_zero_matches: bool = True,  # se GT=0 e sinal LIME=0, conta como acerto
):
    """
    Retorna:
      - score_percent (0–100)
      - frac_raw (0–1)
      - detDir (detalhamento completo p/ debug)
    """
    eps = 1e-12
    num_features = num_features or len(feat_cols_eff)

    # 1) f(x)
    fx = float(classifier_fn(x_vec.reshape(1, -1))[0, class_index])

    # 2) LIME em x
    exp = lime_explainer.explain_instance(
        x_vec, classifier_fn,
        num_features=num_features,
        labels=[class_index]
    )
    w_dense = _lime_weights_dense(exp, feat_cols_eff, class_index)
    abs_max_w = float(np.max(np.abs(w_dense))) if w_dense.size else 0.0

    # 3) Definir R (features relevantes) e GT_i (direção esperada)
    R_feats: List[str] = []
    GT_dir: Dict[str, int] = {}

    def _mask_value_for(col: str):
        if baseline_values and col in baseline_values:
            return baseline_values[col]
        s = X_ref[col]
        if pd.api.types.is_numeric_dtype(s):
            if masking == "mean_mode":
                return float(s.mean())
            elif masking == "median_mode":
                return float(s.median())
            elif masking == "zero":
                return 0.0
            elif masking == "sample":
                vals = s.dropna().values
                return float(np.random.choice(vals)) if len(vals) else 0.0
        # categóricas
        if masking == "sample":
            vals = s.dropna().values
            if len(vals):
                return np.random.choice(vals)
        mode = s.mode()
        return mode.iloc[0] if not mode.empty else (s.dropna().iloc[0] if s.dropna().size else None)

    if R_mode == "given":
        if not direcao_esperada:
            raise ValueError("R_mode='given' requer direcao_esperada={'feat': -1|0|+1}.")
        # usa chaves de direcao_esperada como R
        for feat, gt in direcao_esperada.items():
            if feat in feat_cols_eff:
                R_feats.append(feat)
                GT_dir[feat] = int(np.sign(gt)) if gt in (-1, 0, 1) else int(np.sign(gt))
    elif R_mode == "topk_lime":
        # pega top-|w| do LIME e deriva sinal esperado do próprio efeito de masking
        ranked_idx = np.argsort(-np.abs(w_dense))[:min(topk, len(feat_cols_eff))]
        for j in ranked_idx:
            feat = feat_cols_eff[j]
            R_feats.append(feat)
    elif R_mode == "auto_local":
        # varre todas as colunas, marca relevantes por delta>tau e usa o sinal do delta p/ GT
        for j, col in enumerate(feat_cols_eff):
            x_mask = x_vec.copy()
            x_mask[j] = _mask_value_for(col)
            fx_mask = float(classifier_fn(x_mask.reshape(1, -1))[0, class_index])
            delta = fx - fx_mask  # positivo => presença/valor atual aumenta f(x)
            thr = (tau * fx if tau_mode == "rel_fx" else tau)
            if abs(delta) > thr:
                R_feats.append(col)
                GT_dir[col] = 1 if delta > 0 else -1
    else:
        raise ValueError("R_mode deve ser 'given', 'topk_lime' ou 'auto_local'.")

    # Se R_mode for topk_lime e não veio direcao_esperada, derive GT pela direção do efeito do modelo
    if R_mode == "topk_lime" and not direcao_esperada:
        for feat in R_feats:
            j = feat_cols_eff.index(feat)
            x_mask = x_vec.copy()
            x_mask[j] = _mask_value_for(feat)
            fx_mask = float(classifier_fn(x_mask.reshape(1, -1))[0, class_index])
            delta = fx - fx_mask
            GT_dir[feat] = 1 if delta > 0 else (-1 if delta < 0 else 0)

    # Se R_mode='given' e passou direcao_esperada, já temos GT_dir,
    # mas garanta que ele exista só para as feats de R:
    if R_mode == "given":
        GT_dir = {f: GT_dir.get(f, int(np.sign(direcao_esperada[f])))
                  for f in R_feats}

    # 4) Avaliar coincidência de sinais
    per_feat = []
    matches = 0
    considered = 0

    for feat in R_feats:
        j = feat_cols_eff.index(feat)
        s_hat = _sign_from_weight(w_dense[j], abs_max_w, eps_rel=sign_eps_rel)
        s_exp = GT_dir.get(feat, 0)
        # se for neutro:
        if s_exp == 0 or s_hat == 0:
            ok = (s_exp == 0 and s_hat == 0 and count_zero_matches)
        else:
            ok = (s_exp == s_hat)

        considered += 1
        if ok:
            matches += 1

        per_feat.append({
            "feature": feat,
            "w": float(w_dense[j]),
            "sign_lime": int(s_hat),
            "sign_expected": int(s_exp),
            "match": bool(ok),
        })

    frac = (matches / considered) if considered > 0 else np.nan
    score_percent = float(frac * 100.0) if considered > 0 else np.nan

    detDir = {
        "fx": fx,
        "R_mode": R_mode,
        "R_size": considered,
        "R_feats": R_feats,
        "topk": topk,
        "tau": tau,
        "tau_mode": tau_mode,
        "masking": masking,
        "sign_eps_rel": sign_eps_rel,
        "count_zero_matches": count_zero_matches,
        "per_feature": per_feat
    }
    return score_percent, frac, detDir

def make_directional_soundness_row(inst_id, score_pct, frac_raw, det: dict) -> dict:
    return {
        "instancia": inst_id,
        "directional_soundness_score": None if pd.isna(score_pct) else round(float(score_pct), 4),  # 0–100
        "directional_soundness_frac": None if pd.isna(frac_raw) else round(float(frac_raw), 6),      # 0–1
        "R_mode": det["R_mode"],
        "R_size": det["R_size"],
        "topk": det["topk"],
        "tau": det["tau"],
        "tau_mode": det["tau_mode"],
        "masking": det["masking"],
        "sign_eps_rel": det["sign_eps_rel"],
        "count_zero_matches": det["count_zero_matches"],
        "fx": round(float(det["fx"]), 8),
        "per_feature": det["per_feature"],
        "R_feats": det["R_feats"],
    }

def _lime_weights_dense(exp, feature_names: Sequence[str], class_index: int) -> np.ndarray:
    amap = exp.as_map()
    if class_index not in amap:
        class_index = sorted(list(amap.keys()))[0]
    w = np.zeros(len(feature_names), dtype=float)
    for idx, weight in amap[class_index]:
        if 0 <= idx < len(feature_names):
            w[idx] = weight
    return w

def _split_num_cat(X_ref: pd.DataFrame, feat_cols_eff: Sequence[str]) -> Tuple[List[int], List[int]]:
    num_idx, cat_idx = [], []
    for j, c in enumerate(feat_cols_eff):
        if pd.api.types.is_numeric_dtype(X_ref[c]):
            num_idx.append(j)
        else:
            cat_idx.append(j)
    return num_idx, cat_idx

def _bounds_numeric(X_ref: pd.DataFrame, feat_cols_eff: Sequence[str]) -> Dict[int, Tuple[float, float]]:
    b = {}
    for j, c in enumerate(feat_cols_eff):
        if pd.api.types.is_numeric_dtype(X_ref[c]):
            v = X_ref[c].to_numpy()
            b[j] = (float(np.nanmin(v)), float(np.nanmax(v)))
    return b

def _fit_quantile_edges(X_ref: pd.DataFrame, feat_cols_eff: Sequence[str], num_bins: int = 4) -> Dict[int, Optional[np.ndarray]]:
    edges_map = {}
    for j, c in enumerate(feat_cols_eff):
        s = X_ref[c]
        if not pd.api.types.is_numeric_dtype(s):
            edges_map[j] = None
            continue
        # limites por quantis, garantindo unicidade/ordem
        qs = np.linspace(0, 1, num_bins+1)
        edges = np.unique(np.quantile(s.dropna().values, qs))
        edges_map[j] = edges if len(edges) >= 2 else None
    return edges_map

def _bucket_index(val: float, edges: np.ndarray) -> int:
    idx = np.searchsorted(edges, val, side='right') - 1
    return int(np.clip(idx, 0, len(edges)-2))

def _dist_to_nearest_edge(val: float, edges: Optional[np.ndarray]) -> float:
    if edges is None or len(edges) < 2:
        return np.inf
    i = _bucket_index(val, edges)
    left, right = edges[i], edges[i+1]
    return float(min(abs(val-left), abs(right-val)))

def calcula_stability_lime(
    x_vec: np.ndarray,                # instância 1D (na ordem de feat_cols_eff)
    lime_explainer,
    *,                                 # keyword-only abaixo
    classifier_fn,                     # Xmat -> predict_proba
    X_ref: pd.DataFrame,               # para tipos/bounds/categorias
    feat_cols_eff: Sequence[str],
    class_index: int,
    n_trials: int = 200,               # nº de vizinhos x+δ
    norm: str = "l2",                  # 'l2' ou 'l1'

    # --- ruído para numéricas ---
    sigma_mode: str = "adaptive",      # 'adaptive' | 'relative_range' | 'absolute'
    sigma: float = 0.05,               # usado se 'relative_range' (fração do range) ou 'absolute'
    alpha: float = 0.02,               # usado no 'adaptive' (fração do range)
    beta: float = 0.10,                # usado no 'adaptive' (fração do std)
    num_bins: int = 4,                 # bins p/ estimar bordas (adaptive/avoid crossing)
    avoid_bucket_crossing: bool = True,

    # --- ruído para categóricas ---
    p_flip: float = 0.05,              # prob. de flip de categoria
    baseline_cats: Optional[Dict[str, object]] = None,

    # --- controle do LIME (ruído interno) ---
    n_lime_repeats: int = 1,           # média de n explicações por chamada
    random_state: int = 42
):
    """
    Retorna:
      stability_mean, stability_std, stability_score(0-100), detStb (dict com debug)
    """
    rng = np.random.default_rng(random_state)
    num_idx, cat_idx = _split_num_cat(X_ref, feat_cols_eff)
    bounds = _bounds_numeric(X_ref, feat_cols_eff)
    edges_map = _fit_quantile_edges(X_ref, feat_cols_eff, num_bins=num_bins)

    # helper: pesos do LIME (média de n repetições)
    def _lime_dense(row_1d: np.ndarray) -> np.ndarray:
        ws = []
        for _ in range(max(1, n_lime_repeats)):
            exp = lime_explainer.explain_instance(
                row_1d, classifier_fn,
                num_features=len(feat_cols_eff),
                labels=[class_index]
            )
            ws.append(_lime_weights_dense(exp, feat_cols_eff, class_index))
        return np.mean(ws, axis=0)

    w0 = _lime_dense(x_vec)

    # sigma por feature (apenas numéricas)
    sigma_map: Dict[int, float] = {}
    for j in num_idx:
        lo, hi = bounds.get(j, (x_vec[j], x_vec[j]))
        rng_span = max(0.0, hi - lo)
        std_j = float(X_ref.iloc[:, j].std(ddof=0)) if rng_span > 0 else 0.0
        if sigma_mode == "relative_range":
            sigma_map[j] = sigma * (rng_span if rng_span > 0 else 1.0)
        elif sigma_mode == "absolute":
            sigma_map[j] = float(sigma)
        elif sigma_mode == "adaptive":
            dist_edge = _dist_to_nearest_edge(float(x_vec[j]), edges_map.get(j))
            candidates = []
            if rng_span > 0: candidates.append(alpha * rng_span)
            if std_j > 0:   candidates.append(beta * std_j)
            if np.isfinite(dist_edge): candidates.append(0.45 * dist_edge)  # evita cruzar
            sigma_map[j] = max(1e-9, min(candidates)) if candidates else 1e-3
        else:
            raise ValueError("sigma_mode deve ser 'adaptive', 'relative_range' ou 'absolute'.")

    # categóricas: baseline para flip
    if baseline_cats is None:
        baseline_cats = {}
    for j in cat_idx:
        col = feat_cols_eff[j]
        if col not in baseline_cats:
            s = X_ref[col].dropna()
            baseline_cats[col] = (s.mode().iloc[0] if not s.mode().empty else (s.iloc[0] if len(s) else None))

    # gera vizinhos e mede distâncias
    dists = []
    # para debug: média de |delta de peso| por feature
    accum_abs_delta = np.zeros_like(w0)
    bucket_cross_events = 0
    bucket_checks = 0

    for _ in range(n_trials):
        xprime = x_vec.copy()

        # numéricas com jitter gaussiano
        for j in num_idx:
            sig = sigma_map[j]
            noise = float(rng.normal(0.0, sig))
            cand = float(x_vec[j]) + noise

            # conta se cruzaria bucket (apenas para métrica de debug)
            edges = edges_map.get(j)
            if edges is not None:
                bucket_checks += 1
                if _bucket_index(cand, edges) != _bucket_index(float(x_vec[j]), edges):
                    bucket_cross_events += 1

            if avoid_bucket_crossing and edges is not None:
                # clamp para ficar no mesmo intervalo
                i = _bucket_index(float(x_vec[j]), edges)
                left, right = edges[i], edges[i+1]
                eps = 1e-9
                cand = min(max(cand, left+eps), right-eps)

            # clampa em [lo, hi] observados
            lo, hi = bounds.get(j, (cand, cand))
            if np.isfinite(lo) and np.isfinite(hi) and hi > lo:
                cand = min(max(cand, lo), hi)

            xprime[j] = cand

        # categóricas: flip eventual
        for j in cat_idx:
            if rng.random() < p_flip:
                col = feat_cols_eff[j]
                cats = pd.Series(X_ref[col].dropna().unique()).tolist()
                if len(cats) > 1:
                    others = [c for c in cats if c != x_vec[j]]
                    if others:
                        xprime[j] = rng.choice(others)
            else:
                # mantém
                pass

        wprime = _lime_dense(xprime)

        if norm == "l2":
            dist = float(np.linalg.norm(w0 - wprime))
        elif norm == "l1":
            dist = float(np.linalg.norm(w0 - wprime, ord=1))
        else:
            raise ValueError("norm deve ser 'l2' ou 'l1'.")

        dists.append(dist)
        accum_abs_delta += np.abs(w0 - wprime)

    stability_mean = float(np.mean(dists)) if dists else 0.0
    stability_std  = float(np.std(dists, ddof=1)) if len(dists) > 1 else 0.0

    # score 0–100 (maior = melhor), relativo ao tamanho da explicação
    base_norm = float(np.linalg.norm(w0)) if norm == "l2" else float(np.linalg.norm(w0, ord=1))
    eps = 1e-12
    ratio = stability_mean / (base_norm + eps)
    stability_score = 100.0 * max(0.0, 1.0 - min(1.0, ratio))

    # top features que mais variaram em média
    mean_abs_delta = accum_abs_delta / max(1, n_trials)
    top_idx = np.argsort(-mean_abs_delta)[:min(8, len(mean_abs_delta))]
    top_changes = [
        {"feature": feat_cols_eff[j], "mean_abs_delta_w": float(mean_abs_delta[j]), "w0": float(w0[j])}
        for j in top_idx
    ]

    # debug
    detStb = {
        "norm": norm,
        "n_trials": n_trials,
        "sigma_mode": sigma_mode,
        "sigma": sigma,
        "alpha": alpha,
        "beta": beta,
        "num_bins": num_bins,
        "avoid_bucket_crossing": avoid_bucket_crossing,
        "p_flip": p_flip,
        "n_lime_repeats": n_lime_repeats,
        "base_norm_w0": base_norm,
        "dists": dists,
        "bucket_cross_rate": (bucket_cross_events / bucket_checks) if bucket_checks > 0 else 0.0,
        "sigma_map_summary": {
            "mean": float(np.mean([sigma_map[j] for j in sigma_map])) if sigma_map else 0.0,
            "min": float(np.min([sigma_map[j] for j in sigma_map])) if sigma_map else 0.0,
            "max": float(np.max([sigma_map[j] for j in sigma_map])) if sigma_map else 0.0,
        },
        "top_changes": top_changes
    }

    return stability_mean, stability_std, stability_score, detStb

def make_stability_row(inst_id, stab_mean, stab_std, stab_score, det: dict) -> dict:
    return {
        "instancia": inst_id,
        "stability_mean": round(float(stab_mean), 8),
        "stability_std": round(float(stab_std), 8),
        "stability_score(%)": round(float(stab_score), 4),
        "norm": det["norm"],
        "n_trials": det["n_trials"],
        "sigma_mode": det["sigma_mode"],
        "sigma": det["sigma"],
        "alpha": det["alpha"],
        "beta": det["beta"],
        "num_bins": det["num_bins"],
        "avoid_bucket_crossing": det["avoid_bucket_crossing"],
        "p_flip": det["p_flip"],
        "n_lime_repeats": det["n_lime_repeats"],
        "base_norm_w0": round(float(det["base_norm_w0"]), 8),
        "bucket_cross_rate": round(float(det["bucket_cross_rate"]), 6),
        "sigma_map_summary": det["sigma_map_summary"],
        "top_changes": det["top_changes"],
        "dists": det["dists"],
    }



# ------------------ utils ------------------
def _lime_weights_dense(exp, feature_names: Sequence[str], class_index: int) -> np.ndarray:
    amap = exp.as_map()
    if class_index not in amap:
        class_index = sorted(amap.keys())[0]
    w = np.zeros(len(feature_names), dtype=float)
    for idx, weight in amap[class_index]:
        if 0 <= idx < len(feature_names):
            w[idx] = weight
    return w

def _split_num_cat(X_ref: pd.DataFrame, feat_cols_eff: Sequence[str]) -> Tuple[List[int], List[int]]:
    num_idx, cat_idx = [], []
    for j, c in enumerate(feat_cols_eff):
        if pd.api.types.is_numeric_dtype(X_ref[c]):
            num_idx.append(j)
        else:
            cat_idx.append(j)
    return num_idx, cat_idx

def _build_baselines(X_ref: pd.DataFrame, feat_cols_eff: Sequence[str], masking: str, baseline_values: Optional[Dict[str, object]]):
    """Decide baseline numérico/categórico por coluna."""
    baselines = {}
    masking = (masking or "median_mode").lower()
    for c in feat_cols_eff:
        if baseline_values and (c in baseline_values):
            baselines[c] = baseline_values[c]
            continue
        s = X_ref[c].dropna()
        if pd.api.types.is_numeric_dtype(X_ref[c]):
            if masking == "zero":
                baselines[c] = 0.0
            elif masking in ("mean", "mean_mode"):
                baselines[c] = float(s.mean()) if len(s) else 0.0
            else:  # "median_mode" (padrão)
                baselines[c] = float(s.median()) if len(s) else 0.0
        else:
            if masking == "sample":
                baselines[c] = (lambda: s.sample(1).iloc[0] if len(s) else None)
            else:
                baselines[c] = (s.mode().iloc[0] if not s.mode().empty else (s.iloc[0] if len(s) else None))
    return baselines

def _mask_x_keep_S(x_vec: np.ndarray, feat_cols_eff: Sequence[str], S_idx: set,
                   baselines: Dict[str, object], masking: str, rng: np.random.Generator) -> np.ndarray:
    """Constroi x|_S: mantém features de S; fora de S aplica baseline."""
    xS = x_vec.copy()
    for j, c in enumerate(feat_cols_eff):
        if j in S_idx:
            continue
        b = baselines[c]
        if callable(b):   # "sample" para categóricas
            xS[j] = b()
        else:
            xS[j] = b
    return xS

# ------------------ SUFFICIENCY ------------------
def calcula_sufficiency_lime(
    x_vec: np.ndarray,                # 1D na ordem de feat_cols_eff
    lime_explainer,
    *,
    classifier_fn,                    # lambda X: model.predict_proba(pd.DataFrame(X, columns=feat_cols_eff))
    X_ref: pd.DataFrame,
    feat_cols_eff: Sequence[str],
    class_index: int,
    percent_list: Sequence[float] = (0.01, 0.05, 0.10, 0.20),
    ranking: str = "abs",             # "abs" por |w|, ou "signed" por w
    num_features: Optional[int] = None, # quantas features o LIME retorna (None => len(feat_cols_eff))
    masking: str = "median_mode",     # "median_mode"|"mean_mode"|"zero"|"sample"
    baseline_values: Optional[Dict[str, object]] = None,
    f_is_proba: bool = True,          # se True, normaliza score usando fx∈[0,1]
    random_state: int = 42
):
    """
    Retorna:
      scores_dict   -> { '1%':score%, '5%':..., '10%':..., '20%':..., 'AOPC_suff(%)':..., 'AOPC_suff_raw':... }
      details       -> dict com fx, fxs_por_K, drops_por_K, Ks_efetivos, topK_features etc.
    """
    rng = np.random.default_rng(random_state)
    num_features = num_features or len(feat_cols_eff)
    d = len(feat_cols_eff)

    # 1) f(x)
    fx = float(classifier_fn(x_vec.reshape(1, -1))[0, class_index])

    # 2) pesos do LIME e ranking
    exp = lime_explainer.explain_instance(
        x_vec, classifier_fn, num_features=num_features, labels=[class_index]
    )
    w = _lime_weights_dense(exp, feat_cols_eff, class_index)
    order = np.argsort(-np.abs(w)) if ranking == "abs" else np.argsort(-w)
    # Mapeia para nomes das features
    ordered_feats = [(feat_cols_eff[i], i, float(w[i])) for i in order]

    # 3) baselines e máscara
    baselines = _build_baselines(X_ref, feat_cols_eff, masking, baseline_values)

    # 4) para cada %K, monta S e calcula f(x|S)
    k_list = []
    fx_Sk = []
    drops = []
    S_lists = []
    for p in percent_list:
        K = int(np.ceil(max(1, p * d)))
        top_idx = set([idx for _, idx, _ in ordered_feats[:K]])
        xS = _mask_x_keep_S(x_vec, feat_cols_eff, top_idx, baselines, masking, rng)
        fxS = float(classifier_fn(xS.reshape(1, -1))[0, class_index])
        drop = fx - fxS
        k_list.append(K)
        fx_Sk.append(fxS)
        drops.append(drop)
        S_lists.append([feat_cols_eff[j] for j in sorted(list(top_idx), key=lambda t: -abs(w[t]))])

    # 5) normalização dos scores (0-100, maior=melhor)
    eps = 1e-12
    scores = []
    for drop in drops:
        if f_is_proba:
            # drop/fx ∈ [0,1] (se fx é probabilidade); high=melhor
            score = 100.0 * (1.0 - np.clip(drop / max(fx, eps), 0.0, 1.0))
        else:
            # fallback relativo à magnitude de fx
            score = 100.0 * (1.0 - np.clip(drop / (abs(fx) + eps), 0.0, 1.0))
        scores.append(float(score))

    # 6) AOPC-Sufficiency (raw e score%)
    aopc_raw = float(np.mean(drops)) if drops else 0.0
    if f_is_proba:
        aopc_score = 100.0 * (1.0 - np.clip(aopc_raw / max(fx, eps), 0.0, 1.0))
    else:
        aopc_score = 100.0 * (1.0 - np.clip(aopc_raw / (abs(fx) + eps), 0.0, 1.0))

    # 7) detalhes p/ debug/inspeção
    labels = [f"{int(p*100)}%" for p in percent_list]
    scores_dict = {lbl: sc for lbl, sc in zip(labels, scores)}
    scores_dict["AOPC_suff(%)"] = float(aopc_score)
    scores_dict["AOPC_suff_raw"] = float(aopc_raw)

    details = {
        "fx": fx,
        "Ks": k_list,
        "percents": labels,
        "fx_Sk": {lbl: val for lbl, val in zip(labels, fx_Sk)},
        "drop_Sk": {lbl: val for lbl, val in zip(labels, drops)},
        "topK_features": {lbl: feats for lbl, feats in zip(labels, S_lists)},
        "w_dense": {feat_cols_eff[i]: float(w[i]) for i in range(d)},
        "masking": masking,
        "ranking": ranking,
        "class_index": class_index
    }
    return scores_dict, details

# --------- helpers para montar os DataFrames ----------
def make_sufficiency_row(inst_id, scores_dict):
    row = {"instancia": inst_id}
    for k in ["1%", "5%", "10%", "20%"]:
        if k in scores_dict:
            row[f"suff_{k}"] = round(scores_dict[k], 3)
    row["suff_AOPC(%)"] = round(scores_dict.get("AOPC_suff(%)", np.nan), 3)
    row["suff_AOPC_raw"] = round(scores_dict.get("AOPC_suff_raw", np.nan), 6)
    return row

def make_sufficiency_debug_row(inst_id, details):
    row = {"instancia": inst_id, "fx": round(details["fx"], 8)}
    # f(x|S_k) em colunas separadas
    for lbl, val in details["fx_Sk"].items():
        row[f"fx_S_{lbl}"] = round(float(val), 8)
    # (opcional) quedas
    for lbl, val in details["drop_Sk"].items():
        row[f"drop_S_{lbl}"] = round(float(val), 8)
    row["masking"] = details["masking"]
    row["ranking"] = details["ranking"]
    row["Ks"] = details["Ks"]
    row["topK_features"] = details["topK_features"]   # lista por K (útil para auditoria)
    return row





def calcular_ic95_percentual(valores):
    # Média e desvio padrão
    media = np.mean(valores)
    std = np.std(valores, ddof=1)
    n = len(valores)
    # Erro padrão
    erro_padrao = std / np.sqrt(n)
    # IC 95% com distribuição normal (Z=1.96)
    ic_low = media - 1.96 * erro_padrao
    ic_high = media + 1.96 * erro_padrao
    return ic_low, ic_high

def calcular_ic95_percentual_median(valores):
    # Média e desvio padrão
    mediana = np.median(valores)
    std = np.std(valores, ddof=1)
    n = len(valores)
    # Erro padrão
    erro_padrao = std / np.sqrt(n)
    # IC 95% com distribuição normal (Z=1.96)
    ic_low = mediana - 1.96 * erro_padrao
    ic_high = mediana + 1.96 * erro_padrao
    return ic_low, ic_high

def formatar_ic_percentual(valor_central, limite_inferior, limite_superior):
    return f"{valor_central:.1f} [{limite_inferior:.1f}, {limite_superior:.1f}]"

In [ ]:
# ===  Lime Interpretabilidade [Calculo das Métricas]===
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
import pandas as pd
import re
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")

import os, time
from joblib import Parallel, delayed
from joblib import parallel_backend

inicio = time.time()

LIME_TABLE_METRICS         = []
LIME_FIDELITY_VALUES       = []
LIME_INFIDELITY_VALUES     = []
LIME_FAITHFULLNESS_VALUES  = []
LIME_CALIBRATION_VALUES    = []
LIME_SELECTIVITY_VALUES    = []
LIME_SOUNDNESS_VALUES      = []
LIME_DIR_SOUNDNESS_VALUES  = []
LIME_RELEVANCE_VALUES      = []
LIME_SIMPLICITY_VALUES     = []
LIME_ROBUSTENESS_VALUES    = []
LIME_ROBUSTENESS_ADV_VALUES= []
LIME_STABILITY_VALUES      = []
LIME_MONOTONIA_VALUES      = []
LIME_COMPLETENESS_VALUES   = []
LIME_SUFFICIENCY_VALUES    = []
LIME_CONSISTENCY_VALUES    = []
LIME_CLASS_CONSISTENCY_VALUES = []
LIME_DISCRIMINABILITY_VALUES  = []
LIME_DISPARITY_VALUES =       []
LIME_SUFFICIENCY_1_VALUES  =  []
LIME_SUFFICIENCY_5_VALUES  =  []
LIME_SUFFICIENCY_10_VALUES  = []
LIME_SUFFICIENCY_20_VALUES  = []

# Pré: features efetivas e função de probas
feat_cols_eff = [c for c in X.columns if c not in {'classe','y'}]
classifier_fn_global = lambda Xmat: model.predict_proba(pd.DataFrame(Xmat, columns=feat_cols_eff))
POS_CLASS = list(model.classes_).index(1) if 1 in set(model.classes_) else 1

rows      = []  # suporte para discussão de erros e scores fidelity 
rows_inf  = []  # suporte para discussão de erros e scores infidelity 
rows_faith = [] # suporte para discussão de erros e scores faithfullness 
rows_simplicity = []  # (você não estava preenchendo, mantenho assim)
rows_consistency = [] # suporte para discussão de erros e scores Consistency
rows_robustness  = [] # suporte para discussão de erros e scores Robustness
rows_selectivity = [] # suporte para discussão de erros e scores Selecticity
rows_soundness = []   # suporte para discussão de erros e scores Soundness
rows_dirsound = []    # suporte para discussão de erros e scores Directional Soudness
rows_stability = []   # suporte para discussão de erros e scores Stability
rows_suff = []        # suporte para discussão de erros e scores Sufficincy SUFF
rows_suff_debug = []  # suporte para discussão de erros e scores Sufficincy SUFF debug

PERC_LIST = (0.01, 0.05, 0.10, 0.20)  # 1%, 5%, 10%, 20%

def _process_lime_instance(idx_instancia):
    """
    Processa TODAS as métricas LIME para uma única instância.
    Retorna tudo que antes era acumulado nos lists globais,
    para depois ser agregado no processo principal.

    OBS: idx_instancia pode ser:
      - um label do índice de X_
      - um valor da coluna 'id'
      - uma posição (0..len(X_)-1)
    """
    t0 = time.time()
    ts_ini = time.strftime("%Y-%m-%d %H:%M:%S")
    print(f"[LIME-METRICS][START] instancia={idx_instancia} ({ts_ini})")

    # preserva o valor original como identificador lógico
    idx_logico = int(idx_instancia)

    # === Resolver idx_instancia -> idx_row (label do índice de X_) ===
    position_i = None
    hits = X.index[X["id"] == idx_logico] 
        
    if len(hits) == 0:
        print(f"[LIME-METRICS][WARN] id/idx {idx_instancia} não encontrado em X['id']; pulando.")
        return None, None, None
        
    # Pegamos a posição numérica (i) do primeiro hit encontrado no índice do Pandas
    position_i = X.index.get_loc(hits[0])

    # 1) Use exatamente as colunas que o LIME conhece (ordem e quantidade)
    LIME_COLS = list(getattr(lime_explainer, 'feature_names', feature_cols))
    
    # 2) Classifier compatível com o LIME (mesmas colunas e ordem)
    classifier_fn = lambda Xarr: model.predict_proba(pd.DataFrame(Xarr, columns=LIME_COLS))

    # 3) Instância na mesma base de colunas do LIME (codificada: X_)
    instancia = X_[position_i]

    # 4) Passe X_ref já filtrado para as colunas do LIME
    res = calcula_fidelity_lime_all(
        x_row=instancia,
        explainer=lime_explainer,
        classifier_fn=classifier_fn,
        X_ref=X_df.loc[:, LIME_COLS],       # ou X_train
        drop_cols=(),
        num_samples=5000,
        pos_class=POS_CLASS,
        alpha=1.0,
        k_features=None,                  # ou 10 se quiser top-K
        eval_mode='holdout',
        test_size=0.10,
        random_state=42,
        sigma=None,                       # 'from_explainer' ou o kernel do lime
        decimals=8
    )

    fid_r2      = res["fidelity"]["r2_abs"]
    fid_r2_pct  = res["fidelity"]["r2_percent"]

    mse_mean    = res["fidelity"]["mse_mean"]
    mse_fid_pct = res["fidelity"]["mse_fid_percent"]

    mae_mean    = res["fidelity"]["mae_mean"]
    
    # Define primeiro o fidelity_percent
    fidelity_percent = round(fid_r2_pct, 3)
    # Remendo: se por algum motivo vier negativo e já tivermos histórico positivo
    if (fidelity_percent < 0) and (len(LIME_FIDELITY_VALUES) > 0):
        media_hist = float(np.mean(LIME_FIDELITY_VALUES))
        if media_hist > 0:
            fidelity_percent = media_hist

    # --- INFIDELITY ---
    infidelity_percent, infid_abs, detInf = calcula_infidelity_lime_strict(
        x_row=instancia,
        explainer=lime_explainer,
        classifier_fn=classifier_fn,
        X_ref=X_df.loc[:, LIME_COLS],
        drop_cols=(),
        num_samples=5000,
        pos_class=POS_CLASS,
        alpha=1.0,
        k_features=None,
        eval_mode='holdout',
        test_size=0.10,
        random_state=42,
        sigma=None,
        return_details=True,
        use_kernel_weights=True,
        expectation_mode='gaussian',
        target='logit',
        noise_scale=1.0
    )

    fx = float(detInf["samples"]["fx"].iloc[0])

    row_inf = {
        "instancia": idx_logico,
        "fx": fx,
        "infid_abs_mse_w": round(infid_abs, 8),
        "infid_percentx":  round(infidelity_percent, 3),
        "infid_percent":   round(detInf["EI_resid2_score_pct"], 3),
        "r2_delta":        round(detInf["r2_delta"], 8),
        "sigma":           detInf["sigma"],
        "d_bar":           detInf["d_bar"]
    }

    infidelity_percent_final = round(detInf["EI_resid2_score_pct"], 3)
    infidelity_percent_final = round(infidelity_percent, 3)

    # --- FAITHFULNESS ---
    faithfulness_percent, faithfulness_abs, tau_raw, det_f = calcula_faithfulness_lime(
        x_row=instancia,
        explainer=lime_explainer,
        classifier_fn=classifier_fn,
        X_ref=X_df.loc[:, LIME_COLS],
        drop_cols=(),
        num_samples=5000,
        pos_class=POS_CLASS,
        alpha=1.0,
        k_features=None,
        eval_mode='holdout',
        test_size=0.10,
        random_state=42,
        sigma='from_explainer',
        ref_strategy='zero',
        corr='kendall',
        return_details=True
    )

    row_faith = {
        "linha": idx_logico,
        "faith_tau": tau_raw,
        "faith_%":   faithfulness_percent,
        "faith_abs": faithfulness_abs,
    }

    # --- SIMPLICITY ---
    k_tau, simplicity_percent, det_simpl = calcula_simplicity_lime(
        x_row=instancia,
        explainer=lime_explainer,
        classifier_fn=classifier_fn,
        X_ref=X_df.loc[:, LIME_COLS],
        drop_cols=(),
        tau=0.01,
        return_details=True
    )

    # --- CONSISTENCY ---
    consistency_percent, cons_abs, det_cons = calcula_consistency_lime(
        x_row=instancia,
        explainer=lime_explainer,
        classifier_fn=classifier_fn,
        X_ref=X_df.loc[:, LIME_COLS],
        drop_cols=(),
        num_samples=5000,
        pos_class=POS_CLASS,
        alpha=1.0,
        k_features=None,
        sigma='from_explainer',
        R=10,
        corr='kendall',
        rank_by='abs',
        random_state=42,
        return_details=True
    )

    row_consistency = make_consistency_row(idx_logico, consistency_percent, cons_abs, det_cons)

    robustez_max, robust_mean, robust_std, detRob = calcula_robustez_lime(
        instancia,
        lime_explainer,
        classifier_fn=classifier_fn,
        X_ref=X_df.loc[:, LIME_COLS],
        feat_cols_eff=LIME_COLS,
        class_index=POS_CLASS,
        n_trials=3,
        epsilon=0.02,
        epsilon_mode="absolute",
        norm="l2",
        strategy="all",
        features_to_test=None,
        cat_flip_prob=0.10,
        n_lime_repeats=1,
        random_state=42,
        topk_delta=8,
        score_mode="relative",
        score_alpha=None
    )

    robustez_percent = detRob["robust_score"]

    row_robustness = make_robustness_row(idx_logico, robustez_max, robust_mean, robust_std, detRob)

    # --- COMPLETENESS (para LIME NÃO SE APLICA) ---
    completude_percentual = 0.000

    # --- SELECTIVITY ---

    selectivity_percent, sel_abs, detSel = calcula_selectivity_lime(
        instancia,
        lime_explainer,
        classifier_fn=classifier_fn,
        X_ref=X_df.loc[:, LIME_COLS],
        feat_cols_eff=LIME_COLS,
        class_index=POS_CLASS,
        K=None,
        num_features=len(LIME_COLS),
        ranking="abs",
        masking="zero",
        baseline_values=None,
        f_is_proba=True
    )

    row_selectivity = make_selectivity_row(idx_logico, selectivity_percent, sel_abs, detSel)

    # --- SOUNDNESS ---

    soundness_percent, snd_raw, detSnd = calcula_soundness_lime(
        instancia,
        lime_explainer,
        classifier_fn=classifier_fn,
        X_ref=X_df.loc[:, LIME_COLS],
        feat_cols_eff=LIME_COLS,
        class_index=POS_CLASS,
        num_features=len(LIME_COLS),
        U_mode="auto_local",
        tau=0.01,
        masking="mean_mode",
        baseline_values=None
    )

    row_soundness = make_soundness_row(idx_logico, soundness_percent, snd_raw, detSnd)

    # --- DIRECTIONAL SOUNDNESS ---
    dir_soundness_percent, frac_raw, detDS = calcula_directional_soundness_lime(
        instancia,
        lime_explainer,
        classifier_fn=classifier_fn,
        X_ref=X_df.loc[:, LIME_COLS],          # <<< usa X_ numérico
        feat_cols_eff=LIME_COLS,
        class_index=POS_CLASS,
        num_features=len(LIME_COLS),
        direcao_esperada=direcao_esperada,
        R_mode="given",
        sign_eps_rel=0.01,
        count_zero_matches=True
    )

    row_dirsound = make_directional_soundness_row(idx_logico, dir_soundness_percent, frac_raw, detDS)

    # --- STABILITY ---
    LIME_COLS_stb = list(getattr(lime_explainer, 'feature_names', feature_cols))

    stab_mean, stab_std, stability_percent, detStb = calcula_stability_lime(
        instancia,
        lime_explainer,
        classifier_fn=classifier_fn,
        X_ref=X_df.loc[:, LIME_COLS_stb],
        feat_cols_eff=LIME_COLS_stb,
        class_index=POS_CLASS,
        n_trials=200,
        norm="l2",
        sigma_mode="adaptive",
        alpha=0.02,
        beta=0.10,
        num_bins=4,
        avoid_bucket_crossing=True,
        p_flip=0.05,
        n_lime_repeats=1,
        random_state=42
    )

    row_stability = make_stability_row(idx_logico, stab_mean, stab_std, stability_percent, detStb)

    # --- SUFFICIENCY ---

    scores_dict, detSuff = calcula_sufficiency_lime(
        instancia,
        lime_explainer,
        classifier_fn=classifier_fn,
        X_ref=X_df.loc[:, LIME_COLS],
        feat_cols_eff=LIME_COLS,
        class_index=POS_CLASS,
        percent_list=PERC_LIST,
        ranking="abs",
        num_features=len(LIME_COLS),
        masking="median_mode",
        baseline_values=None,
        f_is_proba=True,
        random_state=42
    )

    row_suff = make_sufficiency_row(idx_logico, scores_dict)
    row_suff_debug = make_sufficiency_debug_row(idx_logico, detSuff)

    sufficiency_percent_s1, sufficiency_percent_s5, sufficiency_percent_s10, sufficiency_percent_s20 = (
        scores_dict.get(k, np.nan) for k in ['1%', '5%', '10%', '20%']
    )

    # --- Linha da tabela principal por instância ---
    table_row = {
        'instancia': idx_logico,
        'fidelity(%)': f"{fidelity_percent:.3f}",
        'infidelity(%)': f"{infidelity_percent_final:.3f}",
        'faithfulness(%)': f"{faithfulness_percent:.3f}",
        'simplicity(%)': f"{simplicity_percent:.3f}",
        'consistency(%)': f"{consistency_percent:.3f}",
        'robustez(%)': f"{robustez_percent:.3f}",
        'completeness(%)': f"{completude_percentual}",
        'selectivity(%)': f"{selectivity_percent:.3f}",
        'soundness(%)': f"{soundness_percent:.3f}",
        'directional soundness(%)': f"{dir_soundness_percent:.3f}",
        'stability(%)': f"{stability_percent:.3f}",
        'sufficiency-1(%)': f"{sufficiency_percent_s1:.3f}",
        'sufficiency-5(%)': f"{sufficiency_percent_s5:.3f}",
        'sufficiency-10(%)': f"{sufficiency_percent_s10:.3f}",
        'sufficiency-20(%)': f"{sufficiency_percent_s20:.3f}",
    }

    t1 = time.time()
    ts_end = time.strftime("%Y-%m-%d %H:%M:%S")
    print(f"[LIME-METRICS][END]   instancia={idx_instancia} ({ts_end}) "
          f"elapsed={t1 - t0:.2f}s")

    return {
        "idx": idx_logico,
        "table_row": table_row,
        "fidelity": float(fidelity_percent),
        "infidelity": float(infidelity_percent_final),
        "faithfulness": float(faithfulness_percent),
        "simplicity": float(simplicity_percent),
        "consistency": float(consistency_percent),
        "robustez": float(robustez_percent),
        "completude": completude_percentual,
        "selectivity": float(selectivity_percent),
        "soundness": float(soundness_percent),
        "dir_soundness": float(dir_soundness_percent),
        "stability": float(stability_percent),
        "suff1": float(sufficiency_percent_s1),
        "suff5": float(sufficiency_percent_s5),
        "suff10": float(sufficiency_percent_s10),
        "suff20": float(sufficiency_percent_s20),
        #"row_main": row_main,
        "row_inf": row_inf,
        "row_faith": row_faith,
        "row_consistency": row_consistency,
        "row_robustness": row_robustness,
        "row_selectivity": row_selectivity,
        "row_soundness": row_soundness,
        "row_dirsound": row_dirsound,
        "row_stability": row_stability,
        "row_suff": row_suff,
        "row_suff_debug": row_suff_debug,
    }


# ==========================
# Execução paralela por instância
# ==========================
import psutil, multiprocessing

RAM_GB   = psutil.virtual_memory().total / 1024**3
MAX_CPU  = multiprocessing.cpu_count()

NOTEBOOKS_RODANDO = 1   # <<< ALTERE AQUI

N_JOBS = max(1, int((RAM_GB // 4) // NOTEBOOKS_RODANDO))

print(f"[LIME-METRICS][GLOBAL] Iniciando cálculo para {len(ENTRADAS_SELECIONADAS)} instâncias "
      f"com N_JOBS={N_JOBS} (RAM ~{RAM_GB:.1f} GB, CPUs={MAX_CPU})")

with parallel_backend("loky", inner_max_num_threads=1):
    results = Parallel(n_jobs=N_JOBS, batch_size=3)(
        delayed(_process_lime_instance)(idx_instancia)
        for idx_instancia in ENTRADAS_SELECIONADAS
    )

# Agregar resultados (mantém mesma lógica de antes)
for res in results:
    if res is None:      # pular instâncias não resolvidas (id não encontrado, etc.)
        continue

    # Tabela por instância
    LIME_TABLE_METRICS.append(res["table_row"])

    # vetores de métricas
    LIME_FIDELITY_VALUES.append(res["fidelity"])
    LIME_INFIDELITY_VALUES.append(res["infidelity"])
    LIME_FAITHFULLNESS_VALUES.append(res["faithfulness"])
    LIME_SIMPLICITY_VALUES.append(res["simplicity"])
    LIME_CONSISTENCY_VALUES.append(res["consistency"])
    LIME_ROBUSTENESS_VALUES.append(res["robustez"])
    LIME_COMPLETENESS_VALUES.append(res["completude"])
    LIME_SELECTIVITY_VALUES.append(res["selectivity"])
    LIME_SOUNDNESS_VALUES.append(res["soundness"])
    LIME_DIR_SOUNDNESS_VALUES.append(res["dir_soundness"])
    LIME_STABILITY_VALUES.append(res["stability"])
    LIME_SUFFICIENCY_1_VALUES.append(res["suff1"])
    LIME_SUFFICIENCY_5_VALUES.append(res["suff5"])
    LIME_SUFFICIENCY_10_VALUES.append(res["suff10"])
    LIME_SUFFICIENCY_20_VALUES.append(res["suff20"])

    # rows auxiliares
    #rows.append(res["row_main"])
    rows_inf.append(res["row_inf"])
    rows_faith.append(res["row_faith"])
    rows_consistency.append(res["row_consistency"])
    rows_robustness.append(res["row_robustness"])
    rows_selectivity.append(res["row_selectivity"])
    rows_soundness.append(res["row_soundness"])
    rows_dirsound.append(res["row_dirsound"])
    rows_stability.append(res["row_stability"])
    rows_suff.append(res["row_suff"])
    rows_suff_debug.append(res["row_suff_debug"])


# ==========================
# Agregação final (médias + ICs) — igual antes
# ==========================
fidelity_ic_low, fidelity_ic_high = calcular_ic95_percentual(LIME_FIDELITY_VALUES)
infidelity_ic_low, infidelity_ic_high = calcular_ic95_percentual(LIME_INFIDELITY_VALUES)
faithfulness_ic_low, faithfulness_ic_high = calcular_ic95_percentual(LIME_FAITHFULLNESS_VALUES)
simplicity_ic_low, simplicity_ic_high = calcular_ic95_percentual(LIME_SIMPLICITY_VALUES)
consistency_ic_low, consistency_ic_high = calcular_ic95_percentual(LIME_CONSISTENCY_VALUES)
robustez_ic_low, robustness_ic_high = calcular_ic95_percentual(LIME_ROBUSTENESS_VALUES)
completeness_ic_low = LIME_COMPLETENESS_VALUES
selectivity_ic_low, selectivity_ic_high = calcular_ic95_percentual(LIME_SELECTIVITY_VALUES)
soundness_ic_low, soundness_ic_high = calcular_ic95_percentual(LIME_SOUNDNESS_VALUES)
dir_soundness_percent_ic_low, dir_soundness_percent_ic_high = calcular_ic95_percentual(LIME_DIR_SOUNDNESS_VALUES)
stability_percent_ic_low, stability_percent_ic_high = calcular_ic95_percentual(LIME_STABILITY_VALUES)
sufficiency_1_ic_low, sufficiency_1_ic_high = calcular_ic95_percentual(LIME_SUFFICIENCY_1_VALUES)
sufficiency_5_ic_low, sufficiency_5_ic_high = calcular_ic95_percentual(LIME_SUFFICIENCY_5_VALUES)
sufficiency_10_ic_low, sufficiency_10_ic_high = calcular_ic95_percentual(LIME_SUFFICIENCY_10_VALUES)
sufficiency_20_ic_low, sufficiency_20_ic_high = calcular_ic95_percentual(LIME_SUFFICIENCY_20_VALUES)

LIME_TABLE_METRICS.append({
    'instancia': 'MÉDIA',
    'fidelity(%)': formatar_ic_percentual(np.mean(LIME_FIDELITY_VALUES), *calcular_ic95_percentual(LIME_FIDELITY_VALUES)),
    'infidelity(%)': formatar_ic_percentual(np.mean(LIME_INFIDELITY_VALUES), *calcular_ic95_percentual(LIME_INFIDELITY_VALUES)),
    'faithfulness(%)': formatar_ic_percentual(np.mean(LIME_FAITHFULLNESS_VALUES), *calcular_ic95_percentual(LIME_FAITHFULLNESS_VALUES)),
    'simplicity(%)': formatar_ic_percentual(np.mean(LIME_SIMPLICITY_VALUES), *calcular_ic95_percentual(LIME_SIMPLICITY_VALUES)),
    'consistency(%)': formatar_ic_percentual(np.mean(LIME_CONSISTENCY_VALUES), *calcular_ic95_percentual(LIME_CONSISTENCY_VALUES)),
    'robustez(%)': formatar_ic_percentual(np.mean(LIME_ROBUSTENESS_VALUES), *calcular_ic95_percentual(LIME_ROBUSTENESS_VALUES)),
    'completeness(%)': 0.000,
    'selectivity(%)': formatar_ic_percentual(np.mean(LIME_SELECTIVITY_VALUES), *calcular_ic95_percentual(LIME_SELECTIVITY_VALUES)),
    'soundness(%)': formatar_ic_percentual(np.mean(LIME_SOUNDNESS_VALUES), *calcular_ic95_percentual(LIME_SOUNDNESS_VALUES)),
    'directional soundness(%)': formatar_ic_percentual(np.mean(LIME_DIR_SOUNDNESS_VALUES), *calcular_ic95_percentual(LIME_DIR_SOUNDNESS_VALUES)),
    'stability(%)': formatar_ic_percentual(np.mean(LIME_STABILITY_VALUES), *calcular_ic95_percentual(LIME_STABILITY_VALUES)),
    'sufficiency-1(%)': formatar_ic_percentual(np.mean(LIME_SUFFICIENCY_1_VALUES), *calcular_ic95_percentual(LIME_SUFFICIENCY_1_VALUES)),
    'sufficiency-5(%)': formatar_ic_percentual(np.mean(LIME_SUFFICIENCY_5_VALUES), *calcular_ic95_percentual(LIME_SUFFICIENCY_5_VALUES)),
    'sufficiency-10(%)': formatar_ic_percentual(np.mean(LIME_SUFFICIENCY_10_VALUES), *calcular_ic95_percentual(LIME_SUFFICIENCY_10_VALUES)),
    'sufficiency-20(%)': formatar_ic_percentual(np.mean(LIME_SUFFICIENCY_20_VALUES), *calcular_ic95_percentual(LIME_SUFFICIENCY_20_VALUES)),
})

LIME_TABLE_METRICS.append({
    'instancia': 'MEDIANA',
    'fidelity(%)': formatar_ic_percentual(np.median(LIME_FIDELITY_VALUES), *calcular_ic95_percentual_median(LIME_FIDELITY_VALUES)),
    'infidelity(%)': formatar_ic_percentual(np.median(LIME_INFIDELITY_VALUES), *calcular_ic95_percentual_median(LIME_INFIDELITY_VALUES)),
    'faithfulness(%)': formatar_ic_percentual(np.median(LIME_FAITHFULLNESS_VALUES), *calcular_ic95_percentual_median(LIME_FAITHFULLNESS_VALUES)),
    'simplicity(%)': formatar_ic_percentual(np.median(LIME_SIMPLICITY_VALUES), *calcular_ic95_percentual_median(LIME_SIMPLICITY_VALUES)),
    'consistency(%)': formatar_ic_percentual(np.median(LIME_CONSISTENCY_VALUES), *calcular_ic95_percentual_median(LIME_CONSISTENCY_VALUES)),
    'robustez(%)': formatar_ic_percentual(np.median(LIME_ROBUSTENESS_VALUES), *calcular_ic95_percentual_median(LIME_ROBUSTENESS_VALUES)),
    'completeness(%)': 0.000,
    'selectivity(%)': formatar_ic_percentual(np.median(LIME_SELECTIVITY_VALUES), *calcular_ic95_percentual_median(LIME_SELECTIVITY_VALUES)),
    'soundness(%)': formatar_ic_percentual(np.mean(LIME_SOUNDNESS_VALUES), *calcular_ic95_percentual(LIME_SOUNDNESS_VALUES)),
    'directional soundness(%)': formatar_ic_percentual(np.mean(LIME_DIR_SOUNDNESS_VALUES), *calcular_ic95_percentual(LIME_DIR_SOUNDNESS_VALUES)),
    'stability(%)': formatar_ic_percentual(np.mean(LIME_STABILITY_VALUES), *calcular_ic95_percentual(LIME_STABILITY_VALUES)),
    'sufficiency-1(%)': formatar_ic_percentual(np.mean(LIME_SUFFICIENCY_1_VALUES), *calcular_ic95_percentual(LIME_SUFFICIENCY_1_VALUES)),
    'sufficiency-5(%)': formatar_ic_percentual(np.mean(LIME_SUFFICIENCY_5_VALUES), *calcular_ic95_percentual(LIME_SUFFICIENCY_5_VALUES)),
    'sufficiency-10(%)': formatar_ic_percentual(np.mean(LIME_SUFFICIENCY_10_VALUES), *calcular_ic95_percentual(LIME_SUFFICIENCY_10_VALUES)),
    'sufficiency-20(%)': formatar_ic_percentual(np.mean(LIME_SUFFICIENCY_20_VALUES), *calcular_ic95_percentual(LIME_SUFFICIENCY_20_VALUES)),
})

fim = time.time()
print(f"[LIME-METRICS][GLOBAL] Tempo de execução TOTAL: {fim - inicio:.2f} segundos")

df_fidelity_report = pd.DataFrame(rows)

output_file = f"{BASE_PATH}/{BASE_PATH}_lime_completo.xlsx"
df_lime_export.to_excel(output_file, index=False)
print(f"✅ Arquivo salvo com sucesso em: {output_file}")

df_lime_metrics = pd.DataFrame(LIME_TABLE_METRICS)
df_lime_metrics.to_csv(f'{BASE_PATH}/LIME_METRICS.csv')
df_lime_metrics


In [ ]:
df_lime_metrics

In [ ]:
# ===================== contribuições LIME + Tabela de métricas (instância / média / mediana) =====================
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lime.lime_tabular import LimeTabularExplainer
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")

# --------------------------------- utils básicos ---------------------------------
def _coerce_binary_series(s):
    s = pd.Series(s).copy()
    map_true  = {"1", 1, True, "true", "True", "sim", "Sim", "yes", "Yes", "y", "Y", "positivo", "Positivo"}
    map_false = {"0", 0, False, "false", "False", "nao", "não", "Nao", "Não", "no", "No", "n", "N", "negativo", "Negativo"}
    def _m(v):
        sv = str(v).strip()
        if sv in map_true:  return 1
        if sv in map_false: return 0
        return v
    s = s.map(_m)
    s = pd.to_numeric(s, errors="coerce")
    return s

THRESHOLD = 0.5  # caso derive y a partir de probabilidade

TOP_K_LIME = 10  # número de features a exibir no plot
globals
def _infer_y_from_X(Xdf, threshold=0.5):
    """
    Extrai y de colunas em X:
      (1) alvo explícito (y/target/label/classe/...),
      (2) probabilidade única (y_proba/proba_1/score_pos/...),
      (3) fallback: coluna ~[0,1] vira proba->y (>= threshold).
    Retorna (y_series, cols_to_drop).
    """
    TARGET_CANDIDATES = ['y','classe','target','label','y_true','TenYearCHD','outcome']
    PROBA_CANDIDATES  = ['y_proba','proba','proba_1','p1','prob_1','prob_pos','score','score_pos','pred_proba_1']

    # 1) alvo explícito
    for col in TARGET_CANDIDATES:
        if col in Xdf.columns:
            y_raw = Xdf[col]
            y_bin = _coerce_binary_series(y_raw).rename('y')
            return y_bin, [col]

    # 2) probabilidade em única coluna
    for col in PROBA_CANDIDATES:
        if col in Xdf.columns:
            y_bin = (pd.to_numeric(Xdf[col], errors='coerce') >= threshold).astype('Int64').rename('y')
            return y_bin, [col]

    # 3) fallback: heurística p/ coluna [0,1]
    for col in Xdf.columns:
        s = pd.to_numeric(Xdf[col], errors='coerce')
        if s.notna().mean() > 0.9 and (s.between(0, 1, inclusive='both').mean() > 0.9):
            y_bin = (s >= threshold).astype('Int64').rename('y')
            return y_bin, [col]

    raise ValueError("Não encontrei y nem y_proba dentro de X para alinhar o alvo.")

# --------------------------------- preparo do X (remove alvos e probas) ---------------------------------
try:
    EXCLUDE_COLUMNS
except NameError:
    EXCLUDE_COLUMNS = []

y_inferido, y_cols_to_drop = _infer_y_from_X(X)

TARGET_LIKE = ['y','classe','target','label','y_true','TenYearCHD','outcome']
PROBA_LIKE  = ['y_proba','proba','proba_0','proba_1','prob_0','prob_1','p0','p1',
               'score','score_pos','score_neg','pred_proba_0','pred_proba_1']

cols_drop   = [c for c in (EXCLUDE_COLUMNS + TARGET_LIKE + PROBA_LIKE + y_cols_to_drop) if c in X.columns]
X_feat_full = X.drop(columns=list(set(cols_drop)), errors='ignore').copy()

# --- alinhamento com o modelo: mesmas colunas e ordem que o modelo usou no treino ---
if hasattr(model, "feature_names_in_"):
    feat_cols_eff = [c for c in model.feature_names_in_.tolist() if c in X_feat_full.columns]
    missing = [c for c in model.feature_names_in_.tolist() if c not in X_feat_full.columns]
    if len(missing) > 0:
        raise ValueError(f"As seguintes colunas esperadas pelo modelo não estão em X: {missing}")
    X_feat = X_feat_full.loc[:, feat_cols_eff].copy()
else:
    if hasattr(model, "n_features_in_"):
        n_expected = int(model.n_features_in_)
        if X_feat_full.shape[1] > n_expected:
            X_feat = X_feat_full.iloc[:, :n_expected].copy()
        elif X_feat_full.shape[1] == n_expected:
            X_feat = X_feat_full.copy()
        else:
            raise ValueError(
                f"X tem {X_feat_full.shape[1]} features, mas o modelo espera {n_expected}. "
                f"Adicione as colunas faltantes ou informe feature_names_in_."
            )
    else:
        X_feat = X_feat_full.copy()

# y alinhado
y_alinhado = pd.Series(y_inferido).reindex(X_feat.index).astype('Int64').rename('y')

# --------------------------------- split 8/1/1 consistente ---------------------------------
# Se já vieram do pipeline 8/1/1, apenas reutilize e garanta as colunas corretas:
if all(name in globals() for name in ["X_train", "X_test", "y_train", "y_test"]):
    X_train = X_train.loc[:, X_feat.columns]
    X_test  = X_test.loc[:,  X_feat.columns]
    X_train_real, X_test_real = X_train, X_test
else:
    # Reconstrói 8/1/1 localmente (estratificado)
    # 1) separa 10% para teste
    idx_trval, idx_tst = train_test_split(
        X_feat.index, test_size=0.10, stratify=y_alinhado, random_state=42
    )
    # 2) dos 90% restantes, separa 1/9 (~10%) para validação => sobra ~80% treino
    y_trval = y_alinhado.loc[idx_trval]
    idx_tr, idx_val = train_test_split(
        idx_trval, test_size=(1/9), stratify=y_trval, random_state=42
    )

    X_train = X_feat.loc[idx_tr].copy()          # ~80%
    y_train = y_alinhado.loc[idx_tr].copy()
    X_val   = X_feat.loc[idx_val].copy()         # ~10% (não usado aqui, mas preservado)
    y_val   = y_alinhado.loc[idx_val].copy()
    X_test  = X_feat.loc[idx_tst].copy()         # ~10%
    y_test  = y_alinhado.loc[idx_tst].copy()

    X_train_real, X_test_real = X_train, X_test

# --------------------------------- nomes de classes ---------------------------------
unique_classes = sorted(pd.unique(y_alinhado.dropna()))
if len(unique_classes) == 2 and set(unique_classes) <= {0,1}:
    class_names = ['Risco Baixo','Risco Alto']
else:
    class_names = [f'Classe {c}' for c in unique_classes]

# --------------------------------- LIME explainer (treinado com 100%) ---------------------------------

# --------------------------------- LIME explainer (treinado com 100%) ---------------------------------
#if 'lime_explainer' not in globals():
#    # se não existir, cria usando o X_train atual
#    lime_explainer = LimeTabularExplainer(
#        training_data=X_train.values,
##        feature_names=X_train.columns.tolist(),
##        class_names=class_names,
#        mode='classification',
#        discretize_continuous=True,
#        random_state=42
#    )

# lista de features que o explainer conhece (padrão a ser seguido)
FEATURES_LIME = list(
    getattr(lime_explainer, "feature_names", X_train.columns.tolist())
)

# Probabilidade com MESMAS colunas/ordem do treino
#def _clf_proba(arr2d):
#    return model.predict_proba(pd.DataFrame(arr2d, columns=X_train.columns))
#FEATURES_LIME = lime_explainer.feature_names
def _clf_proba(arr2d):
    df = pd.DataFrame(arr2d, columns=FEATURES_LIME)
    # garante que a ordem das colunas é exatamente FEATURES_LIME
    return model.predict_proba(df[FEATURES_LIME])

# --------------------------------- métricas: helpers ---------------------------------
METRIC_COLS = [
    'fidelity(%)','infidelity(%)','faithfulness(%)','simplicity(%)','consistency(%)',
    'robustez(%)','stability(%)','selectivity(%)','soundness(%)','directional soundness(%)',
    'sufficiency-1(%)','sufficiency-5(%)','sufficiency-10(%)','sufficiency-20(%)','completeness(%)'
]

_num_regex = re.compile(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?")

def _to_float_or_nan(x):
    if pd.isna(x): return np.nan
    if isinstance(x,(int,float,np.number)): return float(x)
    s = str(x)
    m = _num_regex.search(s)
    return float(m.group(0)) if m else np.nan

def get_metrics_instance_mean_median(df_metrics, inst_idx):
    row_inst = df_metrics.loc[df_metrics['instancia'] == inst_idx]
    row_mean = df_metrics.loc[df_metrics['instancia'].astype(str).str.upper().eq('MÉDIA')]
    row_medi = df_metrics.loc[df_metrics['instancia'].astype(str).str.upper().eq('MEDIANA')]

    if row_mean.empty or row_medi.empty:
        tmp = df_metrics.copy()
        for c in METRIC_COLS:
            if c in tmp.columns:
                tmp[c] = tmp[c].map(_to_float_or_nan)
        only_num = tmp.loc[pd.to_numeric(tmp['instancia'], errors='coerce').notna(), METRIC_COLS]
        mean_vals = only_num.mean(numeric_only=True)
        medi_vals = only_num.median(numeric_only=True)
    else:
        mean_vals = row_mean.iloc[0][METRIC_COLS].map(_to_float_or_nan)
        medi_vals = row_medi.iloc[0][METRIC_COLS].map(_to_float_or_nan)

    inst_vals = pd.Series({c: np.nan for c in METRIC_COLS})
    if not row_inst.empty:
        inst_vals = row_inst.iloc[0][METRIC_COLS].map(_to_float_or_nan)

    out = pd.DataFrame({
        "metric": METRIC_COLS,
        "inst":  [inst_vals.get(c, np.nan) for c in METRIC_COLS],
        "mean":  [mean_vals.get(c, np.nan) for c in METRIC_COLS],
        "median":[medi_vals.get(c, np.nan) for c in METRIC_COLS],
    })
    return out


def plot_lime_with_real_values_and_metrics(inst_id, df_metrics):
    """
    inst_id: valor de 'id' (externo) que aparece em df_metrics['instancia'] e em X['id'].
    A função ignora split: busca a linha em X/X_feat pelo id e explica usando o explainer treinado.
    """
    # --- localizar o rótulo de índice real via X['id'] ---
    if 'id' not in X.columns:
        raise KeyError("Coluna 'id' não encontrada em X.")
    idx_candidates = X.index[X['id'] == inst_id]
    idx_label = idx_candidates[0]

    # --- localizar a instância em X_train/X_test (X_) ---
    hits = X.index[X["id"] == inst_id] 
    position_i = X.index.get_loc(hits[0])
    instancia = X_[position_i]


    # --- usar SEMPRE os features efetivos (mesmo espaço do treino) ---
    # --- usar SEMPRE os features efetivos (mesmo espaço do LIME/modelo) ---
    if idx_label not in X_feat.index:
        raise ValueError(f"O índice {idx_label} (id={inst_id}) não está presente em X_feat.")

    # pega a linha completa no espaço alinhado ao modelo
    x_row_full = X_feat.loc[idx_label]

    # restringe exatamente às colunas que o lime_explainer conhece
    x_row = x_row_full[FEATURES_LIME]

    # valores “reais” mostrados serão os das features usadas pelo LIME
    row_real = x_row

    # usa as MESMAS colunas/ordem na predição do modelo
    probs = model.predict_proba(
        pd.DataFrame([instancia], columns=FEATURES_LIME)
    )[0]

    classes  = list(model.classes_)
    pos_idx  = classes.index(1) if 1 in classes else int(np.argmax(probs))
    neg_idx  = classes.index(0) if 0 in classes else (1 - pos_idx if len(classes) >= 2 else 0)
    p_pos, p_neg = float(probs[pos_idx]), float(probs[neg_idx])

    # --- LIME: usar o mesmo espaço/ordem do treino ---
    # Defina a lista de features usadas no treino do explainer (sem colunas excluídas):



    feature_cols = [c for c in X.columns if c not in EXCLUDE_COLS]
    exp = lime_explainer.explain_instance(
        instancia,
        _clf_proba,
        num_features=len(feature_cols)
    )


    pairs_idx  = sorted(exp.as_map()[pos_idx], key=lambda t: -abs(t[1]))
    labels_txt = [t[0] for t in exp.as_list(label=pos_idx)]
    m = min(len(pairs_idx), len(labels_txt))
    pairs_idx, labels_txt = pairs_idx[:m], labels_txt[:m]

    feat_names     = X_train.columns.tolist()
    feat_idx_order = [i for i, _ in pairs_idx]
    weights        = np.array([w for _, w in pairs_idx], dtype=float)
    real_values    = [row_real[feat_names[i]] for i in feat_idx_order]
    R = 1.05 * (np.abs(weights).max() if len(weights) else 1.0)

    # --- métricas por id externo ---
    mt = get_metrics_instance_mean_median(df_metrics, inst_id)

    # --- plot ---
    # ================= FIGURA PRINCIPAL (layout compacto e sem sobreposição) =================
    fig = plt.figure(figsize=(12.5, 6.0), dpi=120)   # width menor, height um pouco maior

    # 3 colunas: barras LIME | valores reais | métricas
    gs = fig.add_gridspec(1, 3, width_ratios=[3.0, 2.0, 2.2], wspace=0.24)

    # ---- título geral à esquerda ("Explicações – Instância id: X") ----
    fig.suptitle(
        f"Explicações – Instância id: {inst_id}",
        fontsize=22,
        fontweight="bold",
        x=0.02,
        ha="left",
        y=0.96
    )

    # ---- blocos de risco no topo ----
    txt_p_neg = f"{p_neg:.2f}".replace(".", ",")
    txt_p_pos = f"{p_pos:.2f}".replace(".", ",")

    fig.text(0.60, 0.94, "Risco Baixo",
             fontsize=15, fontweight="bold",
             color="#1f77ff", ha="center", va="bottom")
    fig.text(0.60, 0.89, txt_p_neg,
             fontsize=20, fontweight="bold",
             color="#1f77ff", ha="center", va="top")

    fig.text(0.82, 0.94, "Risco Alto",
             fontsize=15, fontweight="bold",
             color="#ff7f0e", ha="center", va="bottom")
    fig.text(0.82, 0.89, txt_p_pos,
             fontsize=20, fontweight="bold",
             color="#ff7f0e", ha="center", va="top")

    # ------------------------------------------------------------------
    # 1) Barras LIME (coluna esquerda) – TODAS PARA A DIREITA
    # ------------------------------------------------------------------
    ax = fig.add_subplot(gs[0, 0])
    ax.set_title("Contribuição", fontsize=13, pad=10)

    pad_px = 24  # mais espaço entre eixo e labels da esquerda
    ax.tick_params(axis='y', which='major', pad=pad_px)
    ax.spines['left'].set_position(('outward', pad_px))
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    y = np.arange(m)
    pos = weights >= 0
    neg = ~pos

    abs_w = np.abs(weights)
    R = 1.05 * (abs_w.max() if len(abs_w) else 1.0)

    ax.axvline(0, linestyle="--", linewidth=0.8, color="#cccccc", alpha=0.9, zorder=1)

    ax.barh(y[pos], abs_w[pos], color="#ff7f0e", edgecolor="none",
            zorder=2, alpha=0.95, height=0.7)
    ax.barh(y[neg], abs_w[neg], color="#1f77ff", edgecolor="none",
            zorder=2, alpha=0.95, height=0.7)

    for yi, v, av in zip(y, weights, abs_w):
        ax.text(av + 0.015*R, yi, f"{v:+.2f}",
                va="center", ha="left", fontsize=9, color="#333")

    ax.set_xlim(0, R)
    ax.set_yticks(y)
    ax.set_yticklabels(labels_txt, fontsize=9)
    ax.invert_yaxis()
    ax.grid(axis="x", linestyle=":", linewidth=0.6, alpha=0.5)
    ax.set_xlabel("Contribuição", fontsize=10)

    # ------------------------------------------------------------------
    # 2) Tabela de valores reais (coluna central)
    # ------------------------------------------------------------------
    axr = fig.add_subplot(gs[0, 1])
    axr.axis("off")

    axr.text(0.00, 0.98, "Feature", fontsize=11, fontweight="bold",
             ha="left", va="top")
    axr.text(0.88, 1.25, "Valor (real)", fontsize=11, fontweight="bold",
             ha="right", va="top")

    def _fmt_val(v):
        try:
            return f"{float(v):.2f}".replace(".", ",")
        except Exception:
            return str(v)

    y0, dy = 0.92, 0.055
    for k, idx_f in enumerate(feat_idx_order):
        if y0 - k*dy < 0.05:
            break
        fname = feat_names[idx_f]
        fval = real_values[k]

        axr.text(0.00, y0 - k*dy, fname, ha="left", va="top", fontsize=9)
        axr.text(1.25, y0 - k*dy, _fmt_val(fval), ha="right", va="top", fontsize=9)

    # ------------------------------------------------------------------
    # 3) Tabela de métricas (coluna direita, POSIÇÕES AINDA MAIS AFASTADAS)
    # ------------------------------------------------------------------
    axm = fig.add_subplot(gs[0, 2])
    axm.axis("off")

    from matplotlib.lines import Line2D

        # ---- linha vertical separando "Valor (real)" das métricas ----
    # Pega as posições dos eixos central (axr) e direito (axm) em coordenadas da figura
    bbox_val = axr.get_position()
    bbox_met = axm.get_position()
    # x da linha: meio do espaço entre o fim de axr e o início de axm
    x_sep = (bbox_val.x1 + bbox_met.x0) / 2.0

    # desenha a linha em coordenadas da figura
    linha_sep = Line2D(
        [x_sep+0.028, x_sep+0.028],       # x0, x1
        [0.10, 0.90],         # y0, y1 (ajuste se quiser a linha mais curta/longa)
        transform=fig.transFigure,
        color="#cccccc",
        linewidth=1.0,
        alpha=0.8
    )
    fig.add_artist(linha_sep)




    # x mais espaçados para evitar qualquer sobreposição
    x_feat   = 0.1
    x_inst   = 1.00
    x_media  = 1.38
    x_mediana= 1.78

    axm.text(x_feat,    0.98, "Métrica",   fontsize=11, fontweight="bold", ha="left",  va="top")
    axm.text(x_inst,    0.98, "Valor",     fontsize=11, fontweight="bold", ha="right", va="top")
    axm.text(x_media,   0.98, "Média",     fontsize=11, fontweight="bold", ha="right", va="top")
    axm.text(x_mediana, 0.98, "Mediana",   fontsize=11, fontweight="bold", ha="right", va="top")

    def _fmt_metric(v):
        return "0,00" if pd.isna(v) else f"{float(v):.3f}".replace(".", ",")

    y0, dy = 0.92, 0.055
    for i, row in mt.iterrows():
        if y0 - i*dy < 0.05:
            break

        axm.text(x_feat,  y0 - i*dy, row['metric'],           ha="left",  va="top", fontsize=9)
        axm.text(x_inst,  y0 - i*dy, _fmt_metric(row['inst']),   ha="right", va="top", fontsize=9)
        axm.text(x_media, y0 - i*dy, _fmt_metric(row['mean']),   ha="right", va="top", fontsize=9)
        axm.text(x_mediana, y0 - i*dy, _fmt_metric(row['median']), ha="right", va="top", fontsize=9)

    plt.subplots_adjust(top=0.82)
    plt.show()







# ----------------- rodar para as suas instâncias -----------------
try:
    ENTRADAS_SELECIONADAS
except NameError:
    ENTRADAS_SELECIONADAS = [0]

ENTRADAS_SELECIONADAS = np.asarray(list(ENTRADAS_SELECIONADAS), dtype=int)
# ENTRADAS_SELECIONADAS = ENTRADAS_SELECIONADAS[
#     (ENTRADAS_SELECIONADAS >= 0) & (ENTRADAS_SELECIONADAS < len(X_test))
# ]

# df_lime_metrics deve existir; se não existir, cria um placeholder vazio só para o plot não quebrar
# if 'df_lime_metrics' not in globals():
#     df_lime_metrics = pd.DataFrame({"instancia": [], **{c:[] for c in METRIC_COLS}})

for idx_pos in ENTRADAS_SELECIONADAS:
    plot_lime_with_real_values_and_metrics(int(idx_pos), df_lime_metrics)


In [ ]:
df_infidelity_por_instancia = pd.DataFrame(rows_inf).sort_values("instancia").reset_index(drop=True)
df_infidelity_por_instancia
df_metrics_faith = pd.DataFrame(rows_faith).sort_values("linha").reset_index(drop=True)

print("✅ df_metrics_faith criado:")
print(df_metrics_faith.head(10))

simplicity_df = pd.DataFrame(rows_simplicity)
simplicity_df


df_cons = pd.DataFrame(rows_consistency)
print(df_cons)


df_robustness = pd.DataFrame(rows_robustness)
df_robustness


df_selectivity = pd.DataFrame(rows_selectivity)
df_selectivity


df_soundness = pd.DataFrame(rows_soundness)
print(df_soundness.head())

df_directional_soundness = pd.DataFrame(rows_dirsound)
df_directional_soundness



df_stability = pd.DataFrame(rows_stability)
df_stability


df_sufficiency = pd.DataFrame(rows_suff).sort_values("instancia").reset_index(drop=True)
df_sufficiency_debug = pd.DataFrame(rows_suff_debug).sort_values("instancia").reset_index(drop=True)

df_sufficiency.head()
df_sufficiency_debug.head()

In [ ]:
# ===  Lime Interpretabilidade [Heatmap das Métricas]===

import seaborn as sns

# Filtra as instâncias, removendo linhas de média/mediana
df_lime_metrics_filter = df_lime_metrics[df_lime_metrics["instancia"] != "MÉDIA"].copy()
df_lime_metrics_filter = df_lime_metrics_filter[df_lime_metrics_filter["instancia"] != "MEDIANA"].copy()

# (opcional) Remover coluna que não deve ser incluída
df_lime_metrics_filter = df_lime_metrics_filter.drop(columns=["class_consistency()"], errors='ignore')

# Remove colunas constantes (100% ou 0%) com base no DF filtrado
cols_to_remove = [
    col for col in df_lime_metrics_filter.columns
    if col != "instancia"
    and df_lime_metrics_filter[col].nunique(dropna=False) == 1
    and (df_lime_metrics_filter[col].iloc[0] == '100.0%' or df_lime_metrics_filter[col].iloc[0] == '0.0%')
]

# Drop de colunas indesejadas + 'instancia' + (opcional) 'completeness(%)'
df_lime_metrics_filter = df_lime_metrics_filter.drop(
    columns=cols_to_remove + ['instancia', 'completeness(%)'],
    errors='ignore'
)

# Converter percentuais "xx.x%" para numérico
df_lime_metrics_filter = (
    df_lime_metrics_filter
    .replace('%', '', regex=True)
    .apply(pd.to_numeric, errors='coerce')
)

# Calcular correlação de Pearson
corr_matrix = df_lime_metrics_filter.corr(method='pearson')

# manter só correlações > 0.3 ou < -0.3
keep = (corr_matrix > 0.3) | (corr_matrix < -0.3)

# Ordenação simples (poderia ser outra): usar a ordem original das colunas
order = corr_matrix.columns.tolist()
ordered = corr_matrix.loc[order, order]

# máscara alinhada ao 'ordered' (True = esconder)
mask = ~keep.loc[order, order].copy()
np.fill_diagonal(mask.values, True)  # (opcional) esconde a diagonal

# Plotar heatmap
plt.figure(figsize=(14, 10))
sns.heatmap(
    ordered,
    annot=True, fmt=".2f", cmap="coolwarm",
    mask=mask, vmin=-1, vmax=1, linewidths=0.5
)
plt.title("Heatmap de Correlação (Pearson) entre Métricas - Apenas valores com |r| > 0.3")
plt.tight_layout()
plt.savefig(f'{BASE_PATH}/heatmap_metricas_filtradas.png', dpi=600)
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.cluster import hierarchy
from scipy.spatial.distance import squareform

# --- escolha: usar correlação absoluta (ignora o sinal) ou assinada
use_abs = True   # True => distância = 1 - |corr| ; False => 1 - corr

# matriz de correlação -> numpy float
C = corr_matrix.to_numpy(dtype=float)

# CORREÇÃO 1: aplicar abs (se for o caso)
C = np.abs(C) if use_abs else C

# CORREÇÃO 2: substituir NaN / inf por 0 (sem correlação)
#  -> isso faz com que a distância vire 1 (máxima)
C = np.nan_to_num(C, nan=0.0, posinf=0.0, neginf=0.0)

# distância (0=correlação perfeita; 1=sem correlação se use_abs=True)
D = 1.0 - C
np.fill_diagonal(D, 0.0)

# linkage precisa do formato "condensed"
condensed_D = squareform(D, checks=False)

# método de ligação; "average" é estável para distâncias de correlação
Z = hierarchy.linkage(condensed_D, method='average')

# dendrograma
plt.figure(figsize=(12, 6))
dn = hierarchy.dendrogram(
    Z,
    labels=corr_matrix.columns.tolist(),
    leaf_rotation=90
)
plt.title(f"Dendrograma das métricas (distância = 1 - {'|corr|' if use_abs else 'corr'})")
plt.ylabel("Distância")
plt.tight_layout()
plt.savefig(f'{BASE_PATH}/dendrogram_metricas.png', dpi=600)
plt.show()

# (opcional) ordem das métricas segundo o dendrograma
order = [corr_matrix.columns[i] for i in dn["leaves"]]
print(order)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from scipy.spatial.distance import squareform

def venn_from_corr_mds_labels(
    corr_matrix,
    radius=1.0,               # raio (mesmo para todos)
    epsilon_frac=0.06,        # ε aplicado SÓ quando r>0: d <- d + ε*R
    sign_mode="both",         # "both" | "positive" | "negative"
    use_abs=False,            # se True, ignora sinais (sign_mode fica sem efeito)
    random_state=42,
    n_init=8,
    max_iter=600,
    figsize=(8,8),
    face_alpha=0.30,
    edge_lw=1.5,
    label_fontsize=8,         # fonte 8
    label_weight='normal',    # sem negrito
    label_jitter=0.03,        # jitter vertical inicial (fração do raio)
    repel_iters=150,          # iterações de repulsão dos rótulos
    repel_step=0.02,          # passo da repulsão (fração do raio)
    inside_margin=0.12,       # margem p/ manter texto dentro do círculo
    title="Venn adaptado (d = (1 − r)·R, ε>0 só para r>0, filtro de sinal)"
):
    """
    Distância alvo entre centros: d(r) = (1 − r)*R  (+ ε*R se r>0).
    - sign_mode:
        * "both": usa r como está;
        * "positive": se r<=0, força r_eff=-1 (distância máxima 2R);
        * "negative": se r>=0, força r_eff=-1 (distância máxima 2R).
    - use_abs=True ignora sinais e, portanto, sign_mode não tem efeito.
    """

    # ---------- 1) correlação -> distância alvo (com epsilon e filtro de sinal) ----------
    C_raw = corr_matrix.to_numpy(dtype=float)
    C_raw = np.clip(C_raw, -1.0, 1.0)
    n = C_raw.shape[0]

    # máscara de r>0 (para epsilon)
    mask_pos = (C_raw > 0)

    if use_abs:
        C_eff = np.abs(C_raw)
    else:
        # aplica filtro de sinal
        if sign_mode == "positive":
            # mantém só r>0; demais vão para -1 (distância máxima 2R)
            C_eff = np.where(C_raw > 0, C_raw, -1.0)
        elif sign_mode == "negative":
            # mantém só r<0; demais vão para -1
            C_eff = np.where(C_raw < 0, C_raw, -1.0)
        else:
            C_eff = C_raw

    R = float(radius)
    T = (1.0 - C_eff) * R  # d(r) base
    # aplica ε apenas para r>0 (com base no sinal original, como pedido)
    if epsilon_frac and epsilon_frac > 0:
        T = T + (epsilon_frac * R) * mask_pos.astype(float)

    # simetriza e zera diagonal
    T = 0.5 * (T + T.T)
    np.fill_diagonal(T, 0.0)

    # ---------- 2) embedding por MDS (SMACOF; fallback p/ MDS clássico) ----------
    try:
        from sklearn.manifold import MDS
        mds = MDS(
            n_components=2, metric=True, dissimilarity='precomputed',
            n_init=n_init, max_iter=max_iter, random_state=random_state
        )
        X = mds.fit_transform(T)
    except Exception:
        # MDS clássico (Torgerson)
        J = np.eye(n) - np.ones((n, n))/n
        B = -0.5 * J @ (T**2) @ J
        eigvals, eigvecs = np.linalg.eigh(B)
        idx = np.argsort(eigvals)[::-1]
        L = np.clip(eigvals[idx][:2], 0, None)
        V = eigvecs[:, idx][:, :2]
        X = V * np.sqrt(L + 1e-12)

    # reescala global para casar melhor T
    DX = np.sqrt(((X[:,None,:] - X[None,:,:])**2).sum(axis=2))
    iu = np.triu_indices_from(DX, k=1)
    num = np.sum(DX[iu] * T[iu])
    den = np.sum(DX[iu]**2) + 1e-12
    s = num/den if den > 0 else 1.0
    X *= s
    X -= X.mean(axis=0, keepdims=True)  # centraliza

    labels = corr_matrix.columns.tolist()

    # ---------- 3) paleta de cores ----------
    def _lc(s): return str(s).strip().lower()
    color_fixed = {
        "robustness":            "#9467bd",
        "stability":             "#17becf",
        "soundness":             "#f7b6d2",
        "directional soundness": "#ff9896",
        "selectivity":           "#c7c7c7",
        "fidelity":              "#2ca02c",
        "infidelity":            "#d62728",
    }
    cmap = plt.get_cmap("tab20")
    colors, k_auto = [], 0
    for name in labels:
        key = _lc(name)
        colors.append(color_fixed.get(key, cmap(k_auto % 20)))
        if key not in color_fixed: k_auto += 1

    # ---------- 4) desenha círculos (mesmo raio) ----------
    fig, ax = plt.subplots(figsize=figsize, dpi=150, facecolor="white")
    for (x, y), col in zip(X, colors):
        circ = Circle((x, y), R, facecolor=col, edgecolor=col,
                      alpha=face_alpha, lw=edge_lw)
        ax.add_patch(circ)

    # ---------- 5) rótulos centralizados + repulsão vertical ----------
    rng = np.random.default_rng(random_state)
    text_xy = X.copy()
    text_xy[:, 1] += (rng.uniform(-label_jitter, label_jitter, size=n)) * R

    def clamp_inside(i):
        cx, cy = X[i]
        tx, ty = text_xy[i]
        vx, vy = tx - cx, ty - cy
        dist = np.hypot(vx, vy)
        maxd = R * (1.0 - inside_margin)
        if dist > maxd and dist > 1e-12:
            k = maxd / dist
            text_xy[i] = np.array([cx + k*vx, cy + k*vy])

    for _ in range(repel_iters):
        moved = False
        for i in range(n):
            for j in range(i+1, n):
                xi, yi = text_xy[i]
                xj, yj = text_xy[j]
                # caixa aproximada do texto
                w = 0.9 * R
                h = 0.35 * R
                if (abs(xi - xj) < w) and (abs(yi - yj) < h):
                    dy = repel_step * R
                    text_xy[i, 1] += dy
                    text_xy[j, 1] -= dy
                    clamp_inside(i)
                    clamp_inside(j)
                    moved = True
        if not moved:
            break

    for (tx, ty), name in zip(text_xy, labels):
        ax.text(tx, ty, name, ha="center", va="center",
                fontsize=label_fontsize, fontweight=label_weight,
                color="black", zorder=3)

    # limites e estética
    xmin = X[:,0].min() - R*1.3
    xmax = X[:,0].max() + R*1.3
    ymin = X[:,1].min() - R*1.3
    ymax = X[:,1].max() + R*1.3
    ax.set_aspect("equal")
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    ax.axis("off")
    ax.set_title(title, fontsize=10)
    plt.tight_layout()
    plt.show()

# ===== Exemplos =====
# 1) ambos os sinais, com epsilon:
# venn_from_corr_mds_labels(corr_matrix, radius=1.0, epsilon_frac=4, sign_mode="both", use_abs=False)

# 2) só correlações positivas:
venn_from_corr_mds_labels(corr_matrix, radius=1.0, epsilon_frac=0.5, sign_mode="positive", use_abs=False)

# 3) só correlações negativas:
# venn_from_corr_mds_labels(corr_matrix, radius=1.0, epsilon_frac=4, sign_mode="negative", use_abs=False)


In [ ]:
import re
import unicodedata
import difflib
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

# --- normalizador de nomes (case/acentos/espacos/simbolos) ---
def _normalize_name(s: str) -> str:
    s = str(s)
    # remove acentos
    s = unicodedata.normalize('NFKD', s)
    s = ''.join(ch for ch in s if not unicodedata.combining(ch))
    s = s.lower().strip()
    # remove marcadores como "(%)", "%", "()", etc
    s = re.sub(r'\(\s*%\s*\)', '', s)
    s = s.replace('%', '')
    # remove espaços, underscores e hifens
    s = re.sub(r'[\s_\-]+', '', s)
    # mantém apenas alfa-num
    s = re.sub(r'[^a-z0-9]', '', s)
    return s

def _resolve_metric_names(corr_columns, requested_names):
    """Mapeia nomes pedidos para colunas reais da corr_matrix via normalização + fuzzy."""
    # dicionário: normalizado -> nome original da coluna
    norm_to_col = {}
    for col in corr_columns:
        norm_to_col.setdefault(_normalize_name(col), col)

    resolved = []
    missing  = []
    suggestions = {}

    for name in requested_names:
        key = _normalize_name(name)
        if key in norm_to_col:
            resolved.append(norm_to_col[key])
        else:
            # tenta fuzzy contra as chaves normalizadas
            choices = list(norm_to_col.keys())
            match = difflib.get_close_matches(key, choices, n=1, cutoff=0.6)
            if match:
                resolved.append(norm_to_col[match[0]])
                suggestions[name] = norm_to_col[match[0]]
            else:
                missing.append(name)

    return resolved, missing, suggestions

def venn_from_corr_subset(
    corr_matrix,
    metrics=None,            # lista com os nomes desejados (em qualquer formato)
    radius=1.0,              # raio R (igual p/ todos)
    epsilon_pos_frac=0.06,   # ε (fração de R) só p/ pares com r>0
    random_state=42,
    n_init=8,
    max_iter=600,
    figsize=(8,8),
    face_alpha=0.30,
    edge_lw=1.5,
    label_fontsize=8,
    label_weight='normal',
    label_jitter=0.03,
    repel_iters=150,
    repel_step=0.02,
    inside_margin=0.12,
    title="Venn adaptado (subset; d = (1 − r)·R; ε>0 só para r>0)"
):
    # ---------- 0) resolver nomes do subset ----------
    if metrics is not None:
        resolved, missing, sugg = _resolve_metric_names(corr_matrix.columns, metrics)
        if missing and not resolved:
            cols_list = ', '.join(map(str, corr_matrix.columns))
            raise ValueError(
                "Nenhuma métrica do subset encontrada nas colunas da corr_matrix.\n"
                f"Pedidas: {metrics}\n"
                f"Colunas disponíveis: {cols_list}"
            )
        if missing:  # avisa mas segue com as resolvidas
            print("[aviso] Métricas não encontradas e ignoradas:", missing)
            if sugg:
                print("[aviso] Sugestões aplicadas:", sugg)
        if not resolved:
            raise ValueError("Após normalização, não sobrou nenhuma métrica válida para plotar.")
        labels = resolved
        C_raw = corr_matrix.loc[labels, labels].to_numpy(dtype=float)
    else:
        labels = corr_matrix.columns.tolist()
        C_raw = corr_matrix.to_numpy(dtype=float)

    # ---------- 1) correlação → distância-alvo ----------
    # CORREÇÃO: garantir intervalo [-1, 1] e remover NaN/inf
    C_raw = np.clip(C_raw, -1.0, 1.0)
    C_raw = np.nan_to_num(C_raw, nan=0.0, posinf=0.0, neginf=0.0)

    R = float(radius)

    # d(r) base
    T = (1.0 - C_raw) * R

    # aplica ε apenas quando r>0 (não aplica em r<=0)
    if epsilon_pos_frac and epsilon_pos_frac > 0:
        mask_pos = (C_raw > 0).astype(float)
        T += (epsilon_pos_frac * R) * mask_pos

    # simetriza, zera diagonal e limpa qualquer NaN residual
    np.fill_diagonal(T, 0.0)
    T = 0.5 * (T + T.T)
    T = np.nan_to_num(T, nan=R, posinf=R, neginf=R)

    n = T.shape[0]

    # ---------- 2) embedding (MDS SMACOF; fallback p/ clássico; fallback aleatório) ----------
    X = None
    try:
        from sklearn.manifold import MDS
        mds = MDS(
            n_components=2,
            metric=True,
            dissimilarity='precomputed',
            n_init=n_init,
            max_iter=max_iter,
            random_state=random_state
        )
        X = mds.fit_transform(T)
    except Exception as e:
        # fallback: MDS clássico (eigendecomposição)
        try:
            J = np.eye(n) - np.ones((n, n))/n
            B = -0.5 * J @ (T**2) @ J
            eigvals, eigvecs = np.linalg.eigh(B)
            idx = np.argsort(eigvals)[::-1]
            L = np.clip(eigvals[idx][:2], 0, None)
            V = eigvecs[:, idx][:, :2]
            X = V * np.sqrt(L + 1e-12)
        except Exception as e2:
            # fallback final: posição aleatória mas reprodutível
            print("[aviso] MDS clássico falhou; usando embedding aleatório. Motivo:", e2)
            rng = np.random.default_rng(random_state)
            X = rng.normal(scale=R, size=(n, 2))

    # ---------- 3) reescala global para aproximar T ----------
    DX = np.sqrt(((X[:,None,:]-X[None,:,:])**2).sum(axis=2))
    iu = np.triu_indices_from(DX, k=1)
    num = np.sum(DX[iu] * T[iu])
    den = np.sum(DX[iu]**2) + 1e-12
    s = num/den if den > 0 else 1.0
    X *= s
    X -= X.mean(axis=0, keepdims=True)

    # ---------- 4) cores ----------
    def _lc(s): return str(s).strip().lower()
    color_fixed = {
        "robustness":            "#9467bd",
        "stability":             "#17becf",
        "soundness":             "#f7b6d2",
        "directional soundness": "#ff9896",
        "selectivity":           "#c7c7c7",
        "fidelity":              "#2ca02c",
        "infidelity":            "#d62728",
    }
    cmap = plt.get_cmap("tab20")
    colors, k_auto = [], 0
    for name in labels:
        key = _lc(name)
        colors.append(color_fixed.get(key, cmap(k_auto % 20)))
        if key not in color_fixed:
            k_auto += 1

    # ---------- 5) círculos ----------
    fig, ax = plt.subplots(figsize=figsize, dpi=150, facecolor="white")
    for (x, y), col in zip(X, colors):
        circ = Circle((x, y), R, facecolor=col, edgecolor=col, alpha=face_alpha, lw=edge_lw)
        ax.add_patch(circ)

    # ---------- 6) rótulos centralizados + repulsão vertical ----------
    rng = np.random.default_rng(random_state)
    text_xy = X.copy()
    text_xy[:, 1] += (rng.uniform(-label_jitter, label_jitter, size=len(labels))) * R

    def clamp_inside(i):
        cx, cy = X[i]
        tx, ty = text_xy[i]
        vx, vy = tx - cx, ty - cy
        dist = np.hypot(vx, vy)
        maxd = R * (1.0 - inside_margin)
        if dist > maxd and dist > 1e-12:
            k = maxd / dist
            text_xy[i] = np.array([cx + k*vx, cy + k*vy])

    for _ in range(repel_iters):
        moved = False
        for i in range(len(labels)):
            for j in range(i+1, len(labels)):
                xi, yi = text_xy[i]
                xj, yj = text_xy[j]
                # caixa aproximada do texto
                w = 0.9 * R
                h = 0.35 * R
                if (abs(xi - xj) < w) and (abs(yi - yj) < h):
                    dy = repel_step * R
                    text_xy[i, 1] += dy
                    text_xy[j, 1] -= dy
                    clamp_inside(i)
                    clamp_inside(j)
                    moved = True
        if not moved:
            break

    for (tx, ty), name in zip(text_xy, labels):
        ax.text(tx, ty, name, ha="center", va="center",
                fontsize=label_fontsize, fontweight=label_weight,
                color="black", zorder=3)

    # ---------- 7) estética ----------
    xmin = X[:,0].min() - R*1.3
    xmax = X[:,0].max() + R*1.3
    ymin = X[:,1].min() - R*1.3
    ymax = X[:,1].max() + R*1.3
    ax.set_aspect("equal")
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    ax.axis("off")
    ax.set_title(title, fontsize=10)
    plt.tight_layout()
    plt.show()


# ===== Exemplo de uso =====
subset = ["Robustness", "Fidelity", "Infidelity"]
venn_from_corr_subset(
    corr_matrix,
    metrics=subset,
    radius=1.0,
    epsilon_pos_frac=0.06,
    figsize=(8, 8),
    label_fontsize=8
)


In [ ]:
# ===  Lime Interpretabilidade [Gráfico de Radar das Instâncias]===
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from math import ceil

# Filtra as instâncias, removendo a linha de média
# Filtra as instâncias, removendo a linha de média
df_lime_metrics_filter = df_lime_metrics[df_lime_metrics["instancia"] != "MÉDIA"].copy()
df_lime_metrics_filter = df_lime_metrics_filter[df_lime_metrics_filter["instancia"] != "MEDIANA"].copy()

# (opcional) Remover coluna que não deve ser incluída
df_lime_metrics_filter = df_lime_metrics_filter.drop(columns=["class_consistency()"], errors='ignore')


cols_to_remove = [col for col in df_lime_metrics_filter.columns
                  if df_lime_metrics[col].nunique() == 1 and (df_lime_metrics[col].iloc[0] == '100.0%' or df_lime_metrics[col].iloc[0] == '0.0%')]

# df_lime_metrics_filter = df_lime_metrics_filter.drop(columns=cols_to_remove + ['instancia', 'completeness(%)', 'monotony(%)', 'relevance(%)'])
df_lime_metrics_filter = df_lime_metrics_filter.drop(columns=cols_to_remove + [ 'completeness(%)'])
df_lime_metrics_filter = df_lime_metrics_filter.replace('%', '', regex=True).apply(pd.to_numeric, errors='ignore')


# Lista de métricas e ângulos
metrics = [col for col in df_lime_metrics_filter.columns if col != 'instancia']
num_metrics = len(metrics)
angles = np.linspace(0, 2 * np.pi, num_metrics, endpoint=False).tolist()
angles += angles[:1]  # fecha o polígono

# Layout do PDF
plots_per_row = 2
rows_per_page = 4
plots_per_page = plots_per_row * rows_per_page
total_instances = len(df_lime_metrics_filter)
pages = ceil(total_instances / plots_per_page)

# Nome do arquivo de saída
pdf_path = "graficos_radar_angular_limpo.pdf"
pdf = PdfPages(pdf_path)

# Loop de páginas
for page in range(pages):
    fig, axs = plt.subplots(rows_per_page, plots_per_row, figsize=(8.27, 11.69), subplot_kw=dict(polar=True))
    fig.suptitle("Análise de Métricas XAI por Instâncias", fontsize=14, y=0.96, color='navy')
    axs = axs.flatten()

    for i in range(plots_per_page):
        idx = page * plots_per_page + i
        if idx >= total_instances:
            axs[i].axis('off')
            continue

        row = df_lime_metrics_filter.iloc[idx]
        values = [float(str(row[m]).replace('', '')) / 100 for m in metrics]
        values += values[:1]  # fecha o polígono

        ax = axs[i]

        # --- radar da instância ---
        ax.plot(angles, values, linewidth=1.25, linestyle='solid', color='navy', zorder=3)
        ax.fill(angles, values, color='cornflowerblue', alpha=0.32, zorder=2)

        # --- remover grades e valores radiais ---
        ax.grid(False)
        ax.xaxis.grid(False)
        ax.yaxis.grid(False)
        ax.set_yticks([])                       # sem círculos/valores radiais
        ax.spines['polar'].set_visible(False)
        ax.set_rlabel_position(0)

        # --- limites (um pouco acima de 1 para dar espaço aos rótulos) ---
        ax.set_ylim(0, 1.05)

        # --- rótulos das métricas (negrito + mais espaçados) ---
        # escondemos os ticks padrão e desenhamos rótulos manualmente levemente fora do raio 1
        ax.set_xticks([])
        label_r = 1.03
        for theta, label in zip(angles[:-1], metrics):
            ax.text(theta, label_r, label,
                    ha='center', va='center',
                    fontsize=6, fontweight='bold',
                    clip_on=False)

        # --- polígono externo (mesmo nº de lados que as métricas) ---
        outer_r = [1.0] * len(angles)           # angles já está fechado
        ax.fill(angles, outer_r, color='#f1f1f1', zorder=0)
        ax.plot(angles, outer_r, linewidth=1.0, color='#444', zorder=1)

        # --- título ---
        ax.set_title(f"Instância {int(row['instancia'])}", size=9, pad=8)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95], h_pad=3.0)
    plt.show()
    # pdf.savefig(fig)
    # plt.close()

pdf.close()
print(f"✅ PDF salvo com sucesso em: {pdf_path}")


In [ ]:
# ===  Lime Interpretabilidade [Gráfico de Radar Global]===
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ===== Atualização: gráfico único com todas as instâncias sobrepostas =====
df_lime_metrics_filter = df_lime_metrics[df_lime_metrics["instancia"] != "MÉDIA"].copy()
df_lime_metrics_filter = df_lime_metrics_filter[df_lime_metrics_filter["instancia"] != "MEDIANA"].copy()

# (opcional) Remover coluna que não deve ser incluída
df_lime_metrics_filter = df_lime_metrics_filter.drop(columns=["class_consistency()"], errors='ignore')


cols_to_remove = [col for col in df_lime_metrics_filter.columns
                  if df_lime_metrics[col].nunique() == 1 and (df_lime_metrics[col].iloc[0] == '100.0%' or df_lime_metrics[col].iloc[0] == '0.0%')]

# df_lime_metrics_filter = df_lime_metrics_filter.drop(columns=cols_to_remove + ['instancia', 'completeness(%)', 'monotony(%)', 'relevance(%)'])
df_lime_metrics_filter = df_lime_metrics_filter.drop(columns=cols_to_remove + [ 'completeness(%)'])
df_lime_metrics_filter = df_lime_metrics_filter.replace('%', '', regex=True).apply(pd.to_numeric, errors='ignore')

# Métricas e ângulos
metrics = [col for col in df_lime_metrics_filter.columns if col != 'instancia']
angles = np.linspace(0, 2 * np.pi, len(metrics), endpoint=False).tolist()
angles += angles[:1]

# Gráfico único
fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))
fig.suptitle("Todas as Instâncias sobrepostas", fontsize=16, color='darkorange')

# Rótulos e grades
rgrid_values = [10, 30, 50, 70, 90]
ax.set_xticks(angles[:-1])
ax.set_xticklabels(metrics, fontsize=8)
ax.set_yticks(rgrid_values)
ax.set_yticklabels([str(v) for v in rgrid_values], fontsize=7)
ax.set_rlabel_position(0)
ax.spines['polar'].set_visible(True)
ax.spines['polar'].set_color('darkorange')
ax.grid(True, linestyle='dotted', linewidth=0.8, alpha=0.6, color='orange')

# Plotar cada instância
for _, row in df_lime_metrics_filter.iterrows():
    values = [float(str(row[m]).replace('%', '')) for m in metrics]
    values += values[:1]
    ax.plot(angles, values, linewidth=1.2, linestyle='solid', alpha=0.5, label=f"Instância {int(row['instancia'])}")
    ax.fill(angles, values, alpha=0.05, color='orange')

ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.05), fontsize=7)
plt.tight_layout()
plt.show()

In [ ]:
# ===  Lime Interpretabilidade [Gráfico de Radar por Intrancias e Grupos de Métricas]===
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from math import ceil

# ===== 1. Filtrar e preparar =====
df_lime_metrics_filter = df_lime_metrics[df_lime_metrics["instancia"] != "MÉDIA"].copy()
df_lime_metrics_filter = df_lime_metrics_filter[df_lime_metrics_filter["instancia"] != "MEDIANA"].copy()

# (opcional) Remover coluna que não deve ser incluída
df_lime_metrics_filter = df_lime_metrics_filter.drop(columns=["class_consistency()"], errors='ignore')


cols_to_remove = [col for col in df_lime_metrics_filter.columns
                  if df_lime_metrics[col].nunique() == 1 and (df_lime_metrics[col].iloc[0] == '100.0%' or df_lime_metrics[col].iloc[0] == '0.0%')]

# df_lime_metrics_filter = df_lime_metrics_filter.drop(columns=cols_to_remove + ['instancia', 'completeness(%)', 'monotony(%)', 'relevance(%)'])
df_lime_metrics_filter = df_lime_metrics_filter.drop(columns=cols_to_remove + [ 'completeness(%)'])
df_lime_metrics_filter = df_lime_metrics_filter.replace('%', '', regex=True).apply(pd.to_numeric, errors='ignore')

# ===== 2. Definir grupos =====
grupos = {
    "Fidelidade": [
        "fidelity(%)",
        "infidelity(%)",
        "faithfulness(%)",
        "soundness(%)",
        "directional soundness(%)",
    ],
    "Relevância": [
        "selectivity(%)",  # 'relevance(%)' não aparece no seu dataset
    ],
    "Simplicidade": [
        "simplicity(%)",
    ],
    "Robustez": [
        "robustez(%)",
        "stability(%)",    # 'robustez_adversarial(%)' não disponível
    ],
    "Generalização": [
        # "completeness(%)",
        "consistency(%)",  # no seu dataset vem com '%'
        "sufficiency-1(%)",
        "sufficiency-5(%)",
        "sufficiency-10(%)",
        "sufficiency-20(%)",
    ],
}

# Verificar quais colunas estão no DataFrame
for k, v in grupos.items():
    grupos[k] = [col for col in v if col in df_lime_metrics_filter.columns]

# ===== 3. Calcular médias por grupo =====
df_grupos = pd.DataFrame()
df_grupos['instancia'] = df_lime_metrics_filter['instancia']

for nome_grupo, cols in grupos.items():
    df_grupos[nome_grupo] = df_lime_metrics_filter[cols].replace('%', '', regex=True).astype(float).mean(axis=1)

# ===== 4. Gerar gráfico radar com médias por grupo =====

# Lista de métricas agregadas e ângulos
metrics = [col for col in df_grupos.columns if col != 'instancia']
angles = np.linspace(0, 2 * np.pi, len(metrics), endpoint=False).tolist()
angles += angles[:1]

# Layout
plots_per_row = 2
rows_per_page = 4
plots_per_page = plots_per_row * rows_per_page
total_instances = len(df_grupos)
pages = ceil(total_instances / plots_per_page)
rgrid_values = [10, 30, 50, 70, 90]

# Saída PDF
pdf_path = "graficos_teia_laranja_por_grupo.pdf"
pdf = PdfPages(pdf_path)

for page in range(pages):
    fig, axs = plt.subplots(rows_per_page, plots_per_row, figsize=(8.27, 11.69), subplot_kw=dict(polar=True))
    fig.suptitle("Análise de Grupos de Métricas XAI por Instância", fontsize=14, y=0.96, color='darkorange')
    axs = axs.flatten()

    for i in range(plots_per_page):
        idx = page * plots_per_page + i
        if idx >= total_instances:
            axs[i].axis('off')
            continue

        row = df_grupos.iloc[idx]
        raw_values = [row[m] for m in metrics]
        values = raw_values + raw_values[:1]

        ax = axs[i]
        ax.plot(angles, values, linewidth=2, linestyle='solid', color='darkorange')
        ax.fill(angles, values, color='orange', alpha=0.3)
        ax.set_xticks(angles[:-1])
        ax.set_xticklabels(metrics, fontsize=7)

        # Melhor espaçamento dos labels angulares
        for label in ax.get_xticklabels():
            label.set_horizontalalignment('center')
            label.set_fontsize(6)
            label.set_rotation(15)

        ax.set_yticks(rgrid_values)
        ax.set_yticklabels([str(v) for v in rgrid_values], fontsize=5)
        ax.set_rlabel_position(0)
        ax.set_title(f"Instância {row['instancia']}", size=9, pad=10, color='darkorange')
        ax.spines['polar'].set_visible(True)
        ax.spines['polar'].set_color('darkorange')
        ax.grid(True, linestyle='dotted', linewidth=0.8, alpha=0.6, color='orange')

    plt.tight_layout(rect=[0, 0.03, 1, 0.95], h_pad=3.0)
    plt.show()
    # pdf.savefig(fig)
    # plt.close()

pdf.close()
print(f"PDF salvo em: {pdf_path}")


In [ ]:
# ===  Lime Interpretabilidade [Gráfico de Radar por Intrancias e Global por Grupos de Métricas]===
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df_lime_metrics_filter = df_lime_metrics[df_lime_metrics["instancia"] != "MÉDIA"].copy()
df_lime_metrics_filter = df_lime_metrics_filter[df_lime_metrics_filter["instancia"] != "MEDIANA"].copy()

# (opcional) Remover coluna que não deve ser incluída
df_lime_metrics_filter = df_lime_metrics_filter.drop(columns=["class_consistency()"], errors='ignore')


cols_to_remove = [col for col in df_lime_metrics_filter.columns
                  if df_lime_metrics[col].nunique() == 1 and (df_lime_metrics[col].iloc[0] == '100.0%' or df_lime_metrics[col].iloc[0] == '0.0%')]

# df_lime_metrics_filter = df_lime_metrics_filter.drop(columns=cols_to_remove + ['instancia', 'completeness(%)', 'monotony(%)', 'relevance(%)'])
df_lime_metrics_filter = df_lime_metrics_filter.drop(columns=cols_to_remove + [ 'completeness(%)'])
df_lime_metrics_filter = df_lime_metrics_filter.replace('%', '', regex=True).apply(pd.to_numeric, errors='ignore')

# ===== 2. Definir grupos =====
# ===== 2. Definir grupos =====
grupos = {
    "Fidelidade": [
        "fidelity(%)",
        "infidelity(%)",
        "faithfulness(%)",
        "soundness(%)",
        "directional soundness(%)",
    ],
    "Relevância": [
        "selectivity(%)",  # 'relevance(%)' não aparece no seu dataset
    ],
    "Simplicidade": [
        "simplicity(%)",
    ],
    "Robustez": [
        "robustez(%)",
        "stability(%)",    # 'robustez_adversarial(%)' não disponível
    ],
    "Generalização": [
        # "completeness(%)",
        "consistency(%)",  # no seu dataset vem com '%'
        "sufficiency-1(%)",
        "sufficiency-5(%)",
        "sufficiency-10(%)",
        "sufficiency-20(%)",
    ],
}

# Verificar colunas válidas
for k, v in grupos.items():
    grupos[k] = [col for col in v if col in df_lime_metrics_filter.columns]

# ===== 3. Calcular médias por grupo =====
df_grupos = pd.DataFrame()
df_grupos['instancia'] = df_lime_metrics_filter['instancia']

for nome_grupo, cols in grupos.items():
    df_grupos[nome_grupo] = df_lime_metrics_filter[cols].replace('%', '', regex=True).astype(float).mean(axis=1)

# ===== 4. Gráfico único com todas as instâncias =====
metrics = [col for col in df_grupos.columns if col != 'instancia']
angles = np.linspace(0, 2 * np.pi, len(metrics), endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))
fig.suptitle("Todas as Instâncias sobrepostas (Agrupadas por Métricas)", fontsize=16, color='darkorange')

rgrid_values = [10, 30, 50, 70, 90]
ax.set_xticks(angles[:-1])
ax.set_xticklabels(metrics, fontsize=8)
ax.set_yticks(rgrid_values)
ax.set_yticklabels([str(v) for v in rgrid_values], fontsize=7)
ax.set_rlabel_position(0)
ax.spines['polar'].set_visible(True)
ax.spines['polar'].set_color('darkorange')
ax.grid(True, linestyle='dotted', linewidth=0.8, alpha=0.6, color='orange')

# Plotar todas as instâncias agrupadas
for _, row in df_grupos.iterrows():
    values = [row[m] for m in metrics]
    values += values[:1]
    ax.plot(angles, values, linewidth=1.5, linestyle='solid', alpha=0.5, label=f"Instância {int(row['instancia'])}")
    ax.fill(angles, values, alpha=0.05, color='orange')

ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.05), fontsize=7)
plt.tight_layout()
plt.show()


In [ ]:
# === UpSet plot das métricas LIME (sobreposição por instância) ===
# Usa df_lime_metrics já existente. Salva em BASE_PATH / "upset_lime_metricas.png"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# upsetplot
try:
    from upsetplot import UpSet, from_indicators
except Exception:
    raise ImportError("Instale o pacote 'upsetplot': pip install upsetplot")

# ----------------- CONFIG -----------------
THRESHOLD_ABS = 60.0       # limiar absoluto em %
USE_PERCENTILE = False     # se True, usa percentil por métrica
PERCENTILE_Q = 75

# BASE_PATH seguro (aceita str ou Path; se não existir, usa cwd)
SAFE_BASE = Path(BASE_PATH) if 'BASE_PATH' in globals() else Path.cwd()
OUTFILE   = SAFE_BASE / "upset_lime_metricas.png"

# ----------------- PREPARO -----------------
# preserva média/mediana numa cópia separada (para referência)
agg_rows = df_lime_metrics[
    df_lime_metrics["instancia"].astype(str).str.upper().isin(["MÉDIA", "MEDIANA"])
].copy()

# trabalha numa cópia sem média/mediana para o UpSet
dfu = df_lime_metrics[
    ~df_lime_metrics["instancia"].astype(str).str.upper().isin(["MÉDIA", "MEDIANA"])
].copy()

# pega colunas de métricas "(%)"
metric_cols = [c for c in dfu.columns if c.endswith("(%)")]
if not metric_cols:
    raise ValueError("Não encontrei colunas de métricas no formato 'nome(%)'.")

# “xx.x%” -> float
M = (
    dfu[metric_cols]
    .replace('%', '', regex=True)
    .replace('N/A', np.nan)
    .apply(pd.to_numeric, errors='coerce')
)

# limiares por métrica
if USE_PERCENTILE:
    thresholds = M.quantile(PERCENTILE_Q / 100.0, numeric_only=True)
else:
    thresholds = pd.Series(THRESHOLD_ABS, index=M.columns, dtype=float)

# indicadores booleanos: métrica >= limiar
indicators = M.ge(thresholds, axis=1)

# constrói dados para o UpSet
data_upset = from_indicators(indicators=indicators.columns.tolist(), data=indicators)

# ----------------- PLOT -----------------
plt.figure(figsize=(12, 7), dpi=150)
up = UpSet(
    data_upset,
    show_counts=True,
    sort_by="cardinality",
    sort_categories_by="cardinality",
    element_size=28
)
up.plot()

title = (f"UpSet LIME — interseções de métricas ≥ P{PERCENTILE_Q}"
         if USE_PERCENTILE else
         f"UpSet LIME — interseções de métricas ≥ {THRESHOLD_ABS:.0f}%")
plt.suptitle(title, y=1.02)

plt.savefig(OUTFILE, dpi=600, bbox_inches="tight")
plt.show()
print(f"✅ UpSet salvo em: {OUTFILE}")
# opcional: acessar agg_rows se quiser consultar média/mediana preservadas


In [ ]:
# === UpSet de Correlação entre Métricas LIME ===
# Nomes das métricas à esquerda (padrão) + replicados na base (vertical)
# Requer: df_lime_metrics e (opcional) BASE_PATH (str ou Path)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# upsetplot
try:
    from upsetplot import UpSet, from_memberships
except Exception:
    raise ImportError("Instale: pip install upsetplot")

# ----------------- CONFIG -----------------
TAU = 0.30        # limiar de |correlação| p/ considerar “alta correlação”
MIN_GROUP = 2     # grupos com pelo menos 2 métricas
SAFE_BASE = Path(BASE_PATH) if 'BASE_PATH' in globals() else Path.cwd()
OUTFILE = SAFE_BASE / "upset_lime_corr.png"

# ----------------- PREPARO -----------------
# remove média/mediana
dfu = df_lime_metrics[
    ~df_lime_metrics["instancia"].astype(str).str.upper().isin(["MÉDIA","MEDIANA"])
].copy()

# seleciona métricas "(%)" e converte p/ float
metric_cols = [c for c in dfu.columns if c.endswith("(%)")]
if not metric_cols:
    raise ValueError("Não encontrei colunas de métricas no formato 'nome(%)'.")

M = (
    dfu[metric_cols]
      .replace('%', '', regex=True)
      .replace('N/A', np.nan)
      .apply(pd.to_numeric, errors='coerce')
)

# correlação coluna-coluna
corr = M.corr(method="pearson")

# memberships: para cada métrica i, conjunto das que têm |r| ≥ TAU com i
metrics = corr.columns.tolist()
memberships, labels_anchor = [], []
for i in metrics:
    group = [
        j for j in metrics
        if j != i and np.isfinite(corr.loc[i, j]) and abs(corr.loc[i, j]) >= TAU
    ]
    if len(group) >= MIN_GROUP:
        memberships.append(group)
        labels_anchor.append(i)

if not memberships:
    print(f"⚠️ Nenhum grupo de métricas com |r| ≥ {TAU:.2f}. Tente reduzir TAU.")
else:
    data = from_memberships(memberships)

    # ----------------- PLOT (horizontal: nomes à esquerda) -----------------
    fig = plt.figure(figsize=(22, 10), dpi=300)
    up = UpSet(
        data,
        show_counts=True,
        sort_by="cardinality",
        sort_categories_by="cardinality",
        element_size=30,
        orientation="horizontal",  # mantém nomes à esquerda
        subset_size="count"        # 👈 CORREÇÃO: lida com grupos não únicos
    )
    up.plot()

    # --------- Faixa auxiliar com nomes das métricas na base ----------
    # Encontrar, de forma robusta, o eixo que contém os rótulos Y das métricas
    fig = plt.gcf()
    metric_set = set(metrics)
    left_axis = None
    for ax in fig.axes:
        ytxt = [t.get_text() for t in ax.get_yticklabels()]
        match_count = sum(1 for t in ytxt if t in metric_set)
        if match_count >= max(3, len(metric_set) // 2):  # heurística robusta
            left_axis = ax
            break

    if left_axis is None:
        # fallback: usa o eixo com mais rótulos de métricas
        best_ax, best_score = None, -1
        for ax in fig.axes:
            score = sum(1 for t in ax.get_yticklabels() if t.get_text() in metric_set)
            if score > best_score:
                best_ax, best_score = ax, score
        left_axis = best_ax

    # Se ainda assim não achou, evita crash
    if left_axis is not None:
        pos = left_axis.get_position()  # bbox do eixo da matriz

        # >>> AJUSTES APENAS NOS LABELS DA PARTE INFERIOR <<<
        gutter_h = 0.10  # aumenta a altura da faixa inferior para não sobrepor
        bottom = max(0.02, pos.y0 - gutter_h - 0.02)
        ax_bottom = fig.add_axes([pos.x0, bottom, pos.width, gutter_h])
        ax_bottom.set_xlim(0, 1)
        ax_bottom.set_ylim(-0.2, 1.0)
        ax_bottom.axis("off")

        # ordem dos nomes conforme aparecem à esquerda
        ylabels = [lab.get_text() for lab in left_axis.get_yticklabels() if lab.get_text()]
        n = len(ylabels)
        if n > 0:
            xs = np.linspace(0.0, 1.0, n, endpoint=True)
            for x, name in zip(xs, ylabels):
                ax_bottom.text(
                    x, -0.12, name,
                    rotation=90,
                    ha="center",
                    va="top",
                    fontsize=10
                )

    plt.suptitle(f"UpSet LIME — grupos de métricas com |corr| ≥ {TAU:.2f}", y=0.98)
    plt.savefig(OUTFILE, dpi=600, bbox_inches="tight")
    plt.show()
    print(f"✅ UpSet (correlação) LIME salvo em: {OUTFILE}")
